# 1. Inicialização e Conexões


## 1.1 Importação de Bibliotecas


In [1]:
# ==============================================================================
# 1.1 IMPORTAÇÃO DE BIBLIOTECAS
# ==============================================================================
import os
import sys
import re
import uuid
import unicodedata
from pathlib import Path
from datetime import datetime
from io import BytesIO
from urllib.parse import urljoin

import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text
from sqlalchemy.exc import SQLAlchemyError

# Recarrega automaticamente funções alteradas durante o uso do notebook
%load_ext autoreload
%autoreload 2


# ==============================================================================
# FUNÇÕES
# ==============================================================================
PASTA_FUNCOES = Path(r'C:\Users\Guilherme\LABOR RURAL\Analytics - Departamento Analytics\TEMP\GUILHERME\SCRIPTS\projetcs\lr-functions\functions')

if not PASTA_FUNCOES.exists():
    raise FileNotFoundError(f'Pasta das funções não encontrada:\n{PASTA_FUNCOES}')

if str(PASTA_FUNCOES) not in sys.path:
    sys.path.insert(0, str(PASTA_FUNCOES))

from excel_format import (
    exportar_varias_abas_xlsx,
    exportar_xlsx_formatado,
    aplicar_estilo_listrado_xlsx,
)

print('Funções de Excel importadas com sucesso.')

Funções de Excel importadas com sucesso.


## 1.2 Configuração das Variáveis de Ambiente e Conexão


In [2]:
# ==============================================================================
# 1.2 CONFIGURAÇÃO DAS VARIÁVEIS DE AMBIENTE E CONEXÃO
# ==============================================================================
# Carrega as variáveis do arquivo .env
load_dotenv()

# Configurações de conexão
PG_HOST = os.getenv("PG_HOST")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DATABASE = os.getenv("PG_DATABASE")
PG_USER = os.getenv("PG_USER")
PG_PASSWORD = os.getenv("PG_PASSWORD")
PG_SCHEMA = os.getenv("PG_SCHEMA", "analytics_mart")

# Verifica se as configurações obrigatórias foram preenchidas
configuracoes = {
    "PG_HOST": PG_HOST,
    "PG_DATABASE": PG_DATABASE,
    "PG_USER": PG_USER,
    "PG_PASSWORD": PG_PASSWORD,
}

faltantes = [
    nome
    for nome, valor in configuracoes.items()
    if valor is None or str(valor).strip() == ""
]

if faltantes:
    raise ValueError(
        "As seguintes variáveis não foram preenchidas no arquivo .env: "
        + ", ".join(faltantes)
    )


# Criação segura da URL de conexão
url_conexao = URL.create(
    drivername="postgresql+psycopg",
    username=PG_USER,
    password=PG_PASSWORD,
    host=PG_HOST,
    port=PG_PORT,
    database=PG_DATABASE,
)


# Engine de conexão
engine = create_engine(
    url_conexao,
    pool_pre_ping=True,
)


## 1.3 Teste de Conectividade com o Banco de Dados


In [3]:
# ==============================================================================
# 1.3 TESTE DE CONECTIVIDADE COM O BANCO DE DADOS
# ==============================================================================
try:
    with engine.connect() as conexao:
        resultado = conexao.execute(
            text(
                """
                SELECT
                    current_database() AS banco,
                    current_user AS usuario,
                    current_schema() AS schema_atual,
                    version() AS versao
                """
            )
        ).mappings().one()
    
    print("Conexão realizada com sucesso!")
    print(f"Banco: {resultado['banco']}")
    print(f"Usuário: {resultado['usuario']}")
    print(f"Schema atual: {resultado['schema_atual']}")

except SQLAlchemyError as erro:
    print("Não foi possível conectar ao PostgreSQL.")
    raise erro


Conexão realizada com sucesso!
Banco: postgres
Usuário: lradmin
Schema atual: public


## 1.4 Mapeamento e Configuração das Views SQL


In [4]:
# ==============================================================================
# 1.4 MAPEAMENTO E CONFIGURAÇÃO DAS VIEWS SQL
# ==============================================================================
# Chaves usadas nos merges
CHAVES = ["id_property", "reference_month"]

# Período analisado
DATA_INICIAL = pd.Timestamp("2023-01-01")

# Primeiro dia do mês atual
DATA_FINAL = (pd.Timestamp.today().to_period("M").to_timestamp())

# View principal da tabela final
VIEW_BASE = "vw_revenue"

# Views que serão adicionadas à view principal
VIEWS_MERGE = [
    "vw_cattle",
    "vw_expense",
    "vw_feeding",
    "vw_labor",
    "vw_own_milk",
    "mvw_area_land_summary",
    "mvw_asset_payment_history",
    "vw_dairy_production_system_monthly",
]

VIEWS_FORRAGEIRA = [
    "vw_forage_cost",
    "vw_forage_cost_stage",
    "vw_forage_production",
    'vw_feeding_entries_needing_unit_price'
]

# Lista completa de views autorizadas para importação
views = list(dict.fromkeys( [VIEW_BASE] + VIEWS_MERGE + VIEWS_FORRAGEIRA))

# ==============================================================================
# CONFIGURAÇÃO ESPECÍFICA DA ÁREA ATIVA
# ==============================================================================

COLUNAS_AREA_ATIVA = [
    "id_property",
    "reference_month",
    "hectares_owned_benfeitorias_estradas",
    "hectares_owned_app_reserva_legal",
    "hectares_owned_forrageiras",
    "hectares_rented_benfeitorias_estradas",
    "hectares_rented_app_reserva_legal",
    "hectares_rented_forrageiras",
    "raw_land_value_benfeitorias_estradas",
    "raw_land_value_app_reserva_legal",
    "raw_land_value_forrageiras",
]

# ==============================================================================
# FUNÇÃO DE IMPORTAÇÃO
# ==============================================================================

def importar_view(nome_view: str, engine, schema: str, ordenar_por: str | None = None,) -> pd.DataFrame:
    """
    Importa uma view PostgreSQL para um DataFrame.

    A view precisa estar cadastrada na lista `views`. 
    A ordenação é feita no pandas somente quando a coluna informada existir.
    """

    if nome_view not in views:
        raise ValueError( f"View não autorizada: {nome_view}" )
    
    print(f"Importando {schema}.{nome_view}...")
    
    consulta = text(f''' SELECT * FROM "{schema}"."{nome_view}"; ''')
    
    df = pd.read_sql_query(sql=consulta, con=engine)

    if (ordenar_por is not None and ordenar_por in df.columns):
        df = (df.sort_values(ordenar_por).reset_index(drop=True) )

    return df


# ==============================================================================
# CONFERÊNCIA DAS CONFIGURAÇÕES
# ==============================================================================

print(f"View principal: {VIEW_BASE}")

print("\nViews usadas nos merges:")
for nome_view in VIEWS_MERGE:
    print(f"- {nome_view}")

print("\nViews autorizadas para importação:")
for nome_view in views:
    print(f"- {nome_view}")

print(f"\nPeríodo: " f"{DATA_INICIAL:%Y-%m} a {DATA_FINAL:%Y-%m}" )


View principal: vw_revenue

Views usadas nos merges:
- vw_cattle
- vw_expense
- vw_feeding
- vw_labor
- vw_own_milk
- mvw_area_land_summary
- mvw_asset_payment_history
- vw_dairy_production_system_monthly

Views autorizadas para importação:
- vw_revenue
- vw_cattle
- vw_expense
- vw_feeding
- vw_labor
- vw_own_milk
- mvw_area_land_summary
- mvw_asset_payment_history
- vw_dairy_production_system_monthly
- vw_forage_cost
- vw_forage_cost_stage
- vw_forage_production
- vw_feeding_entries_needing_unit_price

Período: 2023-01 a 2026-07


## 1.5 Conexão Direta com Driver Supabase / PostgreSQL


In [5]:
# ==============================================================================
# 1.5 CONEXÃO DIRETA COM DRIVER SUPABASE / POSTGRESQL
# ==============================================================================
from supabase import create_client, Client

service_key = os.getenv('SUPABASE_SERVICE_KEY')
print("Chave carregada?", service_key is not None)

# URL do Projeto
project_url = 'https://mrjrkkbecjyzzwkvouxx.supabase.co'

# Acesso ao cliente
global supabase
supabase: Client = create_client(project_url, service_key)
print("Supabase conectado!")


Chave carregada? True
Supabase conectado!


# 2. Correção Monetária (Índice IGP-DI)


In [6]:
# ==============================================================================
# 2. CORREÇÃO MONETÁRIA (ÍNDICE IGP-DI)
# ==============================================================================
URL_IGPDI = "https://sindusconpr.com.br/igp-di-fgv-308-p/"

HEADERS_IGPDI = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/150.0.0.0 Safari/537.36"
    )
}


def localizar_link_igpdi(timeout=30):
    """
    Acessa a página do Sinduscon-PR e localiza automaticamente o link de download da série histórica do IGP-DI.
    """
    resposta = requests.get(
        URL_IGPDI,
        headers=HEADERS_IGPDI,
        timeout=timeout
    )
    resposta.raise_for_status()

    soup = BeautifulSoup(resposta.text, "html.parser")

    # Busca prioritária: botão DOWNLOAD dentro da linha do IGP-DI
    for link in soup.select("a[href]"):
        texto_link = link.get_text(" ", strip=True).upper()

        linha = link.find_parent("tr")
        contexto = (
            linha.get_text(" ", strip=True).upper()
            if linha is not None
            else link.parent.get_text(" ", strip=True).upper()
        )

        if "DOWNLOAD" in texto_link and "IGP" in contexto:
            return urljoin(URL_IGPDI, link["href"])

    # Busca alternativa pelos links de download da página
    for link in soup.select("a[href]"):
        href = link.get("href", "")

        if "/download/" in href:
            return urljoin(URL_IGPDI, href)

    raise RuntimeError(
        "Não foi possível localizar o arquivo XLSX do IGP-DI na página."
    )


def carregar_igpdi_supabase():
    """
    Carrega a tabela atual do IGP-DI armazenada no Supabase.
    """
    response = (
        supabase
        .table("tab_igpdi")
        .select("data,igpdi,igpdi_atual,deflator")
        .execute()
    )

    df = pd.DataFrame(response.data or [])

    if df.empty:
        return df

    df["data"] = pd.to_datetime(df["data"], errors="coerce")

    for coluna in ["igpdi", "igpdi_atual", "deflator"]:
        df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

    return (
        df
        .dropna(subset=["data"])
        .sort_values("data")
        .reset_index(drop=True)
    )


def series_igpdi_iguais(df_site, df_supabase):
    """
    Verifica se a série do site é igual à série do Supabase.
    """
    if df_site.empty or df_supabase.empty:
        return False

    site = (
        df_site[["data", "igpdi"]]
        .dropna(subset=["data"])
        .sort_values("data")
        .reset_index(drop=True)
    )

    supa = (
        df_supabase[["data", "igpdi"]]
        .dropna(subset=["data"])
        .sort_values("data")
        .reset_index(drop=True)
    )

    if len(site) != len(supa):
        return False

    mesmas_datas = site["data"].equals(supa["data"])

    mesmos_valores = np.allclose(
        site["igpdi"].to_numpy(dtype=float),
        supa["igpdi"].to_numpy(dtype=float),
        equal_nan=True
    )

    return mesmas_datas and mesmos_valores


def baixar_dados_igpdi(timeout=30):
    """
    Baixa a planilha do IGP-DI diretamente, sem Selenium.

    Se a série estiver igual à armazenada no Supabase, retorna os dados do Supabase.

    Se houver alteração, recalcula o deflator e atualiza toda a tabela no Supabase.
    """
    # 1. Localizar o arquivo
    link_download = localizar_link_igpdi(timeout=timeout)

    # 2. Baixar o XLSX diretamente
    resposta = requests.get(
        link_download,
        headers={
            **HEADERS_IGPDI,
            "Referer": URL_IGPDI
        },
        timeout=timeout
    )
    resposta.raise_for_status()

    # Arquivos XLSX são arquivos ZIP e normalmente começam com PK
    if not resposta.content.startswith(b"PK"):
        raise RuntimeError(
            "O conteúdo baixado não parece ser um arquivo XLSX válido."
        )

    # 3. Ler e tratar sem salvar na pasta Downloads
    arquivo_memoria = BytesIO(resposta.content)

    # A Plan1 possui três linhas de título antes da série histórica.
    df_site = pd.read_excel(
        arquivo_memoria,
        sheet_name="Plan1",
        header=None,
        skiprows=3,
        usecols=[0, 1],
        names=["data", "igpdi"],
    )

    df_site["data"] = pd.to_datetime(df_site["data"], errors="coerce")
    df_site["igpdi"] = pd.to_numeric(df_site["igpdi"], errors="coerce")

    df_site = (
        df_site
        .dropna(subset=["data", "igpdi"])
        .sort_values("data")
        .drop_duplicates(subset=["data"], keep="last")
        .reset_index(drop=True)
    )

    if df_site.empty:
        raise ValueError("Nenhum registro válido de IGP-DI foi encontrado na planilha.")

    # Mantém a convenção existente: deflator = índice do mês / índice mais recente.
    igpdi_atual = df_site["igpdi"].iloc[-1]
    df_site["igpdi_atual"] = igpdi_atual
    df_site["deflator"] = df_site["igpdi"] / igpdi_atual

    # 4. Consultar Supabase
    try:
        df_supabase = carregar_igpdi_supabase()
    except Exception as erro:
        print(f"⚠️ Não foi possível consultar o Supabase: {erro}")
        df_supabase = pd.DataFrame()

    # 5. Retornar Supabase se não houver alteração
    if series_igpdi_iguais(df_site, df_supabase):
        print(f"✅ IGP-DI já está atualizado no Supabase. Último mês: {df_supabase['data'].max():%m/%Y}" )

        return df_supabase

    # 6. Preparar os registros para envio
    df_upload = df_site.copy()
    df_upload["data"] = df_upload["data"].dt.strftime("%Y-%m-%d")
    df_upload = ( df_upload .astype(object) .where(pd.notna(df_upload), None) )

    registros = df_upload.to_dict("records")

    # 7. Atualizar toda a tabela porque o deflator histórico muda
    try:
        (supabase.table("tab_igpdi").delete().gte("data", "1900-01-01").execute())
        (supabase .table("tab_igpdi").insert(registros).execute())
        print(f"✅ Supabase atualizado com {len(registros)} registros. Último mês: {df_site['data'].max():%m/%Y}")

    except Exception as erro:
        raise RuntimeError( f"Erro ao atualizar a tabela tab_igpdi: {erro}" ) from erro

    return df_site

df_igpdi = baixar_dados_igpdi()


✅ IGP-DI já está atualizado no Supabase. Último mês: 06/2026


# 3. Indicadores Mensais - Extração, Tratamento e Consistência


In [7]:
# ==============================================================================
# 3. CONSULTA INICIAL E VERIFICAÇÃO DE CONEXÃO
# ==============================================================================
consulta_conexao = text("""
    SELECT
        current_database() AS banco_atual,
        current_user AS usuario_atual,
        current_schema() AS schema_atual,
        current_setting('search_path') AS search_path,
        inet_server_addr() AS endereco_servidor,
        inet_server_port() AS porta_servidor;
""")

with engine.connect() as conexao:
    diagnostico_conexao = pd.read_sql_query(consulta_conexao, conexao)

display(diagnostico_conexao)


,banco_atual,usuario_atual,schema_atual,search_path,endereco_servidor,porta_servidor
0,postgres,lradmin,public,"""$user"", public",10.34.0.4,5432


## 3.1 Importação Individual das Views SQL


In [8]:
# ==============================================================================
# 3.1 IMPORTAÇÃO INDIVIDUAL DAS VIEWS SQL
# ==============================================================================
# Cada view recebe um DataFrame próprio para permitir tratamentos específicos.
df_revenue                         = importar_view(nome_view="vw_revenue",                         engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_cattle                          = importar_view(nome_view="vw_cattle",                          engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_expense                         = importar_view(nome_view="vw_expense",                         engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_feeding                         = importar_view(nome_view="vw_feeding",                         engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_labor                           = importar_view(nome_view="vw_labor",                           engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_own_milk                        = importar_view(nome_view="vw_own_milk",                        engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_area                            = importar_view(nome_view="mvw_area_land_summary",              engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_asset_payment_history           = importar_view(nome_view="mvw_asset_payment_history",          engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_dairy_production_system_monthly = importar_view(nome_view="vw_dairy_production_system_monthly", engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")

df_forage_cost                     = importar_view(nome_view="vw_forage_cost",                     engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_forage_cost_stage               = importar_view(nome_view="vw_forage_cost_stage",               engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_forage_production               = importar_view(nome_view="vw_forage_production",               engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_feeding_entries_needing_unit_price= importar_view(nome_view="vw_feeding_entries_needing_unit_price", engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")


Importando analytics_mart.vw_revenue...
Importando analytics_mart.vw_cattle...
Importando analytics_mart.vw_expense...
Importando analytics_mart.vw_feeding...
Importando analytics_mart.vw_labor...
Importando analytics_mart.vw_own_milk...
Importando analytics_mart.mvw_area_land_summary...
Importando analytics_mart.mvw_asset_payment_history...
Importando analytics_mart.vw_dairy_production_system_monthly...
Importando analytics_mart.vw_forage_cost...
Importando analytics_mart.vw_forage_cost_stage...
Importando analytics_mart.vw_forage_production...
Importando analytics_mart.vw_feeding_entries_needing_unit_price...


In [9]:
# ==============================================================================
# 3.1.1 CORREÇÃO E AGREGAÇÃO DA VIEW DE ATIVOS
# ==============================================================================
colunas_necessarias_ativos = [
    "id_property",
    "classification",
    "acquired_at",
    "reference_month",
    "monthly_depreciation",
    "monthly_average_capital_stock",
]

colunas_ausentes_ativos = [
    coluna
    for coluna in colunas_necessarias_ativos
    if coluna not in df_asset_payment_history.columns
]

if colunas_ausentes_ativos:
    raise KeyError(
        "Colunas ausentes em df_asset_payment_history: "
        f"{colunas_ausentes_ativos}"
    )

df_assets = df_asset_payment_history.copy()

df_assets["acquisition_month"] = (
    pd.to_datetime(df_assets["acquired_at"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)

df_assets["reference_month"] = (
    pd.to_datetime(df_assets["reference_month"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)

# Uma linha por mês na série do IGP-DI.
df_igpdi_assets = df_igpdi[["data", "igpdi"]].copy()
df_igpdi_assets["data"]  = (pd.to_datetime(df_igpdi_assets["data"], errors="coerce") .dt.to_period("M") .dt.to_timestamp())
df_igpdi_assets["igpdi"] = pd.to_numeric( df_igpdi_assets["igpdi"], errors="coerce", )
df_igpdi_assets = df_igpdi_assets.dropna(subset=["data", "igpdi"])

if df_igpdi_assets.duplicated("data").any():
    raise ValueError("Existem meses duplicados na série do IGP-DI.")

# Primeiro merge: índice do mês em que o ativo foi adquirido.
df_assets = df_assets.merge(
    df_igpdi_assets.rename(columns={"data": "acquisition_month", "igpdi": "igpdi_acquired"}),
    on="acquisition_month",
    how="left",
    validate="many_to_one",
)

DATA_CORTE_IGPDI = pd.Timestamp("1994-08-01")
mask_antes_serie = df_assets["acquisition_month"] < DATA_CORTE_IGPDI
df_assets.loc[mask_antes_serie, "igpdi_acquired"] = 100.0

# Segundo merge: índice do mês de referência da depreciação.
df_assets = df_assets.merge(
    df_igpdi_assets.rename(columns={"data": "reference_month", "igpdi": "igpdi_month"}),
    on="reference_month",
    how="left",
    validate="many_to_one",
)

sem_igpdi_ativo = (df_assets["igpdi_acquired"].isna() | df_assets["igpdi_month"].isna())

if sem_igpdi_ativo.any():
    print(f"⚠️ Itens sem IGP-DI na aquisição ou no mês de referência; será utilizado fator 1 em {sem_igpdi_ativo.sum():,} linhas." )

igpdi_valido = (df_assets["igpdi_acquired"].gt(0) & df_assets["igpdi_month"].gt(0) )

# Atualiza o valor da data de aquisição para o poder monetário do mês.
df_assets["asset_update_factor"] = 1.0
df_assets.loc[igpdi_valido, "asset_update_factor"] = ( df_assets.loc[igpdi_valido, "igpdi_month"] / df_assets.loc[igpdi_valido, "igpdi_acquired"] )

COLUNAS_MONETARIAS_ATIVOS = [
    "monthly_depreciation",
    "monthly_average_capital_stock",
]

for coluna in COLUNAS_MONETARIAS_ATIVOS:
    df_assets[coluna] = (pd.to_numeric(df_assets[coluna], errors="coerce") * df_assets["asset_update_factor"])

# Normaliza acentos e capitalização para consolidar as classificações.
df_assets["classification_normalized"] = (
    df_assets["classification"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
)

mascara_benfeitorias = df_assets["classification_normalized"].str.contains("benfeitoria|construcao|edificacao", na=False)
mascara_maquinas = ~mascara_benfeitorias

df_assets["monthly_depreciation_benfeitorias"]                     = (df_assets["monthly_depreciation"].where(mascara_benfeitorias, 0))
df_assets["monthly_depreciation_maquinas_e_equipamentos"]          = (df_assets["monthly_depreciation"].where(mascara_maquinas, 0))
df_assets["monthly_average_capital_stock_benfeitorias"]            = (df_assets["monthly_average_capital_stock"].where(mascara_benfeitorias, 0))
df_assets["monthly_average_capital_stock_maquinas_e_equipamentos"] = (df_assets["monthly_average_capital_stock"].where(mascara_maquinas, 0))

COLUNAS_MENSAIS_ATIVOS = [
    "monthly_depreciation_benfeitorias",
    "monthly_depreciation_maquinas_e_equipamentos",
    "monthly_average_capital_stock_benfeitorias",
    "monthly_average_capital_stock_maquinas_e_equipamentos",
]

df_asset_payment_history_monthly = (
    df_assets.loc[df_assets["reference_month"].ge(DATA_INICIAL) & df_assets["reference_month"].le(DATA_FINAL) ]
    .groupby(["id_property", "reference_month"], as_index=False)[ COLUNAS_MENSAIS_ATIVOS ]
    .sum()
    .sort_values(["id_property", "reference_month"])
    .reset_index(drop=True)
)

# Deflacionar 
df_asset_payment_history_monthly

print( "Ativos corrigidos e agrupados: " f"{df_asset_payment_history_monthly.shape[0]:,} linhas mensais." )

⚠️ Itens sem IGP-DI na aquisição ou no mês de referência; será utilizado fator 1 em 112,340 linhas.
Ativos corrigidos e agrupados: 41,050 linhas mensais.


In [10]:
# ==============================================================================
# 3.1.2 INSPEÇÃO INICIAL DOS DADOS DE ATIVOS
# ==============================================================================
df_asset_payment_history_monthly.head()


,id_property,reference_month,monthly_depreciation_benfeitorias,monthly_depreciation_maquinas_e_equipamentos,monthly_average_capital_stock_benfeitorias,monthly_average_capital_stock_maquinas_e_equipamentos
0,00220277-a58e-4b9e-9b89-e6acf8a1a400,2023-01-01,10848.612600,0.0,1.851697e+06,0.0
1,00220277-a58e-4b9e-9b89-e6acf8a1a400,2023-02-01,10852.501124,0.0,1.852361e+06,0.0
2,00220277-a58e-4b9e-9b89-e6acf8a1a400,2023-03-01,10815.379944,0.0,1.846025e+06,0.0
3,00220277-a58e-4b9e-9b89-e6acf8a1a400,2023-04-01,10705.818404,0.0,1.827324e+06,0.0
4,00220277-a58e-4b9e-9b89-e6acf8a1a400,2023-05-01,10456.393288,0.0,1.784751e+06,0.0


In [11]:
# ==============================================================================
# 3.1.3 PADRONIZAÇÃO DO DICIONÁRIO DADOS_VIEWS
# ==============================================================================
# Mantém compatibilidade com as funções de validação e merge existentes.
dados_views = {
    "vw_revenue": df_revenue,
    "vw_cattle": df_cattle,
    "vw_expense": df_expense,
    "vw_feeding": df_feeding,
    "vw_labor": df_labor,
    "vw_own_milk": df_own_milk,
    "mvw_area_land_summary": df_area,   # nome atualizado
    "mvw_asset_payment_history": df_asset_payment_history_monthly,
    "vw_dairy_production_system_monthly": df_dairy_production_system_monthly,
}

for nome_view, df_view in dados_views.items():
    print(
        f"{nome_view}: {df_view.shape[0]:,} linhas e "
        f"{df_view.shape[1]:,} colunas."
    )


vw_revenue: 15,736 linhas e 20 colunas.
vw_cattle: 15,849 linhas e 16 colunas.
vw_expense: 15,978 linhas e 21 colunas.
vw_feeding: 15,046 linhas e 11 colunas.
vw_labor: 15,728 linhas e 6 colunas.
vw_own_milk: 13,012 linhas e 11 colunas.
mvw_area_land_summary: 243,646 linhas e 11 colunas.
mvw_asset_payment_history: 41,050 linhas e 6 colunas.
vw_dairy_production_system_monthly: 30,386 linhas e 3 colunas.


In [12]:
# ==============================================================================
# 3.1.4 CÓPIA DE SEGURANÇA DOS DADOS DAS VIEWS
# ==============================================================================
CHAVES = ["id_property", "reference_month"]

resumo_duplicidades = []
exemplos_duplicidades = {}

for nome_view, df_original in dados_views.items():

    print(f"Verificando {nome_view}...")

    # Trabalhar com uma cópia para não alterar o dado bruto
    df = df_original.copy()

# ==============================================================================
    # TRATAMENTO DA VIEW DE ALIMENTAÇÃO
# ==============================================================================
    if nome_view == "vw_feeding":

        # A coluna unit não será utilizada
        df = df.drop(
            columns=["unit"],
            errors="ignore",
        )
    
# ==============================================================================
    # CONFERIR SE AS CHAVES EXISTEM
# ==============================================================================

    colunas_ausentes = [
        coluna
        for coluna in CHAVES
        if coluna not in df.columns
    ]

    if colunas_ausentes:

        resumo_duplicidades.append({
            "view": nome_view,
            "linhas_totais": len(df),
            "linhas_em_chaves_duplicadas": None,
            "chaves_duplicadas": None,
            "chave_unica": False,
            "status": f"Chaves ausentes: {colunas_ausentes}",
        })

        print(f"Não foi possível verificar {nome_view}. Colunas ausentes: {colunas_ausentes}" )

        continue

# ==============================================================================
    # PADRONIZAR AS CHAVES
# ==============================================================================

    df["id_property"] = ( df["id_property"] .astype("string") .str.strip() )
    df["reference_month"] = ( pd.to_datetime( df["reference_month"], errors="coerce", ) .dt.to_period("M") .dt.to_timestamp() )

# ==============================================================================
    # VERIFICAR DUPLICIDADES
# ==============================================================================

    mascara_duplicada = df.duplicated( subset=CHAVES, keep=False, )

    df_duplicados = ( df.loc[mascara_duplicada] .sort_values(CHAVES) .copy() )
    quantidade_linhas_duplicadas = len( df_duplicados )
    quantidade_chaves_duplicadas = ( df_duplicados[CHAVES] .drop_duplicates() .shape[0] )
    
    resumo_duplicidades.append({
        "view": nome_view,
        "linhas_totais": len(df),
        "linhas_em_chaves_duplicadas": ( quantidade_linhas_duplicadas ),
        "chaves_duplicadas": ( quantidade_chaves_duplicadas ),
        "chave_unica": ( quantidade_linhas_duplicadas == 0 ),
        "status": ( "OK" if quantidade_linhas_duplicadas == 0 else "Possui duplicidades" ),
    })

    if quantidade_linhas_duplicadas > 0:
        exemplos_duplicidades[nome_view] = ( df_duplicados.head(20) )
        print( f"{nome_view}: " f"{quantidade_chaves_duplicadas:,} " "chaves duplicadas.\n")

    else:
        print( f"{nome_view}: nenhuma duplicidade.\n")


# ==============================================================================
# RESULTADO
# ==============================================================================

df_resumo_duplicidades = (
    pd.DataFrame(resumo_duplicidades)
    .sort_values(by=["chave_unica", "view"], ascending=[True, True])
    .reset_index(drop=True)
)

display(df_resumo_duplicidades)


Verificando vw_revenue...
vw_revenue: nenhuma duplicidade.

Verificando vw_cattle...
vw_cattle: nenhuma duplicidade.

Verificando vw_expense...
vw_expense: nenhuma duplicidade.

Verificando vw_feeding...
vw_feeding: nenhuma duplicidade.

Verificando vw_labor...
vw_labor: nenhuma duplicidade.

Verificando vw_own_milk...
vw_own_milk: nenhuma duplicidade.

Verificando mvw_area_land_summary...
mvw_area_land_summary: 1 chaves duplicadas.

Verificando mvw_asset_payment_history...
mvw_asset_payment_history: nenhuma duplicidade.

Verificando vw_dairy_production_system_monthly...
vw_dairy_production_system_monthly: nenhuma duplicidade.



,view,linhas_totais,linhas_em_chaves_duplicadas,chaves_duplicadas,chave_unica,status
0,mvw_area_land_summary,243646,18693,1,False,Possui duplicidades
1,mvw_asset_payment_history,41050,0,0,True,OK
2,vw_cattle,15849,0,0,True,OK
3,vw_dairy_production_system_monthly,30386,0,0,True,OK
4,vw_expense,15978,0,0,True,OK
5,vw_feeding,15046,0,0,True,OK
6,vw_labor,15728,0,0,True,OK
7,vw_own_milk,13012,0,0,True,OK
8,vw_revenue,15736,0,0,True,OK


## 3.2 Tratamento Pré-Merge e Validação de Chaves


### 3.2.1 Funções Auxiliares de Tratamento e Validação


In [13]:
# ==============================================================================
# 3.2.1 FUNÇÕES AUXILIARES DE TRATAMENTO E VALIDAÇÃO
# ==============================================================================
def preparar_view(df: pd.DataFrame, nome_view: str) -> pd.DataFrame:
    """
    Prepara uma view mensal para os merges.
    
    - Padroniza id_property;
    - Padroniza o mês de referência;
    - Trata a view de área ativa;
    - Remove unit da view de alimentação;
    - Remove registros sem chave;
    - Filtra o período;
    - Ordena o resultado.
    """
    
    if df is None:
        raise ValueError( f"A view {nome_view} não foi importada." )

    # Trabalhar com uma cópia para preservar dados_views
    df = df.copy()
    
# ==============================================================================
    # TRATAMENTO ESPECÍFICO DA ÁREA ATIVA
# ==============================================================================
    if nome_view == "mvw_area_land_summary":

        colunas_hectares_owned = [
            "hectares_owned_benfeitorias_estradas",
            "hectares_owned_app_reserva_legal",
            "hectares_owned_forrageiras",
        ]

        colunas_hectares_rented = [
            "hectares_rented_benfeitorias_estradas",
            "hectares_rented_app_reserva_legal",
            "hectares_rented_forrageiras",
        ]

        colunas_raw_land_value = [
            "raw_land_value_benfeitorias_estradas",
            "raw_land_value_app_reserva_legal",
            "raw_land_value_forrageiras",
        ]

        # Área própria: soma das 3 categorias de hectares próprios
        df["hectares_propria"] = df[colunas_hectares_owned].sum(axis=1, skipna=True)

        # Área da atividade: tudo (próprio + arrendado) exceto Reserva Legal e APP
        df["hectares_atividade"] = (
            df["hectares_owned_benfeitorias_estradas"].fillna(0)
            + df["hectares_owned_forrageiras"].fillna(0)
            + df["hectares_rented_benfeitorias_estradas"].fillna(0)
            + df["hectares_rented_forrageiras"].fillna(0)
        )

        # Área total: soma de todas as 6 colunas (próprio + arrendado, todas categorias)
        df["hectares_total"] = (
            df[colunas_hectares_owned].sum(axis=1, skipna=True)
            + df[colunas_hectares_rented].sum(axis=1, skipna=True)
        )

        # Valor médio da terra: média ponderada pelas hectares próprios de cada categoria
        soma_valor_ponderado = sum(
            df[col_valor].fillna(0) * df[col_hectares].fillna(0)
            for col_valor, col_hectares in zip(colunas_raw_land_value, colunas_hectares_owned)
        )

        df["raw_land_value_medio_ponderado"] = (
            soma_valor_ponderado / df["hectares_propria"].replace(0, pd.NA)
        )
# ==============================================================================
    # TRATAMENTO ESPECÍFICO DA ALIMENTAÇÃO
# ==============================================================================
# ==============================================================================
    # CONFERIR AS CHAVES
# ==============================================================================
    colunas_ausentes = [
        coluna
        for coluna in CHAVES
        if coluna not in df.columns
    ]

    if colunas_ausentes:
        raise KeyError(f"A view {nome_view} não possui as colunas {colunas_ausentes}.")

# ==============================================================================
    # PADRONIZAR ID_PROPERTY
# ==============================================================================

    df["id_property"] = (df["id_property"] .astype("string") .str.strip())

    # Transformar texto vazio em ausente
    df["id_property"] = df["id_property"].replace( "", pd.NA)

# ==============================================================================
    # PADRONIZAR REFERENCE_MONTH
# ==============================================================================

    df["reference_month"] = (
        pd.to_datetime(df["reference_month"], errors="coerce")
        .dt.to_period("M")
        .dt.to_timestamp()
    )

# ==============================================================================
    # REMOVER LINHAS SEM CHAVE
# ==============================================================================

    registros_sem_chave = ( df[CHAVES] .isna() .any(axis=1) )
    quantidade_sem_chave = int( registros_sem_chave.sum() )

    if quantidade_sem_chave > 0:
        print(f"{nome_view}: removendo {quantidade_sem_chave:,} linhas sem chave." )
        df = df.loc[ ~registros_sem_chave ].copy()

# ==============================================================================
    # FILTRAR O PERÍODO
# ==============================================================================
    df = df.loc[ df["reference_month"].between(DATA_INICIAL, DATA_FINAL, inclusive="both") ].copy()

# ==============================================================================
    # ORGANIZAR O RESULTADO
# ==============================================================================
    df = (df.sort_values(CHAVES).reset_index(drop=True))

    return df

def verificar_chave_unica(df: pd.DataFrame, nome_view: str) -> None:
    """
    Verifica se existe mais de uma linha para a mesma combinação
    de id_property e reference_month.

    O processamento é interrompido caso existam duplicidades.
    """

    # Identificar todas as linhas que fazem parte de chaves duplicadas
    mascara_duplicadas = df.duplicated(subset=CHAVES, keep=False)

    df_duplicadas = (df.loc[mascara_duplicadas] .copy())

    if not df_duplicadas.empty:

        quantidade_linhas_duplicadas = len( df_duplicadas )
        quantidade_chaves_duplicadas = ( df_duplicadas[CHAVES] .drop_duplicates() .shape[0] )
        exemplos = ( df_duplicadas[CHAVES] .drop_duplicates() .sort_values(CHAVES) .head(10) )

        raise ValueError(
            f"\nA view {nome_view} possui duplicidades.\n"
            f"Linhas envolvidas: "
            f"{quantidade_linhas_duplicadas:,}\n"
            f"Chaves duplicadas: "
            f"{quantidade_chaves_duplicadas:,}\n\n"
            f"Exemplos:\n"
            f"{exemplos.to_string(index=False)}"
        )

    print(
        f"{nome_view}: chave única confirmada "
        f"em {len(df):,} linhas."
    )


### 3.2.2 Preparação Sequencial dos DataFrames


In [14]:
# ==============================================================================
# 3.2.2 PREPARAÇÃO SEQUENCIAL DOS DATAFRAMES
# ==============================================================================
dados_preparados = {}

for nome_view, df_bruto in dados_views.items():
    
    print(f"\nPreparando {nome_view}...")

    df_preparado = preparar_view( df=df_bruto, nome_view=nome_view)
    
    verificar_chave_unica( df=df_preparado, nome_view=nome_view)

    dados_preparados[nome_view] = ( df_preparado.copy())

print("\nTodas as views foram preparadas.")

print("\nViews disponíveis em dados_preparados:")

for nome_view in dados_preparados:
    print(f"- {nome_view}")



Preparando vw_revenue...
vw_revenue: chave única confirmada em 15,685 linhas.

Preparando vw_cattle...
vw_cattle: chave única confirmada em 15,778 linhas.

Preparando vw_expense...
vw_expense: chave única confirmada em 15,918 linhas.

Preparando vw_feeding...
vw_feeding: chave única confirmada em 14,995 linhas.

Preparando vw_labor...
vw_labor: chave única confirmada em 15,674 linhas.

Preparando vw_own_milk...
vw_own_milk: chave única confirmada em 12,965 linhas.

Preparando mvw_area_land_summary...
mvw_area_land_summary: removendo 18,693 linhas sem chave.
mvw_area_land_summary: chave única confirmada em 40,410 linhas.

Preparando mvw_asset_payment_history...
mvw_asset_payment_history: chave única confirmada em 41,050 linhas.

Preparando vw_dairy_production_system_monthly...
vw_dairy_production_system_monthly: chave única confirmada em 17,476 linhas.

Todas as views foram preparadas.

Views disponíveis em dados_preparados:
- vw_revenue
- vw_cattle
- vw_expense
- vw_feeding
- vw_labor

In [15]:
# ==============================================================================
# 3.2.3 VALIDAÇÃO DE UNICIDADE E INTEGRIDADE DAS CHAVES
# ==============================================================================
df_final = (
    dados_preparados[VIEW_BASE]
    .copy()
    .sort_values(CHAVES)
    .reset_index(drop=True)
)

quantidade_linhas_base = len(df_final)

print(f"\nBase revenue criada com " f"{quantidade_linhas_base:,} linhas.")

print(f"Propriedades: " f"{df_final['id_property'].nunique():,}")

print(f"Período: " f"{df_final['reference_month'].min():%Y-%m} " f"a {df_final['reference_month'].max():%Y-%m}")



Base revenue criada com 15,685 linhas.
Propriedades: 1,066
Período: 2023-01 a 2026-07


## 3.3 Execução dos Merges Sequenciais (LEFT JOIN)


In [16]:
# ==============================================================================
# 3.3 EXECUÇÃO DOS MERGES SEQUENCIAIS (LEFT JOIN)
# ==============================================================================
resumo_merge = []

for nome_view in VIEWS_MERGE:

    print(f"\nAdicionando {nome_view}...")

# ------------------------------------------------------------------------------
    # 1. Conferir se a view foi preparada
# ------------------------------------------------------------------------------

    if nome_view not in dados_preparados:
        raise KeyError(f"A view {nome_view} não foi encontrada " "em dados_preparados.")
    
    df_auxiliar = dados_preparados[nome_view].copy()

# ------------------------------------------------------------------------------
    # 2. Conferir se as chaves existem
# ------------------------------------------------------------------------------

    colunas_ausentes = [
        coluna
        for coluna in CHAVES
        if coluna not in df_auxiliar.columns
    ]

    if colunas_ausentes:
        raise KeyError(
            f"A view {nome_view} não possui as chaves "
            f"{colunas_ausentes}."
        )

# ------------------------------------------------------------------------------
    # 3. Conferir novamente se a chave é única
# ------------------------------------------------------------------------------

    duplicadas_auxiliar = df_auxiliar.duplicated(
        subset=CHAVES,
        keep=False,
    )

    if duplicadas_auxiliar.any():

        exemplos = (
            df_auxiliar.loc[
                duplicadas_auxiliar,
                CHAVES,
            ]
            .drop_duplicates()
            .sort_values(CHAVES)
            .head(10)
        )

        raise ValueError(
            f"A view {nome_view} possui duplicidades.\n\n"
            f"{exemplos.to_string(index=False)}"
        )

# ------------------------------------------------------------------------------
    # 4. Renomear colunas que já existem no df_final
# ------------------------------------------------------------------------------

    import re

    prefixo = re.sub(r"^(vw_|mvw_)", "", nome_view)

    colunas_conflitantes = [
        coluna
        for coluna in df_auxiliar.columns
        if coluna not in CHAVES
        and coluna in df_final.columns
    ]

    if colunas_conflitantes:

        df_auxiliar = df_auxiliar.rename(
            columns={
                coluna: f"{prefixo}_{coluna}"
                for coluna in colunas_conflitantes
            }
        )

        print("Colunas renomeadas:", colunas_conflitantes)

# ------------------------------------------------------------------------------
    # 5. Verificar a cobertura antes do merge
# ------------------------------------------------------------------------------

    cobertura = (
        df_final[CHAVES]
        .merge(
            df_auxiliar[CHAVES],
            on=CHAVES,
            how="left",
            indicator=True,
            validate="one_to_one",
        )
    )

    chaves_encontradas = int(
        cobertura["_merge"]
        .eq("both")
        .sum()
    )

    chaves_sem_correspondencia = int(
        cobertura["_merge"]
        .eq("left_only")
        .sum()
    )

    linhas_antes = len(df_final)

# ------------------------------------------------------------------------------
    # 6. Realizar o left merge
# ------------------------------------------------------------------------------

    df_final = df_final.merge(
        df_auxiliar,
        on=CHAVES,
        how="left",
        validate="one_to_one",
    )

    linhas_depois = len(df_final)

# ------------------------------------------------------------------------------
    # 7. Validar se a quantidade de linhas foi preservada
# ------------------------------------------------------------------------------

    if linhas_antes != linhas_depois:
        raise ValueError(
            f"O merge com {nome_view} alterou a quantidade "
            f"de linhas de {linhas_antes:,} para "
            f"{linhas_depois:,}."
        )

# ------------------------------------------------------------------------------
    # 8. Registrar o resumo
# ------------------------------------------------------------------------------

    cobertura_percentual = round(
        chaves_encontradas / linhas_antes * 100,
        2,
    )

    resumo_merge.append({
        "view": nome_view,
        "linhas_view": len(df_auxiliar),
        "chaves_encontradas": chaves_encontradas,
        "chaves_sem_correspondencia": (
            chaves_sem_correspondencia
        ),
        "cobertura_percentual": cobertura_percentual,
        "colunas_adicionadas": (
            len(df_auxiliar.columns)
            - len(CHAVES)
        ),
        "linhas_antes": linhas_antes,
        "linhas_depois": linhas_depois,
    })

    print( f"Correspondências: " f"{chaves_encontradas:,}" )
    print( f"Sem correspondência: " f"{chaves_sem_correspondencia:,}" )
    print( f"Cobertura: " f"{cobertura_percentual:.2f}%" )
    print( f"Linhas antes/depois: " f"{linhas_antes:,} / {linhas_depois:,}" )


# ==============================================================================
# ORGANIZAR O RESULTADO FINAL
# ==============================================================================

df_final = (
    df_final
    .sort_values(CHAVES)
    .reset_index(drop=True)
)


# ==============================================================================
# VALIDAÇÕES FINAIS
# ==============================================================================

assert len(df_final) == quantidade_linhas_base, (
    "Os merges alteraram a quantidade de linhas da revenue."
)

assert not df_final.duplicated(CHAVES).any(), (
    "A tabela final possui duplicidades por "
    "id_property e reference_month."
)


# ==============================================================================
# RESUMO DOS MERGES
# ==============================================================================

df_resumo_merge = pd.DataFrame(
    resumo_merge
)

print("\nTodos os merges foram concluídos com sucesso.")
print(f"Linhas da base revenue: {quantidade_linhas_base:,}" )
print(f"Linhas da tabela final: {df_final.shape[0]:,}" )
print(f"Colunas da tabela final: {df_final.shape[1]:,}" )
print(f"Duplicidades por propriedade e mês: {df_final.duplicated(CHAVES).sum()}")

display(df_resumo_merge)
display(df_final.head())



Adicionando vw_cattle...
Correspondências: 14,827
Sem correspondência: 858
Cobertura: 94.53%
Linhas antes/depois: 15,685 / 15,685

Adicionando vw_expense...
Correspondências: 15,406
Sem correspondência: 279
Cobertura: 98.22%
Linhas antes/depois: 15,685 / 15,685

Adicionando vw_feeding...
Correspondências: 14,638
Sem correspondência: 1,047
Cobertura: 93.32%
Linhas antes/depois: 15,685 / 15,685

Adicionando vw_labor...
Correspondências: 15,277
Sem correspondência: 408
Cobertura: 97.40%
Linhas antes/depois: 15,685 / 15,685

Adicionando vw_own_milk...
Colunas renomeadas: ['hired_labor_quantity', 'family_labor_quantity']
Correspondências: 12,827
Sem correspondência: 2,858
Cobertura: 81.78%
Linhas antes/depois: 15,685 / 15,685

Adicionando mvw_area_land_summary...
Correspondências: 15,333
Sem correspondência: 352
Cobertura: 97.76%
Linhas antes/depois: 15,685 / 15,685

Adicionando mvw_asset_payment_history...
Correspondências: 15,197
Sem correspondência: 488
Cobertura: 96.89%
Linhas antes/de

,view,linhas_view,chaves_encontradas,chaves_sem_correspondencia,cobertura_percentual,colunas_adicionadas,linhas_antes,linhas_depois
0,vw_cattle,15778,14827,858,94.53,14,15685,15685
1,vw_expense,15918,15406,279,98.22,19,15685,15685
2,vw_feeding,14995,14638,1047,93.32,9,15685,15685
3,vw_labor,15674,15277,408,97.40,4,15685,15685
4,vw_own_milk,12965,12827,2858,81.78,9,15685,15685
5,mvw_area_land_summary,40410,15333,352,97.76,13,15685,15685
6,mvw_asset_payment_history,41050,15197,488,96.89,4,15685,15685
7,vw_dairy_production_system_monthly,17476,10343,5342,65.94,1,15685,15685


,id_property,reference_month,milk_sold_revenue,milk_volume_sold,milk_unit_price,ccs,cpp,fat,protein,unit_price_derivative,...,raw_land_value_forrageiras,hectares_propria,hectares_atividade,hectares_total,raw_land_value_medio_ponderado,monthly_depreciation_benfeitorias,monthly_depreciation_maquinas_e_equipamentos,monthly_average_capital_stock_benfeitorias,monthly_average_capital_stock_maquinas_e_equipamentos,production_system
0,00220277-a58e-4b9e-9b89-e6acf8a1a400,2024-05-01,0.00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,10663.864045,0.000000,1.888052e+06,0.000000,NaN
1,00220277-a58e-4b9e-9b89-e6acf8a1a400,2024-08-01,501822.00,167274.0,3.00,159.0,33.0,3.85,3.35,NaN,...,NaN,NaN,NaN,NaN,NaN,10849.970268,0.000000,2.007212e+06,0.000000,NaN
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,112025.78,35677.0,3.14,421.0,77.0,3.48,3.40,NaN,...,NaN,0.0,24.0,25.18,<NA>,0.000000,1215.722920,0.000000e+00,48346.370420,UNSTRUCTURED_CONFINMENT
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,161358.96,54884.0,2.94,338.0,54.0,3.55,3.29,NaN,...,NaN,0.0,24.0,25.18,<NA>,0.000000,1205.387920,0.000000e+00,53990.366800,UNSTRUCTURED_CONFINMENT
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,157107.69,56311.0,2.79,192.0,102.0,3.46,3.32,NaN,...,NaN,0.0,24.0,25.18,<NA>,0.000000,1183.632817,0.000000e+00,58961.649491,UNSTRUCTURED_CONFINMENT


In [17]:
# ==============================================================================
# 3.3.1 RESUMO DE ESTRUTURA DO DATAFRAME INTEGRADO
# ==============================================================================
df_final.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15685 entries, 0 to 15684
Data columns (total 93 columns):
 #   Column                                                 Non-Null Count  Dtype         
---  ------                                                 --------------  -----         
 0   id_property                                            15685 non-null  string        
 1   reference_month                                        15685 non-null  datetime64[ns]
 2   milk_sold_revenue                                      15685 non-null  float64       
 3   milk_volume_sold                                       15685 non-null  float64       
 4   milk_unit_price                                        15457 non-null  float64       
 5   ccs                                                    14648 non-null  float64       
 6   cpp                                                    14648 non-null  float64       
 7   fat                                                    14650 non-nu

In [18]:
# ==============================================================================
# 3.3.2 LISTAGEM DE COLUNAS DA BASE INTEGRADA
# ==============================================================================
df_final.columns.to_list()


['id_property',
 'reference_month',
 'milk_sold_revenue',
 'milk_volume_sold',
 'milk_unit_price',
 'ccs',
 'cpp',
 'fat',
 'protein',
 'unit_price_derivative',
 'milk_volume_derivatives',
 'milk_derivatives_revenue',
 'received_loans',
 'animal_sale',
 'other_revenues',
 'price_bonus',
 'price_penalty',
 'voluminous_sold',
 'concentrated_sold',
 'surplus_division',
 'lactating_cows',
 'dry_cows',
 'nursing',
 'rearing',
 'males',
 'other_categories',
 'total_cows',
 'total_cattle',
 'lactating_cows_value',
 'dry_cows_value',
 'nursing_value',
 'rearing_value',
 'males_value',
 'other_categories_value',
 'general_expenses',
 'advance_payment',
 'administration',
 'land_lease',
 'technical_assistance',
 'animal_purchase',
 'land_purchase',
 'repairs',
 'loan_interest',
 'hormones',
 'taxes_fees',
 'medicines_vaccines',
 'bedding_replacement',
 'reproduction',
 'milk_replacer',
 'milking_material',
 'milk_calves',
 'energy',
 'fuel',
 'voluminous_purchased_quantity',
 'voluminous_consume

## 3.4 Aplicação da Correção Monetária (Deflação por IGP-DI)


In [19]:
# ==============================================================================
# 3.4 APLICAÇÃO DA CORREÇÃO MONETÁRIA (DEFLAÇÃO POR IGP-DI)
# ==============================================================================
# A deflação ocorre antes do Feature Engineering.
df_integrada = df_final.copy()

# ==============================================================================
# PREPARAR A BASE DO IGP-DI
# ==============================================================================

# Seleciona somente as colunas necessárias.
df_igpdi_aux = df_igpdi[[ "data", "deflator"] ].copy()

# Padroniza a data do IGP-DI para o primeiro dia de cada mês.
df_igpdi_aux["data"] = (
    pd.to_datetime(df_igpdi_aux["data"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)

# Converte o deflator para formato numérico.
df_igpdi_aux["deflator"] = pd.to_numeric(df_igpdi_aux["deflator"], errors="coerce")

# Remove linhas sem data válida.
df_igpdi_aux = df_igpdi_aux.dropna(subset=["data"])

# Verifica se existem meses duplicados na tabela do IGP-DI.
if df_igpdi_aux.duplicated("data").any():
    meses_duplicados = (
        df_igpdi_aux.loc[df_igpdi_aux.duplicated("data", keep=False), "data"]
        .dt.strftime("%Y-%m")
        .unique()
        .tolist()
    )
    raise ValueError( "Existem meses duplicados na base do IGP-DI: " f"{meses_duplicados}" )


# ==============================================================================
# PADRONIZAR O MÊS DA BASE INTEGRADA
# ==============================================================================

df_integrada["reference_month"] = (
    pd.to_datetime(df_integrada["reference_month"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)

# ==============================================================================
# ADICIONAR O DEFLATOR
# ==============================================================================

df_integrada = df_integrada.drop(
    columns=["data", "deflator"],
    errors="ignore",
)

df_integrada = df_integrada.merge(
    df_igpdi_aux,
    left_on="reference_month",
    right_on="data",
    how="left",
    validate="many_to_one",
)


# ==============================================================================
# VALIDAR O DEFLATOR
# ==============================================================================

meses_sem_deflator = (
    df_integrada.loc[ df_integrada["deflator"].isna(), "reference_month", ]
    .dropna()
    .dt.strftime("%Y-%m")
    .unique()
    .tolist()
)

if meses_sem_deflator:
    print(f"⚠️ Meses sem IGP-DI; o deflator 1 será utilizado: {meses_sem_deflator}" )

# Na ausência de IGP-DI, mantém o valor nominal usando deflator igual a 1.
df_integrada["deflator"] = df_integrada["deflator"].fillna(1.0)

if df_integrada["deflator"].le(0).any():
    raise ValueError( "Foram encontrados valores de deflator iguais ou inferiores a zero." )

# ==============================================================================
# DEFINIR AS COLUNAS MONETÁRIAS DE RECEITA
# ==============================================================================

COLUNAS_DEFLACIONAR_RECEITAS = [
    "milk_sold_revenue",         # Receita total da venda de leite.
    "milk_unit_price",           # Preço unitário do leite.
    "received_loans",            # Empréstimos recebidos.
    "animal_sale",               # Receita com venda de animais.
    "other_revenues",            # Outras receitas.
    "price_bonus",               # Bonificação do preço do leite.
    "price_penalty",             # Penalização ou desconto aplicado ao leite.
    "milk_derivatives_revenue",  # Receita total com derivados.
    "unit_price_derivative",     # Preço unitário dos derivados.
    "voluminous_sold",           # Receita com venda de volumoso.
    "concentrated_sold",          
    "surplus_division",          # Receita com divisão de sobras.
]

COLUNAS_DEFLACIONAR_DESPESAS = [
    "general_expenses",
    "advance_payment",
    "administration",
    "land_lease",
    "technical_assistance",
    "animal_purchase",
    "land_purchase",
    "repairs",
    "loan_interest",
    "hormones",
    "taxes_fees",
    "medicines_vaccines",
    "bedding_replacement",
    "reproduction",
    "milk_replacer",
    "milking_material",
    "milk_calves",
    "energy",
    "fuel",
]


COLUNAS_DEFLACIONAR_ANIMAIS = [
    "lactating_cows_value",
    "dry_cows_value",
    "nursing_value",
    "rearing_value",
    "males_value",
    "other_categories_value",
]

COLUNAS_DEFLACIONAR_MAQ_BEN = [
    'monthly_depreciation_benfeitorias',
    'monthly_depreciation_maquinas_e_equipamentos',
    'monthly_average_capital_stock_benfeitorias',
    'monthly_average_capital_stock_maquinas_e_equipamentos'
]

COLUNAS_DEFLACIONAR_FORRAGEIRA = [
    "pre_planting_cost",
    "planting_cost",
    "cultural_treatments_cost",
    "harvest_grain_cost",
    "harvest_whole_plant_cost",
    "total_forage_cost",
]

COLUNAS_DEFLACIONAR = (
    COLUNAS_DEFLACIONAR_RECEITAS
    + COLUNAS_DEFLACIONAR_DESPESAS
    + COLUNAS_DEFLACIONAR_MAQ_BEN
    # + COLUNAS_DEFLACIONAR_FORRAGEIRA
    # + COLUNAS_DEFLACIONAR_ANIMAIS
)

# ==============================================================================
# CONFERIR SE AS COLUNAS EXISTEM
# ==============================================================================

colunas_ausentes = [
    coluna
    for coluna in COLUNAS_DEFLACIONAR
    if coluna not in df_integrada.columns
]

if colunas_ausentes:
    raise KeyError(f"As seguintes colunas monetárias não foram encontradas: {colunas_ausentes}" )


# ==============================================================================
# DEFLACIONAR AS COLUNAS
# ==============================================================================

for coluna in COLUNAS_DEFLACIONAR:
    
    # Converte o valor monetário para número e substitui a própria coluna.
    df_integrada[coluna] = (
        pd.to_numeric(df_integrada[coluna], errors="coerce")
        / df_integrada["deflator"]
    )

# ==============================================================================
# ORGANIZAR A BASE
# ==============================================================================

# Remove a coluna auxiliar de data proveniente do IGP-DI. O deflator é mantido para rastreabilidade.
df_integrada = df_integrada.drop(columns=["data"], errors="ignore")

# Substitui eventuais infinitos por NaN.
df_integrada[COLUNAS_DEFLACIONAR] = (
    df_integrada[COLUNAS_DEFLACIONAR] .replace( [np.inf, -np.inf], np.nan)
)

# ==============================================================================
# CONFERÊNCIA
# ==============================================================================

display(
    df_integrada[
        [
            "id_property",
            "reference_month",
            "deflator",
            "milk_unit_price",
            "milk_sold_revenue",
            "general_expenses",
            "lactating_cows_value",
        ]
    ].head(20)
)

print("Deflação das receitas, despesas e valores dos animais concluída com sucesso.")

⚠️ Meses sem IGP-DI; o deflator 1 será utilizado: ['2026-07']


,id_property,reference_month,deflator,milk_unit_price,milk_sold_revenue,general_expenses,lactating_cows_value
0,00220277-a58e-4b9e-9b89-e6acf8a1a400,2024-05-01,0.925111,NaN,0.000000,NaN,NaN
1,00220277-a58e-4b9e-9b89-e6acf8a1a400,2024-08-01,0.938542,3.196448,534682.569406,21309.650410,10000.0
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,0.991500,3.166920,112986.205837,0.000000,0.0
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,0.983071,2.990629,164137.685128,2543.051919,0.0
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,0.965328,2.890209,162750.562487,0.000000,8000.0
5,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-07-01,0.964694,2.881742,201439.522025,0.000000,8000.0
6,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-08-01,0.966664,2.803456,201389.093719,728.071075,10000.0
7,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-09-01,0.970099,2.731680,189936.448857,110.813438,8000.0
8,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-10-01,0.969795,2.608798,179829.648529,0.000000,8000.0
9,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-11-01,0.969871,2.392071,145392.440798,0.000000,8000.0


Deflação das receitas, despesas e valores dos animais concluída com sucesso.


### 3.4.1 Deflação Específica dos Custos de Forrageira


In [20]:
# ==============================================================================
# 3.4.1 DEFLAÇÃO ESPECÍFICA DOS CUSTOS DE FORRAGEIRA
# ==============================================================================
# 1. Preparar a tabela auxiliar do IGP-DI
df_igpdi_aux = df_igpdi[["data", "deflator"]].copy()
df_igpdi_aux["data"] = (
    pd.to_datetime(df_igpdi_aux["data"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)
df_igpdi_aux["deflator"] = pd.to_numeric(df_igpdi_aux["deflator"], errors="coerce")
df_igpdi_aux = df_igpdi_aux.dropna(subset=["data"])

# ------------------------------------------------------------------------------
# 2. Criar df_forage_cost_deflacionado
# ------------------------------------------------------------------------------
if "df_forage_cost" in locals() and df_forage_cost is not None:
    df_forage_cost_deflacionado = df_forage_cost.copy()
    
    df_forage_cost_deflacionado["reference_month"] = (
        pd.to_datetime(df_forage_cost_deflacionado["reference_month"], errors="coerce")
        .dt.to_period("M")
        .dt.to_timestamp()
    )
    
    df_forage_cost_deflacionado = df_forage_cost_deflacionado.drop(columns=["data", "deflator"], errors="ignore")
    df_forage_cost_deflacionado = df_forage_cost_deflacionado.merge(
        df_igpdi_aux,
        left_on="reference_month",
        right_on="data",
        how="left"
    )
    df_forage_cost_deflacionado["deflator"] = df_forage_cost_deflacionado["deflator"].fillna(1.0)
    
    colunas_cost = [
        "pre_planting_cost",
        "planting_cost",
        "cultural_treatments_cost",
        "harvest_grain_cost",
        "harvest_whole_plant_cost",
        "total_forage_cost"
    ]
    
    for col in colunas_cost:
        if col in df_forage_cost_deflacionado.columns:
            df_forage_cost_deflacionado[col] = (
                pd.to_numeric(df_forage_cost_deflacionado[col], errors="coerce") 
                / df_forage_cost_deflacionado["deflator"]
            )
            
    # Remove as colunas auxiliares do novo DataFrame
    df_forage_cost_deflacionado = df_forage_cost_deflacionado.drop(columns=["data", "deflator"], errors="ignore")

# ------------------------------------------------------------------------------
# 3. Criar df_forage_cost_stage_deflacionado
# ------------------------------------------------------------------------------
if "df_forage_cost_stage" in locals() and df_forage_cost_stage is not None:
    df_forage_cost_stage_deflacionado = df_forage_cost_stage.copy()
    
    df_forage_cost_stage_deflacionado["reference_month"] = (
        pd.to_datetime(df_forage_cost_stage_deflacionado["reference_month"], errors="coerce")
        .dt.to_period("M")
        .dt.to_timestamp()
    )
    
    df_forage_cost_stage_deflacionado = df_forage_cost_stage_deflacionado.drop(columns=["data", "deflator"], errors="ignore")
    df_forage_cost_stage_deflacionado = df_forage_cost_stage_deflacionado.merge(
        df_igpdi_aux,
        left_on="reference_month",
        right_on="data",
        how="left"
    )
    df_forage_cost_stage_deflacionado["deflator"] = df_forage_cost_stage_deflacionado["deflator"].fillna(1.0)
    
    if "total_monthly_cost" in df_forage_cost_stage_deflacionado.columns:
        df_forage_cost_stage_deflacionado["total_monthly_cost"] = (
            pd.to_numeric(df_forage_cost_stage_deflacionado["total_monthly_cost"], errors="coerce") 
            / df_forage_cost_stage_deflacionado["deflator"]
        )
    
    # Remove as colunas auxiliares do novo DataFrame
    df_forage_cost_stage_deflacionado = df_forage_cost_stage_deflacionado.drop(columns=["data", "deflator"], errors="ignore")


In [21]:
# ==============================================================================
# 3.4.2 INSPEÇÃO DA PRODUÇÃO DE FORRAGEIRA
# ==============================================================================
df_forage_production


,id_property,id_production,id_planted_culture,id_culture,id_culture_harvest_product,reference_month,planted_at,harvest_date,planted_area,harvested_area,total_harvested_area,production,culture_name,harvest_product_name,feeding_category,yield_kg_per_ha
0,004cde8f-4688-4b22-acd9-76151b7dcc7a,26799627-0c4b-4814-b128-a8c0019257f7,7c02c790-1c7d-49b3-a5c1-ba76c75d246d,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,689761c8-f128-496c-88be-fa39f446b506,2026-02-01,2025-11-01 12:00:00,2026-02-01 12:00:00,22.00,22.00,22.00,1210.0,Milho,Milho (silagem),VOLUMOSO,55.00
1,0179ab93-910c-4aad-898b-a9006b2fd0da,6bf4c4dc-fa37-43b7-a306-0bc13b2542c5,7080bbe7-97e6-49bf-aa24-ee5cd64b4df2,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,689761c8-f128-496c-88be-fa39f446b506,2025-04-01,2024-12-04 08:00:00,2025-04-08 07:00:00,14.60,14.60,14.60,540.2,Milho,Milho (silagem),VOLUMOSO,37.00
2,01957b94-005f-473e-8dbd-7d569a9aed4a,8cfd7aab-b891-467b-a94a-6463d9e350eb,654f5858-cabf-40d0-88f9-69220e1a5675,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,689761c8-f128-496c-88be-fa39f446b506,2025-02-01,2024-10-28 07:00:00,2025-02-12 08:00:00,150.00,150.00,150.00,8250000.0,Milho,Milho (silagem),VOLUMOSO,55000.00
3,01957b94-005f-473e-8dbd-7d569a9aed4a,a1701e1a-7198-40b2-b5b7-a2f002d2ff82,f1688441-487b-44b8-9725-2957049874cf,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,689761c8-f128-496c-88be-fa39f446b506,2025-06-01,2025-02-11 08:00:00,2025-06-01 07:00:00,100.00,100.00,100.00,4000000.0,Milho,Milho (silagem),VOLUMOSO,40000.00
4,01f268e2-339c-464b-b0a4-c04f97d080ab,3eb51bbf-7a1c-42b6-9e5a-8fa629f0b9bd,c0b43fcd-b7ab-469d-825c-061b27fe3b53,9e5fa3d8-0bf3-46aa-8834-e8e9c22ec8e5,6e15e0f5-8817-4b42-b3b8-d793d5706803,2026-06-01,2024-03-01 12:00:00,2026-06-01 12:00:00,4.00,2.00,2.00,80640.0,Capim elefante,Capim-elefante (silagem),VOLUMOSO,40320.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1373,fd4d768c-165a-4465-9df5-a0f056c35781,c12262a2-f696-4ade-999a-5fab8d568cbc,1bd74ebb-15aa-4223-9e31-8aaddcdcfabc,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,689761c8-f128-496c-88be-fa39f446b506,2024-02-01,2023-11-07 08:00:00,2024-02-26 08:00:00,12.00,12.00,12.00,500000.0,Milho,Milho (silagem),VOLUMOSO,41666.67
1374,fd4d768c-165a-4465-9df5-a0f056c35781,e46199be-2371-4e6c-9ab5-91f7ff6c4e28,73780a3e-f911-4885-b39c-25497b8fd516,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,689761c8-f128-496c-88be-fa39f446b506,2024-02-01,2023-11-08 08:00:00,2024-02-28 08:00:00,1.00,1.00,1.00,45000.0,Milho,Milho (silagem),VOLUMOSO,45000.00
1375,fd4d768c-165a-4465-9df5-a0f056c35781,1380f805-947c-48d2-a735-71edf7dc1465,789ee692-4e55-4369-a994-42f5c44cb127,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,689761c8-f128-496c-88be-fa39f446b506,2025-03-01,2024-11-02 07:00:00,2025-03-01 08:00:00,16.52,16.52,16.52,580000.0,Milho,Milho (silagem),VOLUMOSO,35108.96
1376,ffb1465d-4d88-4488-a58c-972b2fe12b23,16537548-af88-4c22-9deb-c3d1552bc754,c876ede5-fad3-47b6-9148-d46b40f740b2,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,689761c8-f128-496c-88be-fa39f446b506,2025-02-01,2024-10-25 07:00:00,2025-02-10 08:00:00,30.00,30.00,30.00,1800000.0,Milho,Milho (silagem),VOLUMOSO,60000.00


In [22]:
# ==============================================================================
# 3.4.3 INSPEÇÃO DOS CUSTOS DEFLACIONADOS DE FORRAGEIRA
# ==============================================================================
df_forage_cost_deflacionado


,id_property,id_planted_culture,id_culture,id_area,reference_month,culture_name,pre_planting_cost,planting_cost,cultural_treatments_cost,harvest_grain_cost,harvest_whole_plant_cost,total_forage_cost
0,004cde8f-4688-4b22-acd9-76151b7dcc7a,7c02c790-1c7d-49b3-a5c1-ba76c75d246d,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,3ac0355d-8f4b-4a8e-8ac6-c503671de9e6,2025-08-01,Milho,1502.073317,0.000000,0.000000,0.0,0.000000,1502.073317
1,004cde8f-4688-4b22-acd9-76151b7dcc7a,7c02c790-1c7d-49b3-a5c1-ba76c75d246d,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,3ac0355d-8f4b-4a8e-8ac6-c503671de9e6,2025-10-01,Milho,0.000000,31288.043881,2903.107290,0.0,0.000000,34191.151172
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2f042645-1b00-4766-b456-f52c78b3e55e,c3adae35-3a13-441e-b543-ad9c4886de2f,4d8a982a-9d3e-4208-b723-7dedbf0493d7,2025-11-01,Brachiaria,0.000000,556.775044,0.000000,0.0,0.000000,556.775044
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,7c02c790-1c7d-49b3-a5c1-ba76c75d246d,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,3ac0355d-8f4b-4a8e-8ac6-c503671de9e6,2025-11-01,Milho,0.000000,0.000000,4706.605043,0.0,0.000000,4706.605043
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,7c02c790-1c7d-49b3-a5c1-ba76c75d246d,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,3ac0355d-8f4b-4a8e-8ac6-c503671de9e6,2026-01-01,Milho,0.000000,0.000000,4608.837162,0.0,0.000000,4608.837162
...,...,...,...,...,...,...,...,...,...,...,...,...
4788,ffe6cac6-7254-4c63-804d-58a4cc965c80,a3dd5771-0975-4192-b17a-0d678e430e4c,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,2fccfdbf-55b7-458f-a070-7d3f28dd2415,2025-03-01,Milho,0.000000,0.000000,0.000000,0.0,94277.266287,94277.266287
4789,ffe6cac6-7254-4c63-804d-58a4cc965c80,a3dd5771-0975-4192-b17a-0d678e430e4c,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,2fccfdbf-55b7-458f-a070-7d3f28dd2415,2025-05-01,Milho,0.000000,396.716099,28131.748935,0.0,0.000000,28528.465034
4790,ffe6cac6-7254-4c63-804d-58a4cc965c80,2276c335-78b4-47b7-95f5-f480821fea9c,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,2fccfdbf-55b7-458f-a070-7d3f28dd2415,2025-10-01,Milho,0.000000,91899.902834,0.000000,0.0,0.000000,91899.902834
4791,ffe6cac6-7254-4c63-804d-58a4cc965c80,2276c335-78b4-47b7-95f5-f480821fea9c,723fd24b-bca7-4d24-894f-69ceb6b2ad6b,2fccfdbf-55b7-458f-a070-7d3f28dd2415,2025-11-01,Milho,0.000000,0.000000,7689.682003,0.0,0.000000,7689.682003


In [23]:
# ==============================================================================
# 3.4.4 ENTRADAS DE ALIMENTAÇÃO QUE NECESSITAM PREÇO UNITÁRIO
# ==============================================================================
df_feeding_entries_needing_unit_price


,id_expense_entry,id_property,consumed_quantity_kg,current_unit_price,consumption_reference_month,id_production,id_planted_culture,id_culture_harvest_product
0,5a9830ec-8580-4261-aa51-dbb4f985c08f,027d7c06-0ab1-4a29-8383-98a7e2d36a80,264000.0,0.0,2026-02-01,2117cce1-cd22-4418-b2c2-38260563aaf2,6e3f8baf-dd38-4853-b826-1239a2124f48,689761c8-f128-496c-88be-fa39f446b506
1,e08c04cf-9506-4c16-9217-9a9a131e1902,027d7c06-0ab1-4a29-8383-98a7e2d36a80,250000.0,0.0,2025-05-01,0774fc0c-7b67-4218-8a6c-f393fc8df070,db9786a3-daad-4231-950f-3fb6e10093f3,689761c8-f128-496c-88be-fa39f446b506
2,0770ef9b-9c34-4653-951f-be04835fbd29,027d7c06-0ab1-4a29-8383-98a7e2d36a80,10000.0,0.0,2025-10-01,cce58b1a-6b97-49f8-b7f5-6ddb75a50abb,28c13363-90b2-420c-9994-092fe87f30c2,2ce71f82-8048-4208-af3d-bed7083274ca
3,083b173c-e771-40c4-bdd1-67f202fd2f05,027d7c06-0ab1-4a29-8383-98a7e2d36a80,10000.0,0.0,2025-09-01,cce58b1a-6b97-49f8-b7f5-6ddb75a50abb,28c13363-90b2-420c-9994-092fe87f30c2,2ce71f82-8048-4208-af3d-bed7083274ca
4,0f6e2c32-99b8-46fc-b3ff-5b4d80313814,027d7c06-0ab1-4a29-8383-98a7e2d36a80,10000.0,0.0,2025-08-01,cce58b1a-6b97-49f8-b7f5-6ddb75a50abb,28c13363-90b2-420c-9994-092fe87f30c2,2ce71f82-8048-4208-af3d-bed7083274ca
...,...,...,...,...,...,...,...,...
3471,4bbe7f4a-391a-430d-9274-049928d7eb11,ffe6cac6-7254-4c63-804d-58a4cc965c80,180000.0,0.0,2025-12-01,00c9dda9-7f9f-4493-aab1-b51f09e29669,a3dd5771-0975-4192-b17a-0d678e430e4c,689761c8-f128-496c-88be-fa39f446b506
3472,86cf7a18-bf8b-46c6-bb7f-e077957f3783,ffe6cac6-7254-4c63-804d-58a4cc965c80,185000.0,0.0,2026-02-01,00c9dda9-7f9f-4493-aab1-b51f09e29669,a3dd5771-0975-4192-b17a-0d678e430e4c,689761c8-f128-496c-88be-fa39f446b506
3473,db95f890-7c3d-4b60-9c37-e988fdeb8e5c,ffe6cac6-7254-4c63-804d-58a4cc965c80,185000.0,0.0,2026-03-01,00c9dda9-7f9f-4493-aab1-b51f09e29669,a3dd5771-0975-4192-b17a-0d678e430e4c,689761c8-f128-496c-88be-fa39f446b506
3474,ff93d818-cc9c-420c-9c42-24a7e18b6ec0,ffe6cac6-7254-4c63-804d-58a4cc965c80,310000.0,0.0,2025-05-01,00c9dda9-7f9f-4493-aab1-b51f09e29669,a3dd5771-0975-4192-b17a-0d678e430e4c,689761c8-f128-496c-88be-fa39f446b506


### 3.4.2 Rateio e Custo da Forrageira Própria na Alimentação


In [24]:
# ==============================================================================
# 3.4.2.1 CUSTO UNITÁRIO DA FORRAGEIRA PRODUZIDA (RATEIO POR ETAPA E ÁREA)
# ==============================================================================
# Regra reproduzida de ELABORE_IND_ANUAIS.ipynb, seção
# "Encontrar o Valor Unitário da Forrageira Produzida" (etapas 1 e 2).
#
# Entradas:
#   df_forage_cost_deflacionado -> id_property + reference_month + id_planted_culture
#   df_forage_production        -> uma linha por id_production
#
# Saída:
#   df_custo_producao_forrageira -> uma linha por id_property + id_production

# Colunas de custo de vw_forage_cost e a etapa correspondente.
MAPA_ETAPAS_CUSTO = {
    "pre_planting_cost":        "PRE_PLANTIO",
    "planting_cost":            "PLANTIO",
    "cultural_treatments_cost": "TRATOS_CULTURAIS",
    "harvest_grain_cost":       "COLHEITA_ENSILAGEM_GRAO",
    "harvest_whole_plant_cost": "COLHEITA_ENSILAGEM_PLANTA_INTEIRA",
}

CATEGORIA_VOLUMOSO    = "VOLUMOSO"
CATEGORIA_CONCENTRADO = "CONCENTRADO"


# ------------------------------------------------------------------------------
# 1. Custo por plantio e etapa
# ------------------------------------------------------------------------------
# Lançamento de custo sem plantio identificado não pode ser rateado. Na origem
# são os registros de CultureExpenseManagement sem id_culture informado.
custo_sem_plantio = df_forage_cost_deflacionado["id_planted_culture"].isna()

if custo_sem_plantio.any():
    print(
        f"Aviso: {int(custo_sem_plantio.sum()):,} linhas de custo sem "
        "id_planted_culture descartadas do rateio (R$ "
        f"{df_forage_cost_deflacionado.loc[custo_sem_plantio, 'total_forage_cost'].sum():,.2f})."
    )

# vw_forage_cost entrega o custo somado por etapa em colunas separadas.
# O melt volta ao formato longo e a soma consolida os meses de aplicação
# do insumo dentro do mesmo plantio.
df_custo_etapa = (
    df_forage_cost_deflacionado
    .loc[~custo_sem_plantio]
    .melt(
        id_vars=["id_property", "id_planted_culture"],
        value_vars=list(MAPA_ETAPAS_CUSTO),
        var_name="coluna_etapa",
        value_name="valor_total",
    )
)

df_custo_etapa["stage"] = df_custo_etapa["coluna_etapa"].map(MAPA_ETAPAS_CUSTO)

df_custo_etapa["valor_total"] = pd.to_numeric(
    df_custo_etapa["valor_total"],
    errors="coerce",
)

df_custo_etapa = (
    df_custo_etapa
    .groupby(
        ["id_property", "id_planted_culture", "stage"],
        as_index=False,
    )["valor_total"]
    .sum(min_count=1)
)


# ------------------------------------------------------------------------------
# 2. Produção: garantir uma linha por id_production
# ------------------------------------------------------------------------------
COLUNAS_PRODUCAO_RATEIO = [
    "id_property",
    "id_production",
    "id_planted_culture",
    "harvested_area",
    "total_harvested_area",
    "production",
    "feeding_category",
    "harvest_date",
]

producoes_duplicadas = df_forage_production.duplicated(
    subset=["id_property", "id_production"],
    keep=False,
)

if producoes_duplicadas.any():
    print(
        "Aviso: "
        f"{df_forage_production.loc[producoes_duplicadas, 'id_production'].nunique():,} "
        "id_production com mais de uma linha em vw_forage_production "
        "(fan-out do LEFT JOIN com CultureHarvestProduct). "
        "Mantida a primeira ocorrência."
    )

df_producao_rateio = (
    df_forage_production[COLUNAS_PRODUCAO_RATEIO]
    .drop_duplicates(subset=["id_property", "id_production"], keep="first")
    .copy()
)

df_producao_rateio["feeding_category"] = (
    df_producao_rateio["feeding_category"]
    .astype("string")
    .str.strip()
    .str.upper()
)

print(
    "Categorias encontradas na produção:",
    df_producao_rateio["feeding_category"].dropna().unique().tolist(),
)


# ------------------------------------------------------------------------------
# 3. Proporção da área colhida dentro do plantio
# ------------------------------------------------------------------------------
df_producao_rateio["prop_area_produzida"] = (
    pd.to_numeric(df_producao_rateio["harvested_area"], errors="coerce")
    / pd.to_numeric(
        df_producao_rateio["total_harvested_area"],
        errors="coerce",
    ).replace(0, np.nan)
)

sem_proporcao = df_producao_rateio["prop_area_produzida"].isna()

if sem_proporcao.any():
    print(
        f"Aviso: {int(sem_proporcao.sum()):,} produções sem área colhida total "
        "válida; o rateio por área fica indefinido (NaN)."
    )


# ------------------------------------------------------------------------------
# 4. Quantidade de produções por plantio e categoria
# ------------------------------------------------------------------------------
df_contagem_producoes = (
    df_producao_rateio
    .groupby(
        ["id_planted_culture", "feeding_category"],
        as_index=False,
        dropna=False,
    )["id_production"]
    .nunique()
    .rename(columns={"id_production": "quantidade_producoes"})
)

df_producao_rateio = df_producao_rateio.merge(
    df_contagem_producoes,
    on=["id_planted_culture", "feeding_category"],
    how="left",
)


# ------------------------------------------------------------------------------
# 5. Cruzar cada produção com as etapas de custo do plantio
# ------------------------------------------------------------------------------
# Expansão intencional: uma linha por produção x etapa.
df_rateio = df_producao_rateio.merge(
    df_custo_etapa,
    on=["id_property", "id_planted_culture"],
    how="left",
)


# ------------------------------------------------------------------------------
# 6. Fator de rateio
# ------------------------------------------------------------------------------
# Quando o plantio tem uma única produção da categoria, o custo de colheita
# vai integralmente para a categoria correspondente à etapa. Nos demais casos
# o custo é rateado pela proporção da área colhida.
def combinar_mascaras(*mascaras):
    """Combina máscaras booleanas anuláveis em um array booleano puro."""

    resultado = mascaras[0]

    for mascara in mascaras[1:]:
        resultado = resultado & mascara

    return resultado.fillna(False).to_numpy(dtype=bool)


condicoes_rateio = [
    combinar_mascaras(
        df_rateio["stage"].eq("COLHEITA_ENSILAGEM_PLANTA_INTEIRA"),
        df_rateio["feeding_category"].eq(CATEGORIA_VOLUMOSO),
        df_rateio["quantidade_producoes"].eq(1),
    ),
    combinar_mascaras(
        df_rateio["stage"].eq("COLHEITA_ENSILAGEM_PLANTA_INTEIRA"),
        df_rateio["feeding_category"].eq(CATEGORIA_CONCENTRADO),
        df_rateio["quantidade_producoes"].eq(1),
    ),
    combinar_mascaras(
        df_rateio["stage"].eq("COLHEITA_ENSILAGEM_GRAO"),
        df_rateio["feeding_category"].eq(CATEGORIA_CONCENTRADO),
        df_rateio["quantidade_producoes"].eq(1),
    ),
    combinar_mascaras(
        df_rateio["stage"].eq("COLHEITA_ENSILAGEM_GRAO"),
        df_rateio["feeding_category"].eq(CATEGORIA_VOLUMOSO),
        df_rateio["quantidade_producoes"].eq(1),
    ),
]

resultado_rateio = [1.0, 0.0, 1.0, 0.0]

df_rateio["fator_rateio"] = np.select(
    condicoes_rateio,
    resultado_rateio,
    default=df_rateio["prop_area_produzida"],
)

df_rateio["valor_total_rateado"] = (
    df_rateio["valor_total"] * df_rateio["fator_rateio"]
)


# ------------------------------------------------------------------------------
# 7. Custo unitário por produção
# ------------------------------------------------------------------------------
df_custo_producao_forrageira = (
    df_rateio
    .groupby(
        ["id_property", "id_production", "feeding_category", "production"],
        as_index=False,
        dropna=False,
    )["valor_total_rateado"]
    .sum(min_count=1)
)

df_custo_producao_forrageira["custo_unitario_forrageira"] = (
    df_custo_producao_forrageira["valor_total_rateado"]
    / pd.to_numeric(
        df_custo_producao_forrageira["production"],
        errors="coerce",
    ).replace(0, np.nan)
)

if df_custo_producao_forrageira.duplicated(
    subset=["id_property", "id_production"]
).any():
    raise ValueError(
        "df_custo_producao_forrageira não está com uma linha por "
        "id_property e id_production."
    )

print(
    f"\nProduções com custo unitário: "
    f"{int(df_custo_producao_forrageira['custo_unitario_forrageira'].notna().sum()):,} "
    f"de {len(df_custo_producao_forrageira):,}"
)

print("\nDistribuição do custo unitário (R$/kg):")
print(
    df_custo_producao_forrageira["custo_unitario_forrageira"]
    .describe()
    .round(4)
    .to_string()
)

# Valores absurdos indicam production em unidade divergente do custo.
# Conferir antes de aceitar o resultado: nada é filtrado automaticamente.
print("\nMaiores custos unitários (conferir antes de aceitar):")
display(
    df_custo_producao_forrageira
    .nlargest(10, "custo_unitario_forrageira")
    [
        [
            "id_property",
            "id_production",
            "feeding_category",
            "production",
            "valor_total_rateado",
            "custo_unitario_forrageira",
        ]
    ]
)


Aviso: 23 linhas de custo sem id_planted_culture descartadas do rateio (R$ 143,121.32).
Categorias encontradas na produção: ['VOLUMOSO', 'CONCENTRADO']
Aviso: 8 produções sem área colhida total válida; o rateio por área fica indefinido (NaN).

Produções com custo unitário: 1,105 de 1,378

Distribuição do custo unitário (R$/kg):
count    1.105000e+03
mean     3.348295e+03
std      1.076591e+05
min      0.000000e+00
25%      7.540000e-02
50%      1.192000e-01
75%      2.320000e-01
max      3.578419e+06

Maiores custos unitários (conferir antes de aceitar):


,id_property,id_production,feeding_category,production,valor_total_rateado,custo_unitario_forrageira
679,773ceb89-f6ae-4bac-8394-11904bee3862,c5a86dca-b731-429f-b11a-c7c6aec66900,VOLUMOSO,0.01,3.578455e+04,3.578419e+06
1191,d8b1aac4-a108-4278-bb29-0b0babc10888,3e71a78c-a3b7-4535-991a-746f634a9515,VOLUMOSO,260.00,1.265295e+07,4.866521e+04
418,46ca764b-5c1b-4d72-b6fe-757fe5d45abc,619af921-5d8a-4bb0-88a1-a2a965544dbf,CONCENTRADO,1.00,1.964442e+04,1.964442e+04
112,10fb6de6-7c13-4dda-9c0b-e8e01bddb02b,52f1cdb3-1b7b-4848-bc9c-ec5b0271cefb,VOLUMOSO,1.00,1.719761e+04,1.719761e+04
252,2c135321-99af-47f3-b03e-629d3ba40c55,102cee2b-f640-4caf-a4d9-96470939581c,VOLUMOSO,1.00,8.483413e+03,8.483413e+03
1203,dba03118-00ca-4806-9db2-67b0cff9347c,21ba52b8-9883-4df3-96ce-89569b2791ac,VOLUMOSO,1.00,6.834528e+03,6.834528e+03
1094,c555d241-bc85-4047-b79b-ea947220cba1,ac8aad5f-b4c7-4411-83b1-4dec41e3e468,VOLUMOSO,1.00,3.727729e+03,3.727729e+03
398,4610197a-071c-47c6-b11d-7e4fc74f70ec,fd5bc64f-af9c-4fe4-b08d-f480e00e258d,CONCENTRADO,3.00,7.762188e+03,2.587396e+03
655,704583a5-13d6-4e8b-a769-9eeda2de9abb,2116c8af-8f08-41ed-9019-316e208cbcfb,VOLUMOSO,784.00,7.418969e+05,9.462970e+02
1178,d53c389f-3eaf-4666-8362-19553fc48ffe,0f9c90f9-2856-4171-9482-f9416947ea91,VOLUMOSO,40.00,3.679063e+04,9.197657e+02


In [25]:
# ==============================================================================
# 3.4.2.2 CUSTO DA ALIMENTAÇÃO COM FORRAGEIRA PRÓPRIA (PREÇO UNITÁRIO = 0)
# ==============================================================================
# Entrada: df_feeding_entries_needing_unit_price
#   grão real: id_expense_entry x lote de estoque (id_stock_control_item)
# Saída: df_custo_alimentacao_forrageira
#   grão: id_property + reference_month

df_entradas_forrageira = df_feeding_entries_needing_unit_price.copy()


# ------------------------------------------------------------------------------
# 1. Padronizar a chave de produção
# ------------------------------------------------------------------------------
# Na view, id_production vem de StockControlMovement.source_reference_id,
# que é texto. Em df_forage_production a coluna é numérica.
def padronizar_id_producao(serie: pd.Series) -> pd.Series:
    """Devolve a chave de produção como texto comparável entre as views."""

    numerica = pd.to_numeric(serie, errors="coerce")

    if numerica.notna().all():
        return numerica.astype("Int64").astype("string")

    return serie.astype("string").str.strip()


df_entradas_forrageira["chave_producao"] = padronizar_id_producao(
    df_entradas_forrageira["id_production"]
)

df_custo_producao_forrageira["chave_producao"] = padronizar_id_producao(
    df_custo_producao_forrageira["id_production"]
)


# ------------------------------------------------------------------------------
# 2. Lançamento que consome mais de um lote de estoque
# ------------------------------------------------------------------------------
# A view repete consumed_quantity em cada lote consumido. Sem informação da
# divisão real entre lotes, a quantidade é dividida em partes iguais para não
# inflar o consumo total. Hipótese assumida, não é regra do sistema.
df_entradas_forrageira["quantidade_lotes"] = (
    df_entradas_forrageira
    .groupby("id_expense_entry")["id_expense_entry"]
    .transform("size")
)

lancamentos_multilote = df_entradas_forrageira["quantidade_lotes"].gt(1)

if lancamentos_multilote.any():
    print(
        "Aviso: "
        f"{df_entradas_forrageira.loc[lancamentos_multilote, 'id_expense_entry'].nunique():,} "
        "lançamentos consomem mais de um lote; a quantidade foi dividida "
        "igualmente entre os lotes."
    )

df_entradas_forrageira["consumed_quantity_rateada"] = (
    pd.to_numeric(df_entradas_forrageira["consumed_quantity_kg"], errors="coerce")
    / df_entradas_forrageira["quantidade_lotes"]
)


# ------------------------------------------------------------------------------
# 3. Trazer o custo unitário da produção
# ------------------------------------------------------------------------------
df_entradas_forrageira = df_entradas_forrageira.merge(
    df_custo_producao_forrageira[
        [
            "id_property",
            "chave_producao",
            "feeding_category",
            "custo_unitario_forrageira",
        ]
    ],
    on=["id_property", "chave_producao"],
    how="left",
    validate="m:1",
)

sem_custo = df_entradas_forrageira["custo_unitario_forrageira"].isna()

if sem_custo.any():
    print(
        f"Aviso: {int(sem_custo.sum()):,} de {len(df_entradas_forrageira):,} "
        "lançamentos ficaram sem custo unitário de forrageira."
    )

df_entradas_forrageira["custo_forrageira"] = (
    df_entradas_forrageira["consumed_quantity_rateada"]
    * df_entradas_forrageira["custo_unitario_forrageira"]
)


# ------------------------------------------------------------------------------
# 4. Mês de referência e coluna de destino
# ------------------------------------------------------------------------------
# Atenção: consumption_reference_month vem de created_at na view, e não da
# competência do lançamento (ExpenseEntry.reference_month) usada por vw_feeding.
df_entradas_forrageira["reference_month"] = (
    pd.to_datetime(
        df_entradas_forrageira["consumption_reference_month"],
        errors="coerce",
    )
    .dt.to_period("M")
    .dt.to_timestamp()
)

MAPA_CATEGORIA_COLUNA_FEEDING = {
    "VOLUMOSO":    "voluminous_amount_total",
    "CONCENTRADO": "concentrate_amount_total",
    "MINERAIS":    "mineral_amount_total",
}

COLUNAS_ALIMENTACAO_FORRAGEIRA = list(MAPA_CATEGORIA_COLUNA_FEEDING.values())

df_entradas_forrageira["coluna_destino"] = (
    df_entradas_forrageira["feeding_category"]
    .map(MAPA_CATEGORIA_COLUNA_FEEDING)
)

sem_destino = df_entradas_forrageira["coluna_destino"].isna()

if sem_destino.any():
    print(
        f"Aviso: {int(sem_destino.sum()):,} lançamentos sem categoria mapeada; "
        f"R$ {df_entradas_forrageira.loc[sem_destino, 'custo_forrageira'].sum():,.2f} "
        "ficaram de fora."
    )


# ------------------------------------------------------------------------------
# 5. Agregar por propriedade e mês
# ------------------------------------------------------------------------------
df_custo_alimentacao_forrageira = (
    df_entradas_forrageira.loc[~sem_destino]
    .pivot_table(
        index=CHAVES,
        columns="coluna_destino",
        values="custo_forrageira",
        aggfunc="sum",
    )
    .reset_index()
)

df_custo_alimentacao_forrageira.columns.name = None

# Ausência de consumo de forrageira própria na categoria significa custo
# adicional igual a zero, e não valor desconhecido.
for coluna in COLUNAS_ALIMENTACAO_FORRAGEIRA:

    if coluna not in df_custo_alimentacao_forrageira.columns:
        df_custo_alimentacao_forrageira[coluna] = 0.0

    df_custo_alimentacao_forrageira[coluna] = (
        df_custo_alimentacao_forrageira[coluna].fillna(0.0)
    )

df_custo_alimentacao_forrageira = df_custo_alimentacao_forrageira[
    CHAVES + COLUNAS_ALIMENTACAO_FORRAGEIRA
]

if df_custo_alimentacao_forrageira.duplicated(CHAVES).any():
    raise ValueError(
        "df_custo_alimentacao_forrageira possui duplicidades por "
        "id_property e reference_month."
    )

print(
    f"Linhas: {len(df_custo_alimentacao_forrageira):,} | "
    f"Propriedades: {df_custo_alimentacao_forrageira['id_property'].nunique():,}"
)

print(
    df_custo_alimentacao_forrageira[COLUNAS_ALIMENTACAO_FORRAGEIRA]
    .sum()
    .round(2)
    .to_string()
)

display(df_custo_alimentacao_forrageira.head())


Aviso: 20 lançamentos consomem mais de um lote; a quantidade foi dividida igualmente entre os lotes.
Aviso: 330 de 3,476 lançamentos ficaram sem custo unitário de forrageira.
Linhas: 3,042 | Propriedades: 307
voluminous_amount_total     2.169374e+08
concentrate_amount_total    1.357896e+06
mineral_amount_total        0.000000e+00


,id_property,reference_month,voluminous_amount_total,concentrate_amount_total,mineral_amount_total
0,027d7c06-0ab1-4a29-8383-98a7e2d36a80,2025-04-01,10484.848028,0.0,0.0
1,027d7c06-0ab1-4a29-8383-98a7e2d36a80,2025-05-01,10484.848028,0.0,0.0
2,027d7c06-0ab1-4a29-8383-98a7e2d36a80,2025-07-01,9771.878362,0.0,0.0
3,027d7c06-0ab1-4a29-8383-98a7e2d36a80,2025-08-01,10936.566687,0.0,0.0
4,027d7c06-0ab1-4a29-8383-98a7e2d36a80,2025-09-01,11649.536353,0.0,0.0


In [26]:
# ==============================================================================
# 3.4.2.3 CONCATENAÇÃO DO CUSTO DE FORRAGEIRA PRÓPRIA NA VW_FEEDING
# ==============================================================================
# df_feeding tem uma linha por id_property + reference_month. O append
# acrescenta as linhas de custo e reagrega pelas chaves, somando o custo da
# forrageira própria às colunas de alimentação já existentes.
#
# Serve para conferência. O que chega aos indicadores é a célula seguinte,
# porque df_feeding já foi consumida no merge das views.

df_feeding_corrigido = df_feeding.copy()

df_feeding_corrigido["reference_month"] = (
    pd.to_datetime(df_feeding_corrigido["reference_month"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)

chaves_novas = (
    df_custo_alimentacao_forrageira
    .merge(
        df_feeding_corrigido[CHAVES],
        on=CHAVES,
        how="left",
        indicator=True,
    )
    .query("_merge == 'left_only'")
)

if len(chaves_novas) > 0:
    print(
        f"Aviso: {len(chaves_novas):,} pares propriedade/mês do custo de "
        "forrageira não existem em vw_feeding e serão criados pelo append. "
        "Mede a divergência entre created_at e a competência do lançamento."
    )

COLUNAS_VALOR_FEEDING = [
    coluna
    for coluna in df_feeding_corrigido.columns
    if coluna not in CHAVES
]

df_feeding_corrigido = (
    pd.concat(
        [df_feeding_corrigido, df_custo_alimentacao_forrageira],
        ignore_index=True,
    )
    .groupby(CHAVES, as_index=False)[COLUNAS_VALOR_FEEDING]
    .sum(min_count=1)
)

if df_feeding_corrigido.duplicated(CHAVES).any():
    raise ValueError(
        "df_feeding_corrigido possui duplicidades por "
        "id_property e reference_month."
    )

print(
    f"df_feeding: {len(df_feeding):,} linhas | "
    f"df_feeding_corrigido: {len(df_feeding_corrigido):,} linhas"
)

display(df_feeding_corrigido.head())


df_feeding: 15,046 linhas | df_feeding_corrigido: 15,046 linhas


,id_property,reference_month,voluminous_purchased_quantity,voluminous_consumed_quantity,voluminous_amount_total,concentrate_purchased_quantity,concentrate_consumed_quantity,concentrate_amount_total,mineral_purchased_quantity,mineral_consumed_quantity,mineral_amount_total
0,00220277-a58e-4b9e-9b89-e6acf8a1a400,2024-08-01,77550.0,77550.0,9306.0,60246.0,60246.0,80346.36,2646.0,2646.0,10584.0
1,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,84000.0,84000.0,21184.8,16090.0,16090.0,42191.96,800.0,800.0,1692.8
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,90000.0,90000.0,22680.0,15596.0,13596.0,31288.75,0.0,0.0,0.0
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,90000.0,90000.0,22680.0,21115.0,23115.0,51860.44,450.0,450.0,2191.5
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-07-01,96000.0,96000.0,24192.0,35502.0,35502.0,74514.73,600.0,600.0,1314.0


In [27]:
# ==============================================================================
# 3.4.2.4 REPASSE DA CORREÇÃO PARA A BASE INTEGRADA
# ==============================================================================
# df_integrada já recebeu vw_feeding no merge das views. Sem esta etapa a
# correção fica apenas em df_feeding_corrigido e não chega aos indicadores
# calculados adiante (feeding_cost, feeding_cost_liter, COE etc.).

SUFIXO_FORRAGEIRA = "_forrageira_propria"

colunas_ja_aplicadas = [
    coluna
    for coluna in COLUNAS_ALIMENTACAO_FORRAGEIRA
    if f"{coluna}{SUFIXO_FORRAGEIRA}" in df_integrada.columns
]

if colunas_ja_aplicadas:
    raise RuntimeError(
        "A correção de forrageira já foi aplicada em df_integrada nesta "
        "execução. Reexecute a partir da célula de merge das views antes "
        "de repetir esta etapa."
    )

cobertura_forrageira = df_custo_alimentacao_forrageira.merge(
    df_integrada[CHAVES],
    on=CHAVES,
    how="left",
    indicator=True,
)

fora_da_base = int(cobertura_forrageira["_merge"].eq("left_only").sum())

if fora_da_base > 0:
    print(
        f"Aviso: {fora_da_base:,} pares propriedade/mês do custo de forrageira "
        "não existem em df_integrada e serão descartados no merge."
    )

df_integrada = df_integrada.merge(
    df_custo_alimentacao_forrageira.rename(
        columns={
            coluna: f"{coluna}{SUFIXO_FORRAGEIRA}"
            for coluna in COLUNAS_ALIMENTACAO_FORRAGEIRA
        }
    ),
    on=CHAVES,
    how="left",
    validate="one_to_one",
)

for coluna in COLUNAS_ALIMENTACAO_FORRAGEIRA:
    df_integrada[coluna] = (
        df_integrada[[coluna, f"{coluna}{SUFIXO_FORRAGEIRA}"]]
        .sum(axis=1, min_count=1)
    )

print("Custo de forrageira própria incorporado (R$):")

print(
    df_integrada[ [ f"{coluna}{SUFIXO_FORRAGEIRA}" for coluna in COLUNAS_ALIMENTACAO_FORRAGEIRA ] ]
    .sum()
    .round(2)
    .to_string()
)


Aviso: 49 pares propriedade/mês do custo de forrageira não existem em df_integrada e serão descartados no merge.
Custo de forrageira própria incorporado (R$):
voluminous_amount_total_forrageira_propria     2.165311e+08
concentrate_amount_total_forrageira_propria    1.357896e+06
mineral_amount_total_forrageira_propria        0.000000e+00


## 3.5 Engenharia de Recursos e Cálculo de Indicadores Mensais


In [28]:
# ==============================================================================
# 3.5 ENGENHARIA DE RECURSOS E CÁLCULO DE INDICADORES MENSAIS
# ==============================================================================
CHAVES = ["id_property", "reference_month"]

linhas_iniciais = len(df_integrada)

print(f"Linhas: {df_integrada.shape[0]:,}")
print(f"Colunas antes das flags: {df_integrada.shape[1]:,}")

# ==============================================================================
# 2. MAPA DAS VIEWS E COLUNAS DE PRESENÇA
# ==============================================================================
MAPA_PRESENCA = {
    "vw_cattle": "has_cattle_data",
    "vw_expense": "has_expense_data",
    "vw_feeding": "has_feeding_data",
    "vw_labor": "has_labor_data",
    "vw_own_milk": "has_own_milk_data",
    "mvw_asset_payment_history": "has_asset_data",
    "mvw_area_land_summary": "has_active_area_month",
    "vw_dairy_production_system_monthly": "has_dairy_production_system_data",
}

# A revenue é a base da tabela final. Portanto, todas as linhas possuem revenue.
df_integrada["has_revenue_data"] = 1

# ==============================================================================
# 3. CRIAR AS FLAGS DE PRESENÇA
# ==============================================================================
for nome_view, nome_flag in MAPA_PRESENCA.items():

    print(f"Criando indicador de presença: {nome_flag}")

    if nome_view not in dados_preparados:
        raise KeyError( f"A view {nome_view} não foi encontrada em dados_preparados." )

    # Evita duplicação caso a célula seja executada novamente
    df_integrada = df_integrada.drop(
        columns=[nome_flag],
        errors="ignore",
    )

    # Uma linha por propriedade e mês presente na view
    chaves_disponiveis = (
        dados_preparados[nome_view][CHAVES]
        .dropna(subset=CHAVES)
        .drop_duplicates(subset=CHAVES)
        .assign(**{nome_flag: 1})
    )

    linhas_antes = len(df_integrada)

    df_integrada = df_integrada.merge(
        chaves_disponiveis,
        on=CHAVES,
        how="left",
        validate="one_to_one",
    )
    
    linhas_depois = len(df_integrada)

    if linhas_antes != linhas_depois:
        raise ValueError(
            f"O merge da flag {nome_flag} alterou a quantidade de linhas: "
            f"{linhas_antes:,} para {linhas_depois:,}."
        )

    df_integrada[nome_flag] = (
        df_integrada[nome_flag]
        .fillna(0)
        .astype("int8")
    )

def somar_colunas_preservando_ausencia(df: pd.DataFrame, colunas: list[str]) -> pd.Series:
    """
    Soma as colunas informadas.
    Quando todas as colunas estiverem ausentes na linha, mantém o resultado como NaN em vez de transformar em zero.
    """

    colunas_ausentes = [
        coluna
        for coluna in colunas
        if coluna not in df.columns
    ]
    
    if colunas_ausentes:
        raise KeyError("Colunas necessárias não encontradas: " + ", ".join(colunas_ausentes))

    return df[colunas].sum(axis=1, min_count=1)

# ==============================================================================
# 4. ORGANIZAR AS COLUNAS DE PRESENÇA
# ==============================================================================

colunas_presenca = [
    "has_revenue_data",
    "has_cattle_data",
    "has_expense_data",
    "has_feeding_data",
    "has_labor_data",
    "has_own_milk_data",
    "has_asset_data",
    "has_active_area_month",
    "has_dairy_production_system_data"
]


# ==============================================================================
# 5. VALIDAR O RESULTADO
# ==============================================================================

assert len(df_integrada) == linhas_iniciais, ("A criação das flags alterou a quantidade de linhas.")

assert not df_integrada.duplicated(CHAVES).any(), ("Foram geradas duplicidades por id_property e reference_month.")

print("\nFlags de presença criadas com sucesso.")
print(f"Linhas finais: {df_integrada.shape[0]:,}")
print(f"Colunas finais: {df_integrada.shape[1]:,}")


# ==============================================================================
# 6. CALCULAR A COBERTURA DE CADA VIEW
# ==============================================================================
df_cobertura_views = (
    df_integrada[colunas_presenca]
    .mean()
    .mul(100)
    .round(2)
    .rename("coverage_percentage")
    .rename_axis("source")
    .reset_index()
)

display(df_cobertura_views)

# ==============================================================================
# 7. CALCULAR INDICADORES PADRÃO
# ==============================================================================

# Renda do leite consumido
df_integrada['discarded_quantity_milk_revenue']   = df_integrada['discarded_quantity'] * df_integrada['milk_unit_price']
df_integrada['hired_labor_quantity_milk_revenue'] = df_integrada['own_milk_hired_labor_quantity'] * df_integrada['milk_unit_price']
df_integrada['discarded_quantity_milk_revenue']   = df_integrada['discarded_quantity'] * df_integrada['milk_unit_price']
df_integrada['family_labor_milk_revenue']         = df_integrada['own_milk_family_labor_quantity'] * df_integrada['milk_unit_price']
df_integrada['calves_quantity_milk_revenue']      = df_integrada['calves_quantity'] * df_integrada['milk_unit_price']

# Renda do leite
df_integrada['total_milk_revenue'] = df_integrada[[
    'milk_sold_revenue',
    'milk_derivatives_revenue',
    'price_bonus',
    'discarded_quantity_milk_revenue',
    'hired_labor_quantity_milk_revenue',
    'family_labor_milk_revenue',
    'calves_quantity_milk_revenue'
]].sum(axis=1).round(2) - df_integrada['price_penalty']

# Leite produzido
df_integrada['milk_produced'] = df_integrada[[
    'milk_volume_sold',
    'milk_volume_derivatives',
    'discarded_quantity',
    'own_milk_hired_labor_quantity',
    'own_milk_family_labor_quantity',
    'calves_quantity']].sum(axis=1).round(2)

# Renda da Atividade
df_integrada['total_activity_revenue'] = (
    df_integrada[[
        'total_milk_revenue',
        'received_loans',
        'animal_sale',
        'other_revenues',
        'voluminous_sold',
        'concentrated_sold',
        'surplus_division',
    ]].sum(axis=1)
)
# Preço do Leite
df_integrada['milk_revenue_liter'] = df_integrada['total_milk_revenue'] / df_integrada['milk_produced']

# Leite diário
# Quantidade de dias do mês
df_integrada["days_in_month"] = ( df_integrada["reference_month"].dt.days_in_month)

# Produção média diária de leite
df_integrada["milk_daily"] = (df_integrada["milk_produced"] / df_integrada["days_in_month"].replace(0, np.nan))

# Produção diária por vaca em lactação
df_integrada["milk_lactating_cow_day"] = (df_integrada["milk_daily"] / df_integrada["lactating_cows"].replace(0, np.nan))

# Vacas em lactação sobre o total de vacas
df_integrada["lactating_cows_total_cows"] = (df_integrada["lactating_cows"] / df_integrada["total_cows"].replace(0, np.nan)) * 100

# Vacas em lactação sobre o total do rebanho
df_integrada["lactating_cows_total_cattle"] = (df_integrada["lactating_cows"] / df_integrada["total_cattle"].replace(0, np.nan) ) * 100

# Ajustar mão de obra para dias/homem
df_integrada['hired_labor_quantity']  = df_integrada['hired_labor_quantity'] / df_integrada["days_in_month"]
df_integrada['family_labor_quantity'] = df_integrada['family_labor_quantity'] / df_integrada["days_in_month"]

# Quantidade total de mão de obra
df_integrada["total_labor_quantity"] = ( df_integrada[[ "hired_labor_quantity", "family_labor_quantity"]] .sum(axis=1, min_count=1) )

# Produção diária por unidade de mão de obra total
df_integrada["milk_total_labor_day"] = (df_integrada["milk_daily"] / df_integrada["total_labor_quantity"].replace(0, np.nan))

# Vacas em lactação por unidade de mão de obra
df_integrada["lactating_cows_total_labor"] = (df_integrada["lactating_cows"] / df_integrada["total_labor_quantity"].replace(0, np.nan))

# Custo de concentrado e minerais
df_integrada["concentrate_mineral_cost"] = (df_integrada["concentrate_amount_total"].fillna(0) + df_integrada["mineral_amount_total"].fillna(0))

# Custo total da alimentação
df_integrada["feeding_cost"] = (df_integrada["voluminous_amount_total"].fillna(0) + df_integrada["concentrate_amount_total"].fillna(0) + df_integrada["mineral_amount_total"].fillna(0) )

# Quantidade de concentrado 
df_integrada["concentrate_mineral_quantity"] = (df_integrada["concentrate_purchased_quantity"].fillna(0) + df_integrada["mineral_purchased_quantity"].fillna(0))

# Custo da alimentação por litro
df_integrada["feeding_cost_liter"] = (df_integrada["feeding_cost"] / df_integrada["milk_produced"].replace(0, np.nan))

# Custo de volumoso por litro
df_integrada["voluminous_cost_liter"] = (df_integrada["voluminous_amount_total"] / df_integrada["milk_produced"].replace(0, np.nan))

# Custo de concentrado e minerais por litro
df_integrada["concentrate_mineral_cost_liter"] = (df_integrada["concentrate_mineral_cost"] / df_integrada["milk_produced"].replace(0, np.nan))

# Participação do custo da alimentação no preço do leite
df_integrada["feeding_cost_milk_price"] = (df_integrada["feeding_cost_liter"] / df_integrada["milk_revenue_liter"].replace(0, np.nan) ) * 100

# Custo total da mão de obra
df_integrada["total_labor_expenses"] = (df_integrada[["hired_labor_expenses", "family_labor_expenses"]] .sum(axis=1, min_count=1))

# Custo da mão de obra contratada por litro
df_integrada["hired_labor_cost_liter"] = (df_integrada["hired_labor_expenses"] / df_integrada["milk_produced"].replace(0, np.nan))

# Custo da mão de obra familiar por litro
df_integrada["family_labor_cost_liter"] = (df_integrada["family_labor_expenses"] / df_integrada["milk_produced"].replace(0, np.nan))

# Custo total da mão de obra por litro
df_integrada["total_labor_cost_liter"] = (df_integrada["total_labor_expenses"] / df_integrada["milk_produced"].replace(0, np.nan))

# Participação da mão de obra na receita do leite
df_integrada["labor_cost_milk_revenue"] = ( df_integrada["total_labor_expenses"] / df_integrada["total_milk_revenue"].replace(0, np.nan) ) * 100

# Agregação das demais despesas operacionais
COLUNAS_OUTRAS_DESPESAS = [
    "milk_replacer",
    "milking_material",
    "reproduction",
    "hormones",
    "medicines_vaccines",
    "technical_assistance",
    "taxes_fees",
    "land_lease",
    "repairs",
    "administration",
    "general_expenses",
    "bedding_replacement",
]

df_integrada["other_operating_expenses"] = (somar_colunas_preservando_ausencia(df=df_integrada, colunas=COLUNAS_OUTRAS_DESPESAS) )

# Estoque de capital de animais
df_integrada["animal_capital_stock"] = (df_integrada["lactating_cows"] * df_integrada["lactating_cows_value"]) + (df_integrada["dry_cows"] * df_integrada["dry_cows_value"]) + (df_integrada["nursing"] * df_integrada["nursing_value"]) + (df_integrada["rearing"] * df_integrada["rearing_value"]) + (df_integrada["males"] * df_integrada["males_value"]) + ((df_integrada["other_categories"] * df_integrada["other_categories_value"])/2)

# Estoque de capital da terra
df_integrada["land_capital_stock"] = df_integrada["hectares_propria"] * df_integrada["raw_land_value_medio_ponderado"]

# Define os componentes do estoque de capital fixo.
COLUNAS_CAPITAL_FIXO = [
    "monthly_average_capital_stock_benfeitorias",
    "monthly_average_capital_stock_maquinas_e_equipamentos",
]

# Soma o estoque médio de benfeitorias com o estoque médio de máquinas e equipamentos.
#
# min_count=2 exige que os dois valores estejam disponíveis.
df_integrada["fixed_capital_stock"] = (
    df_integrada[COLUNAS_CAPITAL_FIXO]
    .sum(
        axis=1,
        min_count=len(COLUNAS_CAPITAL_FIXO),
    )
)

# Estoque de capital total
COLUNAS_CAPITAL_TOTAL = [
    "animal_capital_stock",
    "land_capital_stock",
    "fixed_capital_stock",
]

df_integrada["total_capital_stock"] = (
    df_integrada[COLUNAS_CAPITAL_TOTAL]
    .sum(axis=1)
)

# Calcula o estoque de capital total por litro de produção diária.
#
# Esse indicador é o equivalente novo de:
# estoqueCapitalcomTerra_leiteDiario.
#
# A produção diária igual a zero é substituída por NaN
# para evitar divisão por zero.
df_integrada["total_capital_stock_milk_daily"] = (
    df_integrada["total_capital_stock"]
    / df_integrada["milk_daily"].replace(0, np.nan)
)


# Substitui eventuais resultados infinitos por NaN.
df_integrada[
    [
        "total_capital_stock",
        "total_capital_stock_milk_daily",
    ]
] = (
    df_integrada[
        [
            "total_capital_stock",
            "total_capital_stock_milk_daily",
        ]
    ]
    .replace([np.inf, -np.inf], np.nan)
)


# ==============================================================================
# INDICADORES DE ÁREA
# ==============================================================================

# Vacas em lactação por hectare de atividade
df_integrada["lactating_cows_hectare_activity"] = (df_integrada["lactating_cows"] / df_integrada["hectares_atividade"].replace(0, np.nan))

# Percentual da área arrendada
df_integrada["hectares_arrendada"] = df_integrada[[
    "hectares_rented_benfeitorias_estradas",
    "hectares_rented_app_reserva_legal",
    "hectares_rented_forrageiras",
]].sum(axis=1, skipna=True)

df_integrada["rented_area_percentage"] = (
    df_integrada["hectares_arrendada"] / df_integrada["hectares_total"].replace(0, np.nan) * 100
)

df_integrada.head(20)


Linhas: 15,685
Colunas antes das flags: 97
Criando indicador de presença: has_cattle_data
Criando indicador de presença: has_expense_data
Criando indicador de presença: has_feeding_data
Criando indicador de presença: has_labor_data
Criando indicador de presença: has_own_milk_data
Criando indicador de presença: has_asset_data
Criando indicador de presença: has_active_area_month
Criando indicador de presença: has_dairy_production_system_data

Flags de presença criadas com sucesso.
Linhas finais: 15,685
Colunas finais: 106


,source,coverage_percentage
0,has_revenue_data,100.00
1,has_cattle_data,94.53
2,has_expense_data,98.22
3,has_feeding_data,93.32
4,has_labor_data,97.40
5,has_own_milk_data,81.78
6,has_asset_data,96.89
7,has_active_area_month,97.76
8,has_dairy_production_system_data,65.94


C:\Users\Guilherme\AppData\Local\Temp\ipykernel_33040\2233461704.py:327: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace([np.inf, -np.inf], np.nan)


,id_property,reference_month,milk_sold_revenue,milk_volume_sold,milk_unit_price,ccs,cpp,fat,protein,unit_price_derivative,...,labor_cost_milk_revenue,other_operating_expenses,animal_capital_stock,land_capital_stock,fixed_capital_stock,total_capital_stock,total_capital_stock_milk_daily,lactating_cows_hectare_activity,hectares_arrendada,rented_area_percentage
0,00220277-a58e-4b9e-9b89-e6acf8a1a400,2024-05-01,0.000000,0.00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2.040893e+06,2.040893e+06,NaN,NaN,0.00,NaN
1,00220277-a58e-4b9e-9b89-e6acf8a1a400,2024-08-01,534682.569406,167274.00,3.196448,159.0,33.0,3.85,3.35,NaN,...,5.584912,45815.748382,2790000.0,NaN,2.138649e+06,4.928649e+06,834.114661,NaN,0.00,NaN
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,112986.205837,35677.00,3.166920,421.0,77.0,3.48,3.40,NaN,...,13.839742,12023.332915,0.0,<NA>,4.876086e+04,4.876086e+04,41.001925,3.500000,25.18,100.0
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,164137.685128,54884.00,2.990629,338.0,54.0,3.55,3.29,NaN,...,13.587017,22924.127904,0.0,<NA>,5.492012e+04,5.492012e+04,30.324234,3.250000,25.18,100.0
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,162750.562487,56311.00,2.890209,192.0,102.0,3.46,3.32,NaN,...,13.817101,11558.215377,862500.0,<NA>,6.107939e+04,9.235794e+05,466.682078,3.375000,25.18,100.0
5,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-07-01,201439.522025,69902.00,2.881742,116.0,38.0,3.39,3.25,NaN,...,8.865863,37483.169287,868000.0,<NA>,6.723865e+04,9.352387e+05,407.070825,3.500000,25.18,100.0
6,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-08-01,201389.093719,71836.00,2.803456,132.0,41.0,3.47,3.24,NaN,...,10.174148,104447.516351,1160000.0,<NA>,7.339792e+04,1.233398e+06,505.223778,3.375000,25.18,100.0
7,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-09-01,189936.448857,69531.00,2.731680,112.0,3.0,3.51,3.20,NaN,...,9.179044,48305.340294,840000.0,<NA>,7.339792e+04,9.133979e+05,368.053319,3.541667,25.18,100.0
8,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-10-01,179829.648529,68932.00,2.608798,145.0,8.0,3.58,3.25,NaN,...,9.747324,29787.727372,805000.0,<NA>,7.339792e+04,8.783979e+05,371.005716,2.962963,28.18,100.0
9,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-11-01,145392.440798,60781.00,2.392071,137.0,3.0,3.50,3.27,NaN,...,12.519860,68583.539669,760000.0,<NA>,7.750410e+04,8.375041e+05,398.995140,2.592593,28.18,100.0


In [29]:
# ==============================================================================
# 3.5.1 LISTAGEM DE COLUNAS DA BASE INTEGRADA FINAL
# ==============================================================================
df_integrada.columns.tolist()  


['id_property',
 'reference_month',
 'milk_sold_revenue',
 'milk_volume_sold',
 'milk_unit_price',
 'ccs',
 'cpp',
 'fat',
 'protein',
 'unit_price_derivative',
 'milk_volume_derivatives',
 'milk_derivatives_revenue',
 'received_loans',
 'animal_sale',
 'other_revenues',
 'price_bonus',
 'price_penalty',
 'voluminous_sold',
 'concentrated_sold',
 'surplus_division',
 'lactating_cows',
 'dry_cows',
 'nursing',
 'rearing',
 'males',
 'other_categories',
 'total_cows',
 'total_cattle',
 'lactating_cows_value',
 'dry_cows_value',
 'nursing_value',
 'rearing_value',
 'males_value',
 'other_categories_value',
 'general_expenses',
 'advance_payment',
 'administration',
 'land_lease',
 'technical_assistance',
 'animal_purchase',
 'land_purchase',
 'repairs',
 'loan_interest',
 'hormones',
 'taxes_fees',
 'medicines_vaccines',
 'bedding_replacement',
 'reproduction',
 'milk_replacer',
 'milking_material',
 'milk_calves',
 'energy',
 'fuel',
 'voluminous_purchased_quantity',
 'voluminous_consume

In [30]:
# ==============================================================================
# 3.5.2 INSPEÇÃO DO ESTOQUE DE CAPITAL ANIMAL
# ==============================================================================
df_integrada[['animal_capital_stock']].head()


,animal_capital_stock
0,NaN
1,2790000.0
2,0.0
3,0.0
4,862500.0


## 3.6 Integração da Dimensão Produtor (SharePoint)


In [31]:
# ==============================================================================
# 3.6 INTEGRAÇÃO DA DIMENSÃO PRODUTOR (SHAREPOINT)
# ==============================================================================
from sqlalchemy import text

# ==============================================================================
# IMPORTAR APENAS AS COLUNAS DIMENSIONAIS NECESSÁRIAS
# ==============================================================================
consulta_dim_property = text(
    """
    SELECT
        id_property,
        property_name,
        labor_rural_code,
        entrepreneur_name,
        agroindustry_name,
        dairy_region,
        property_status
    FROM analytics_mart.vw_dim_property; 
    """
)

df_dim_property = pd.read_sql_query(consulta_dim_property, con=engine)

# Padronizar a chave
df_dim_property["id_property"] = (df_dim_property["id_property"].astype("string").str.strip())

# Remover linhas sem chave
df_dim_property = (df_dim_property.dropna(subset=["id_property"]).reset_index(drop=True) )

# Validar uma linha por propriedade
if df_dim_property.duplicated("id_property").any():
    raise ValueError( "A dimensão possui mais de uma linha por id_property." )

# Torna o merge idempotente: remove versões anteriores das colunas dimensionais.
colunas_dimensionais = [
    coluna
    for coluna in df_dim_property.columns
    if coluna != "id_property"
]

colunas_dimensionais_antigas = [
    nome
    for coluna in colunas_dimensionais
    for nome in (coluna, f"{coluna}_x", f"{coluna}_y")
    if nome in df_integrada.columns
]

df_integrada = df_integrada.drop(
    columns=colunas_dimensionais_antigas,
    errors="ignore",
)

# Merge dimensional
linhas_antes = len(df_integrada)

df_integrada = df_integrada.merge(
    df_dim_property,
    on="id_property",
    how="left"
    # validate="many_to_one",
)

if len(df_integrada) != linhas_antes:
    raise ValueError( "O merge dimensional alterou a quantidade de linhas." )

# Opção A: Remover lançamentos históricos de fazendas inativas (não presentes em vw_dim_property)
inativas_descartadas = df_integrada["property_name"].isna().sum()
if inativas_descartadas > 0:
    print(f"Filtrando apenas fazendas ativas: {inativas_descartadas:,} linhas de fazendas inativas foram removidas.")
    df_integrada = df_integrada.dropna(subset=["property_name"]).reset_index(drop=True)

# Rótulo combinado fazenda - produtor
if {"property_name", "entrepreneur_name"}.issubset(df_integrada.columns):
    df_integrada["property_entrepreneur_label"] = (
        df_integrada["property_name"].fillna("") + " - " + df_integrada["entrepreneur_name"].fillna("")
    )

print("Merge dimensional concluído.")
print(f"Linhas: {df_integrada.shape[0]:,}")
print(f"Colunas: {df_integrada.shape[1]:,}")

df_integrada.head()

# ==============================================================================
# CONSULTOR: IMPORTAR AQUI, MAS NÃO MESCLAR AGORA
# ==============================================================================
# Uma propriedade pode ter mais de um consultor em vw_dim_property_consultant.
# Se mesclássemos isso em df_integrada agora (grão mensal), cada mês de uma
# fazenda com 2+ consultores viraria 2+ linhas ANTES da Feature Engineering e
# da janela anual - o que faria toda soma anual (COE, receita, produção etc.)
# ser contada 2x, 3x etc. Por isso só importamos aqui; o merge (que duplica de
# propósito) acontece só no final, em df_anuais, depois que as somas já estão
# fechadas - ver célula anuais-indicadores.
# A view vw_dim_property_consultant está em analytics_mart.
consulta_dim_consultor = text(
    """
    SELECT
        id_consultant,
        consultant_name,
        id_property
    FROM analytics_mart.vw_dim_property_consultant;
    """
)

df_dim_consultor = pd.read_sql_query(consulta_dim_consultor, con=engine)
df_dim_consultor["id_property"] = (df_dim_consultor["id_property"].astype("string").str.strip())
df_dim_consultor = (df_dim_consultor.dropna(subset=["id_property"]).reset_index(drop=True) )

print(f"Consultores importados: {df_dim_consultor.shape[0]:,} linhas.")
print(f"Propriedades com mais de um consultor: {df_dim_consultor['id_property'].duplicated().sum():,}")

# Agregação de consultores por propriedade para o relatório mensal (evita duplicação de linhas)
if not df_dim_consultor.empty:
    df_consultor_agg = (
        df_dim_consultor.groupby("id_property")["consultant_name"]
        .apply(lambda s: ", ".join(s.dropna().unique()))
        .reset_index()
    )
    df_integrada = df_integrada.drop(columns=["consultant_name"], errors="ignore")
    df_integrada["id_property"] = df_integrada["id_property"].astype("string").str.strip()
    df_consultor_agg["id_property"] = df_consultor_agg["id_property"].astype("string").str.strip()
    df_integrada = df_integrada.merge(df_consultor_agg, on="id_property", how="left")


Filtrando apenas fazendas ativas: 2,557 linhas de fazendas inativas foram removidas.
Merge dimensional concluído.
Linhas: 13,128
Colunas: 150
Consultores importados: 985 linhas.
Propriedades com mais de um consultor: 152


## 3.7 Regras Mensais de Consistência e Qualidade dos Dados


In [32]:
# ==============================================================================
# 3.7 REGRAS MENSAIS DE CONSISTÊNCIA E QUALIDADE DOS DADOS
# ==============================================================================
df_consistencia = df_integrada.copy(deep=True)

# ==============================================================================
# 1. DEFINIR AS COLUNAS NECESSÁRIAS
# ==============================================================================

# Lista das colunas que precisam existir antes de calcular os indicadores e as regras de consistência.
COLUNAS_NECESSARIAS_CONSISTENCIA = [
    "ccs",
    "cpp",
    "fat",
    "protein",
    "lactating_cows",
    "lactating_cows_total_cows",
    "lactating_cows_total_cattle",
    "milk_daily",
    "milk_total_labor_day",
    "lactating_cows_total_labor",
    "feeding_cost_milk_price",
    "voluminous_cost_liter",
    "concentrate_mineral_cost_liter",
    "hired_labor_cost_liter",
    # "total_capital_stock_milk_daily"
]


# Verifica quais colunas da lista acima não existem na df_consistencia.
colunas_ausentes = [
    coluna
    for coluna in COLUNAS_NECESSARIAS_CONSISTENCIA
    if coluna not in df_consistencia.columns
]

# Interrompe a execução caso alguma coluna obrigatória esteja ausente.
if colunas_ausentes:
    raise KeyError( "As seguintes colunas necessárias para a consistência " f"não foram encontradas: {colunas_ausentes}" )


# ==============================================================================
# 2. GARANTIR QUE AS COLUNAS SEJAM NUMÉRICAS
# ==============================================================================

# Percorre cada coluna usada nas regras de consistência.
for coluna in COLUNAS_NECESSARIAS_CONSISTENCIA:
    
    # Converte a coluna para número.
    # Valores que não puderem ser convertidos serão transformados em NaN.
    df_consistencia[coluna] = pd.to_numeric( df_consistencia[coluna], errors="coerce", )


# Substitui valores infinitos positivos e negativos por NaN.
# Isso impede que divisões inválidas sejam classificadas como consistentes.
df_consistencia[COLUNAS_NECESSARIAS_CONSISTENCIA] = (
    df_consistencia[COLUNAS_NECESSARIAS_CONSISTENCIA]
    .replace([np.inf, -np.inf], np.nan)
)

# ==============================================================================
# 4. FUNÇÃO PARA PADRONIZAR AS REGRAS
# ==============================================================================

def preparar_regra_consistencia(condicao: pd.Series) -> pd.Series:
    """
    Converte o resultado de uma regra em verdadeiro ou falso.

    Valores ausentes são classificados como False, ou seja,
    o critério não é considerado atendido quando não há informação.
    """

    # Substitui resultados ausentes por False.
    condicao = condicao.fillna(False)

    # Garante que o resultado final tenha tipo booleano.
    return condicao.astype(bool)


# ==============================================================================
# 5. QUALIDADE DO LEITE
# ==============================================================================

# CCS é considerada consistente quando for maior que 50.
df_consistencia["cons_ccs"] = preparar_regra_consistencia( df_consistencia["ccs"].gt(50) )

# CPP é considerada consistente quando for maior que 1.
df_consistencia["cons_cpp"] = preparar_regra_consistencia( df_consistencia["cpp"].gt(1) )

# Gordura é consistente quando estiver acima de 2,5 e abaixo de 5,5.
df_consistencia["cons_fat"] = preparar_regra_consistencia( df_consistencia["fat"].gt(2.5) & df_consistencia["fat"].lt(5.5) )

# Proteína é consistente quando estiver acima de 2,4 e abaixo de 4,5.
df_consistencia["cons_protein"] = preparar_regra_consistencia( df_consistencia["protein"].gt(2.4) & df_consistencia["protein"].lt(4.5) )


# ==============================================================================
# 6. ESTRUTURA DO REBANHO
# ==============================================================================

# Na base nova, lactating_cows_total_cows está em percentual. 
# O limite antigo de 0,20 a 0,99 corresponde agora a 20% a 99%.
df_consistencia["cons_lactating_cows_total_cows"] = (
    preparar_regra_consistencia(
        df_consistencia["lactating_cows_total_cows"].gt(20)
        & df_consistencia["lactating_cows_total_cows"].lt(99)
    )
)

# Na base nova, lactating_cows_total_cattle também está em percentual. 
# O limite antigo de 0,15 a 0,99 corresponde agora a 15% a 99%.
df_consistencia["cons_lactating_cows_total_cattle"] = (
    preparar_regra_consistencia(
        df_consistencia["lactating_cows_total_cattle"].gt(15)
        & df_consistencia["lactating_cows_total_cattle"].lt(99)
    )
)


# ==============================================================================
# 7. PRODUTIVIDADE E MÃO DE OBRA
# ==============================================================================

# A produção diária por vaca em lactação deve ser maior que 3 e menor que 45 litros por vaca por dia.
df_consistencia["cons_milk_lactating_cow_day"] = (
    preparar_regra_consistencia(
        df_consistencia["milk_lactating_cow_day"].gt(3)
        & df_consistencia["milk_lactating_cow_day"].lt(45)
    )
)


# A produção diária por unidade de mão de obra deve ser maior que 20 e menor que 1.500 litros por trabalhador por dia.
df_consistencia["cons_milk_total_labor_day"] = (
    preparar_regra_consistencia(
        df_consistencia["milk_total_labor_day"].gt(20)
        & df_consistencia["milk_total_labor_day"].lt(1500)
    )
)


# O número de vacas em lactação por unidade de mão de obra deve ser positivo e menor que 70.
# O limite inferior positivo evita que propriedades com zero vacas em lactação sejam classificadas como consistentes.
df_consistencia["cons_lactating_cows_total_labor"] = (
    preparar_regra_consistencia(
        df_consistencia["lactating_cows_total_labor"].gt(0)
        & df_consistencia["lactating_cows_total_labor"].lt(70)
    )
)


# ==============================================================================
# 8. ALIMENTAÇÃO E CUSTOS
# ==============================================================================

# feeding_cost_milk_price está em percentual na base nova.
# O limite antigo de 0,15 a 1,50 corresponde a 15% a 150%.
df_consistencia["cons_feeding_cost_milk_price"] = (
    preparar_regra_consistencia(
        df_consistencia["feeding_cost_milk_price"].gt(15)
        & df_consistencia["feeding_cost_milk_price"].lt(150)
    )
)

# O custo do volumoso deve ser menor que R$ 3,00 por litro em todos os sistemas.
# - Para sistemas à pasto ("PASTURE") ou semi-confinado ("SEMI_CONFINED"): pode ser >= 0.
# - Para os demais sistemas (Compost Barn, Free Stall, Confinado, etc.): deve ser > 0,10.

is_pasto_semi = df_consistencia["production_system"].astype(str).str.upper().isin(["PASTURE", "SEMI_CONFINED"])

# Condição 1: Pasto ou Semi-confinado (0 <= custo < 3.00)
cond_pasto_semi = is_pasto_semi & df_consistencia["voluminous_cost_liter"].ge(0) & df_consistencia["voluminous_cost_liter"].lt(3.0)

# Condição 2: Demais sistemas (0.10 < custo < 3.00)
cond_demais_sistemas = (~is_pasto_semi) & df_consistencia["voluminous_cost_liter"].gt(0.10) & df_consistencia["voluminous_cost_liter"].lt(3.0)

# Aplicação da regra combinada
df_consistencia["cons_voluminous_cost_liter"] = preparar_regra_consistencia(
    cond_pasto_semi | cond_demais_sistemas
)

# O custo de concentrado e minerais deve ser maior que R$ 0,30 e menor que R$ 3,50 por litro.
df_consistencia["cons_concentrate_mineral_cost_liter"] = (
    preparar_regra_consistencia(
        df_consistencia["concentrate_mineral_cost_liter"].gt(0.3)
        & df_consistencia["concentrate_mineral_cost_liter"].lt(3.5)
    )
)

# O custo da mão de obra contratada deve ser menor que R$ 1 por litro.
df_consistencia["cons_hired_labor_cost_liter"] = (
    preparar_regra_consistencia(
        df_consistencia["hired_labor_cost_liter"].lt(1)
    )
)

# ==============================================================================
# 9. ESTOQUE DE CAPITAL
# ==============================================================================

# Verifica se o estoque total de capital por produção diária
# está abaixo do limite de consistência.
# df_consistencia["cons_total_capital_stock"] = (
#     preparar_regra_consistencia(
#         df_consistencia["total_capital_stock_milk_daily"].lt(30000)
#     )
# )

# ==============================================================================
# 10. MAPA DOS CRITÉRIOS
# ==============================================================================

# Relaciona cada coluna booleana ao nome que será exibido
# no relatório de critérios violados.
MAPA_CRITERIOS_CONSISTENCIA = {
    "cons_ccs": "CCS",
    "cons_cpp": "CPP",
    "cons_fat": "Gordura",
    "cons_protein": "Proteína",
    "cons_lactating_cows_total_cows": "VL/Total de vacas",
    "cons_lactating_cows_total_cattle": "VL/Rebanho total",
    "cons_milk_lactating_cow_day": "Produção/VL",
    "cons_milk_total_labor_day": "Produção/MDO",
    "cons_lactating_cows_total_labor": "VL/MDO",
    "cons_feeding_cost_milk_price": "Alimentação/Preço do leite",
    "cons_voluminous_cost_liter": "Custo de volumoso",
    "cons_concentrate_mineral_cost_liter": "Custo de concentrado",
    "cons_hired_labor_cost_liter": "Custo da MDO contratada",
    # "cons_total_capital_stock": "Estoque de capital fixo",
}


# Transforma as chaves do dicionário em uma lista.
# Essa lista contém todas as colunas de consistência.
colunas_criterios = list(
    MAPA_CRITERIOS_CONSISTENCIA.keys()
)


# ==============================================================================
# 11. CONTAR CRITÉRIOS ATENDIDOS E VIOLADOS
# ==============================================================================

# Soma os valores True de cada linha.
# No pandas, True equivale a 1 e False equivale a 0.
df_consistencia["total_consistency_criteria_ok"] = (
    df_consistencia[colunas_criterios]
    .sum(axis=1)
    .astype("int8")
)


# Salva a quantidade total de critérios avaliados.
df_consistencia["total_consistency_criteria"] = len(
    colunas_criterios
)


# Calcula quantos critérios não foram atendidos.
df_consistencia["total_consistency_criteria_violated"] = (
    df_consistencia["total_consistency_criteria"]
    - df_consistencia["total_consistency_criteria_ok"]
)


# ==============================================================================
# 12. CLASSIFICAÇÃO GERAL
# ==============================================================================

# Classifica como Consistente quando nenhum critério foi violado.
# Caso exista pelo menos uma violação, classifica como Inconsistente.
df_consistencia["consistency_status"] = np.where(
    df_consistencia["total_consistency_criteria_violated"].eq(0),
    "Consistente",
    "Inconsistente",
)


# Cria uma classificação numérica.
# Zero representa consistente e um representa inconsistente.
df_consistencia["consistency_id"] = np.where(
    df_consistencia["total_consistency_criteria_violated"].eq(0),
    0,
    1,
).astype("int8")


# ==============================================================================
# 13. LISTAR OS CRITÉRIOS VIOLADOS E DETALHAMENTO DOS MOTIVOS/VALORES
# ==============================================================================

# Cria inicialmente uma coluna vazia.
df_consistencia["violated_consistency_criteria"] = ""


# Percorre cada critério e seu respectivo nome de exibição.
for coluna_criterio, nome_criterio in MAPA_CRITERIOS_CONSISTENCIA.items():

    # Identifica as linhas em que o critério não foi atendido.
    mascara_violacao = ~df_consistencia[coluna_criterio]

    # Adiciona o nome do critério à lista de violações da linha.
    df_consistencia.loc[
        mascara_violacao,
        "violated_consistency_criteria",
    ] += nome_criterio + "; "


# Remove o último ponto e vírgula e os espaços excedentes.
df_consistencia["violated_consistency_criteria"] = (
    df_consistencia["violated_consistency_criteria"]
    .str.rstrip("; ")
    .replace("", "Nenhum")
)

# ==============================================================================
# 13.1 COLUNA DETALHADA COM VALORES E MOTIVOS EXATOS DA VIOLAÇÃO
# ==============================================================================
REGRAS_DETALHADAS = [
    ("cons_ccs", "ccs", 50, None, "CCS", "", ".0f"),
    ("cons_cpp", "cpp", 1, None, "CPP", "", ".0f"),
    ("cons_fat", "fat", 2.5, 5.5, "Gordura", "%", ".2f"),
    ("cons_protein", "protein", 2.4, 4.5, "Proteína", "%", ".2f"),
    ("cons_lactating_cows_total_cows", "lactating_cows_total_cows", 20, 99, "VL/Total de vacas", "%", ".1f"),
    ("cons_lactating_cows_total_cattle", "lactating_cows_total_cattle", 15, 99, "VL/Rebanho total", "%", ".1f"),
    ("cons_milk_lactating_cow_day", "milk_lactating_cow_day", 3, 45, "Produção/VL", " L/vaca/dia", ".1f"),
    ("cons_milk_total_labor_day", "milk_total_labor_day", 20, 1500, "Produção/MDO", " L/trabalhador/dia", ".1f"),
    ("cons_lactating_cows_total_labor", "lactating_cows_total_labor", 0, 70, "VL/MDO", " vacas/trabalhador", ".1f"),
    ("cons_feeding_cost_milk_price", "feeding_cost_milk_price", 15, 150, "Alimentação/Preço do leite", "%", ".1f"),
    ("cons_voluminous_cost_liter", "voluminous_cost_liter", None, 3.0, "Custo de volumoso", " R$/L", ".2f"),
    ("cons_concentrate_mineral_cost_liter", "concentrate_mineral_cost_liter", 0.30, 3.50, "Custo de concentrado", " R$/L", ".2f"),
    ("cons_hired_labor_cost_liter", "hired_labor_cost_liter", None, 1.0, "Custo da MDO contratada", " R$/L", ".2f"),
]

detalhes_lista = []
for _, row in df_consistencia.iterrows():
    motivos = []
    for col_bool, col_val, min_v, max_v, nome, unit, fmt in REGRAS_DETALHADAS:
        if not row.get(col_bool, True):
            val = row.get(col_val, np.nan)
            if pd.isna(val):
                motivos.append(f"{nome} (Ausente/NaN)")
            elif min_v is not None and val <= min_v:
                motivos.append(f"{nome} (Abaixo do mín <{min_v}: {val:{fmt}}{unit})")
            elif max_v is not None and val >= max_v:
                motivos.append(f"{nome} (Acima do máx >{max_v}: {val:{fmt}}{unit})")
            else:
                motivos.append(f"{nome} (Inconsistente: {val:{fmt}}{unit})")
    
    detalhes_lista.append("; ".join(motivos) if motivos else "Nenhum")

df_consistencia["violated_consistency_details"] = detalhes_lista

# Atualiza df_integrada para que a exportação mensal contenha todas as colunas de consistência
df_integrada = df_consistencia.copy()

# ==============================================================================
# 14. CONFERÊNCIA DO RESULTADO
# ==============================================================================

# Cria uma tabela resumida com a quantidade de registros
# consistentes e inconsistentes.
df_resumo_consistencia = (
    df_consistencia["consistency_status"]
    .value_counts(dropna=False)
    .rename_axis("consistency_status")
    .reset_index(name="records")
)


# Calcula o percentual de cada classificação.
df_resumo_consistencia["percentage"] = (
    df_resumo_consistencia["records"]
    .div(len(df_consistencia))
    .mul(100)
    .round(2)
)


# Exibe o resumo da classificação.
display(df_resumo_consistencia)


# Exibe uma amostra das principais colunas geradas.
display(
    df_consistencia[
        [
            "id_property",
            "reference_month",
            "consistency_status",
            "consistency_id",
            "total_consistency_criteria_ok",
            "total_consistency_criteria_violated",
            "violated_consistency_criteria",
            "violated_consistency_details",
        ]
    ]
    .head(20)
)


,consistency_status,records,percentage
0,Consistente,8302,63.24
1,Inconsistente,4826,36.76


,id_property,reference_month,consistency_status,consistency_id,total_consistency_criteria_ok,total_consistency_criteria_violated,violated_consistency_criteria,violated_consistency_details
0,00220277-a58e-4b9e-9b89-e6acf8a1a400,2024-05-01,Inconsistente,1,0,13,CCS; CPP; Gordura; Proteína; VL/Total de vacas...,CCS (Ausente/NaN); CPP (Ausente/NaN); Gordura ...
1,00220277-a58e-4b9e-9b89-e6acf8a1a400,2024-08-01,Inconsistente,1,12,1,Custo de volumoso,Custo de volumoso (Inconsistente: 0.05 R$/L)
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,Consistente,0,13,0,Nenhum,Nenhum
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,Consistente,0,13,0,Nenhum,Nenhum
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,Consistente,0,13,0,Nenhum,Nenhum
5,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-07-01,Consistente,0,13,0,Nenhum,Nenhum
6,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-08-01,Consistente,0,13,0,Nenhum,Nenhum
7,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-09-01,Consistente,0,13,0,Nenhum,Nenhum
8,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-10-01,Consistente,0,13,0,Nenhum,Nenhum
9,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-11-01,Consistente,0,13,0,Nenhum,Nenhum


In [33]:
df_consistencia

,id_property,reference_month,milk_sold_revenue,milk_volume_sold,milk_unit_price,ccs,cpp,fat,protein,unit_price_derivative,...,cons_voluminous_cost_liter,cons_concentrate_mineral_cost_liter,cons_hired_labor_cost_liter,total_consistency_criteria_ok,total_consistency_criteria,total_consistency_criteria_violated,consistency_status,consistency_id,violated_consistency_criteria,violated_consistency_details
0,00220277-a58e-4b9e-9b89-e6acf8a1a400,2024-05-01,0.000000e+00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,0,13,13,Inconsistente,1,CCS; CPP; Gordura; Proteína; VL/Total de vacas...,CCS (Ausente/NaN); CPP (Ausente/NaN); Gordura ...
1,00220277-a58e-4b9e-9b89-e6acf8a1a400,2024-08-01,5.346826e+05,167274.0,3.196448,159.0,33.0,3.85,3.35,NaN,...,False,True,True,12,13,1,Inconsistente,1,Custo de volumoso,Custo de volumoso (Inconsistente: 0.05 R$/L)
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,1.129862e+05,35677.0,3.166920,421.0,77.0,3.48,3.40,NaN,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,1.641377e+05,54884.0,2.990629,338.0,54.0,3.55,3.29,NaN,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,1.627506e+05,56311.0,2.890209,192.0,102.0,3.46,3.32,NaN,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13123,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,2026-02-01,1.966946e+06,765080.0,2.570902,170.0,18.0,3.79,3.31,NaN,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
13124,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,2026-03-01,2.179970e+06,765080.0,2.849336,158.0,3.0,3.82,3.30,NaN,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
13125,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,2026-04-01,2.312813e+06,733654.0,3.152457,167.0,9.0,4.04,3.33,NaN,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
13126,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,2026-05-01,2.335218e+06,751991.0,3.105380,167.0,9.0,4.04,3.33,NaN,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum


## 3.8 Preparação e Formatação dos Dados Mensais para Excel


In [34]:
# ==============================================================================
# 3.8 PREPARAÇÃO E FORMATAÇÃO DOS DADOS MENSAIS PARA EXCEL
# ==============================================================================
def preparar_dataframe_para_excel(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepara um DataFrame para exportação pelo openpyxl.
    """

    df_excel = df.copy()

    # Excel não trabalha com infinito
    df_excel = df_excel.replace([np.inf, -np.inf], np.nan)
    
    # Excel não aceita datas com timezone
    for coluna in df_excel.columns:
        if isinstance(df_excel[coluna].dtype, pd.DatetimeTZDtype, ):
            df_excel[coluna] = ( df_excel[coluna] .dt.tz_localize(None) )
    
    return df_excel


# ==============================================================================
# 3.8.1 TRADUÇÃO E MAPEAMENTO 1-A-1 DOS INDICADORES MENSAIS
# ==============================================================================
COLUNAS_TECNICAS_MENSAIS_EM_ORDEM = [
    "id_property",
    "property_entrepreneur_label",
    "labor_rural_code",
    "agroindustry_name",
    "dairy_region",
    "consultant_name",
    "reference_month",
    "milk_sold_revenue", "milk_volume_sold", "milk_unit_price", "ccs", "cpp", "fat", "protein",
    "unit_price_derivative", "milk_volume_derivatives", "milk_derivatives_revenue", "received_loans", "animal_sale",
    "other_revenues", "price_bonus", "price_penalty", "voluminous_sold", "concentrated_sold", "surplus_division",
    "lactating_cows", "dry_cows", "nursing", "rearing", "males", "other_categories", "total_cows", "total_cattle",
    "lactating_cows_value", "dry_cows_value", "nursing_value", "rearing_value", "males_value", "other_categories_value",
    "general_expenses", "advance_payment", "administration", "land_lease", "technical_assistance", "animal_purchase",
    "land_purchase", "repairs", "loan_interest", "hormones", "taxes_fees", "medicines_vaccines", "bedding_replacement",
    "reproduction", "milk_replacer", "milking_material", "milk_calves", "energy", "fuel",
    "voluminous_purchased_quantity", "voluminous_consumed_quantity", "voluminous_amount_total",
    "concentrate_purchased_quantity", "concentrate_consumed_quantity", "concentrate_amount_total",
    "mineral_purchased_quantity", "mineral_consumed_quantity", "mineral_amount_total",
    "family_labor_expenses", "family_labor_quantity", "hired_labor_expenses", "hired_labor_quantity", "unit_price",
    "own_milk_hired_labor_quantity", "hired_labor_amount_total", "discarded_quantity", "discarded_amount_total",
    "calves_quantity", "calves_amount_total", "own_milk_family_labor_quantity", "family_labor_amount_total",
    "hectares_owned_benfeitorias_estradas", "hectares_owned_app_reserva_legal", "hectares_owned_forrageiras",
    "hectares_rented_benfeitorias_estradas", "hectares_rented_app_reserva_legal", "hectares_rented_forrageiras",
    "raw_land_value_benfeitorias_estradas", "raw_land_value_app_reserva_legal", "raw_land_value_forrageiras",
    "hectares_propria", "hectares_atividade", "hectares_total", "raw_land_value_medio_ponderado",
    "monthly_depreciation_benfeitorias", "monthly_depreciation_maquinas_e_equipamentos",
    "monthly_average_capital_stock_benfeitorias", "monthly_average_capital_stock_maquinas_e_equipamentos",
    "production_system", "deflator", "voluminous_amount_total_forrageira_propria",
    "concentrate_amount_total_forrageira_propria",
    "has_revenue_data", "has_cattle_data", "has_expense_data", "has_feeding_data", "has_labor_data", "has_own_milk_data",
    "has_asset_data", "has_active_area_month", "has_dairy_production_system_data",
    "discarded_quantity_milk_revenue", "hired_labor_quantity_milk_revenue", "family_labor_milk_revenue",
    "calves_quantity_milk_revenue", "total_milk_revenue", "milk_produced", "total_activity_revenue",
    "milk_revenue_liter", "days_in_month", "milk_daily", "milk_lactating_cow_day", "lactating_cows_total_cows",
    "lactating_cows_total_cattle", "total_labor_quantity", "milk_total_labor_day", "lactating_cows_total_labor",
    "concentrate_mineral_cost", "feeding_cost", "concentrate_mineral_quantity", "feeding_cost_liter",
    "voluminous_cost_liter", "concentrate_mineral_cost_liter", "feeding_cost_milk_price", "total_labor_expenses",
    "hired_labor_cost_liter", "family_labor_cost_liter", "total_labor_cost_liter", "labor_cost_milk_revenue",
    "other_operating_expenses", "animal_capital_stock", "land_capital_stock", "fixed_capital_stock",
    "total_capital_stock", "total_capital_stock_milk_daily", "lactating_cows_hectare_activity", "hectares_arrendada",
    "rented_area_percentage", "property_status", "cons_ccs", "cons_cpp", "cons_fat", "cons_protein",
    "cons_lactating_cows_total_cows", "cons_lactating_cows_total_cattle", "cons_milk_lactating_cow_day",
    "cons_milk_total_labor_day", "cons_lactating_cows_total_labor", "cons_feeding_cost_milk_price",
    "cons_voluminous_cost_liter", "cons_concentrate_mineral_cost_liter", "cons_hired_labor_cost_liter",
    "total_consistency_criteria_ok", "total_consistency_criteria", "total_consistency_criteria_violated",
    "consistency_status", "consistency_id", "violated_consistency_criteria", "violated_consistency_details"
]

NOMES_PORTUGUES_EM_ORDEM_MENSAIS = [
    "IdFazenda",
    "Fazenda - Produtor",
    "Código LR",
    "Agroindústria",
    "Região",
    "Consultor",
    "Mês de Referência",
    "Receita com Venda de Leite (R$)", "Volume de Leite Vendido (litros)", "Preço Unitário do Leite (R$/litro)", "CCS (x1000 células/ml)", "CPP (x1000 UFC/ml)", "Gordura (%)", "Proteína (%)",
    "Preço Unitário Derivados (R$)", "Volume de Leite em Derivados (litros)", "Receita de Derivados (R$)", "Empréstimos Recebidos (R$)", "Venda de Animais (R$)",
    "Outras Receitas (R$)", "Bonificação no Preço (R$)", "Penalidade no Preço (R$)", "Volumoso Vendido (R$)", "Concentrado Vendido (R$)", "Divisão de Sobra (R$)",
    "Vacas em Lactação (cabeças)", "Vacas Secas (cabeças)", "Bezerras (cabeças)", "Novilhas (cabeças)", "Machos (cabeças)", "Outras Categorias (cabeças)", "Total de Vacas (cabeças)", "Total de Animais (cabeças)",
    "Valor Vacas em Lactação (R$)", "Valor Vacas Secas (R$)", "Valor Bezerras (R$)", "Valor Novilhas (R$)", "Valor Machos (R$)", "Valor Outras Categorias (R$)",
    "Acessórios e Despesas Gerais (R$)", "Adiantamento (R$)", "Despesas Administrativas (R$)", "Arrendamento (R$)", "Assistência Técnica (R$)", "Compra de Animais (R$)",
    "Compra de Terra (R$)", "Reparos e Consertos (R$)", "Juros de Empréstimos (R$)", "Hormônios (R$)", "Impostos e Taxas (R$)", "Medicamentos e Vacinas (R$)", "Reposição da cama (R$)",
    "Reprodução (R$)", "Sucedâneo (R$)", "Material de Ordenha (R$)", "Leite para Bezerras (R$)", "Energia Elétrica (R$)", "Combustível (R$)",
    "Qtd Comprada Volumoso (kg)", "Qtd Consumida Volumoso (kg)", "Gasto Total Volumoso (R$)",
    "Qtd Comprada Concentrado (kg)", "Qtd Consumida Concentrado (kg)", "Gasto Total Concentrado (R$)",
    "Qtd Comprada Mineral (kg)", "Qtd Consumida Mineral (kg)", "Gasto Total Mineral (R$)",
    "Despesas Mão de Obra Familiar (R$)", "Qtd Mão de Obra Familiar (trabalhadores)", "Despesas Mão de Obra Contratada (R$)", "Qtd Mão de Obra Contratada (trabalhadores)", "Preço Unitário do Leite Próprio (R$/litro)",
    "Leite Próprio MDO Contratada (litros)", "Custo Total MDO Contratada (R$)", "Leite Descartado (litros)", "Valor Lançado Leite Descartado (R$)",
    "Leite Bezerras Consumido (litros)", "Valor Leite Bezerras (R$)", "Leite Próprio MDO Familiar (litros)", "Custo com Leite Consumido (R$)",
    "Área Própria Benfeitorias/Estradas (ha)", "Área Própria APP/Reserva (ha)", "Área Própria Forrageiras (ha)",
    "Área Arrendada Benfeitorias/Estradas (ha)", "Área Arrendada APP/Reserva (ha)", "Área Arrendada Forrageiras (ha)",
    "Valor Terra Nua Benfeitorias/Estradas (R$)", "Valor Terra Nua APP/Reserva (R$)", "Valor Terra Nua Forrageiras (R$)",
    "Área Própria Total (ha)", "Área Destinada à Atividade (ha)", "Área Total da Propriedade (ha)", "Valor Média Ponderada Terra Nua (R$/ha)",
    "Depreciação Mensal Benfeitorias (R$)", "Depreciação Mensal Máquinas (R$)",
    "Estoque de Capital Benfeitorias (R$)", "Estoque de Capital Máquinas (R$)",
    "Sistema de Produção", "Fator Deflator (IGP-DI)", "Custo Forrageira Própria Volumoso (R$)",
    "Custo Forrageira Própria Concentrado (R$)",
    "Possui Dados de Receita", "Possui Dados de Rebanho", "Possui Dados de Despesas", "Possui Dados de Alimentação", "Possui Dados de Mão de Obra", "Possui Dados de Leite Próprio",
    "Possui Dados de Ativos", "Possui Dados de Área Ativa", "Possui Dados Sistema Produção",
    "Valor Leite Descartado a Preço de Venda do Leite (R$)", "Valor Leite MDO Contratada a Preço de Leite (R$)", "Custo com Leite consumido pela MDO familiar (R$)",
    "Custo leite consumido pelas Bezerras (R$)", "Receita Total do Leite (R$)", "Produção Total de Leite (litros)", "Receita Bruta da Atividade (R$)",
    "Receita da Atividade por litro de leite (R$/litro)", "Dias no Mês", "Produção Diária de Leite (litros/dia)", "Produção por Vaca em Lactação (litros/vaca/dia)", "Vacas em Lactação / Total de Vacas (%)",
    "Vacas em Lactação / Rebanho Total (%)", "Mão de Obra Total (trabalhadores)", "Produção por Mão de Obra (litros/trabalhador/dia)", "Vacas em Lactação por Mão de Obra (vacas/trabalhador)",
    "Custo de Concentrado e Mineral (R$)", "Custo Total com Alimentação (R$)", "Quantidade de Concentrado e Mineral (kg)", "Custo de Alimentação por Litro (R$/litro)",
    "Custo de Volumoso por Litro (R$/litro)", "Custo de Concentrado por Litro (R$/litro)", "Alimentação / Preço do Leite (%)", "Custo Total Mão de Obra (R$)",
    "Custo MDO Contratada por Litro (R$/litro)", "Custo MDO Familiar por Litro (R$/litro)", "Custo MDO Total por Litro (R$/litro)", "Mão de Obra / Receita do Leite (%)",
    "Outras Despesas Operacionais (R$)", "Estoque de Capital em Animais (R$)", "Estoque de Capital em Terra (R$)", "Estoque de Capital Fixo (R$)",
    "Estoque de Capital Total (R$)", "Estoque de Capital / Produção Diária (R$/litro/dia)", "Vacas em Lactação / Área (vacas/ha)", "Área Arrendada (ha)",
    "Área Arrendada (%)", "Status Cadastral", "Consistência: CCS", "Consistência: CPP", "Consistência: Gordura", "Consistência: Proteína",
    "Consistência: VL/Total Vacas", "Consistência: VL/Rebanho Total", "Consistência: Produção/VL",
    "Consistência: Produção/MDO", "Consistência: VL/MDO", "Consistência: Alimentação/Preço Leite",
    "Consistência: Custo Volumoso/L", "Consistência: Custo Concentrado/L", "Consistência: Custo MDO Contratada/L",
    "Total Critérios Atendidos", "Total Critérios Avaliados", "Total Critérios Violados",
    "Status de Consistência", "ID de Consistência", "Critérios Violados", "Detalhamento das Inconsistências"
]

assert len(COLUNAS_TECNICAS_MENSAIS_EM_ORDEM) == len(NOMES_PORTUGUES_EM_ORDEM_MENSAIS), (
    f"Incompatibilidade de tamanho: {len(COLUNAS_TECNICAS_MENSAIS_EM_ORDEM)} vs {len(NOMES_PORTUGUES_EM_ORDEM_MENSAIS)}"
)

# Garantir a criação da coluna de rótulo combinado se não existir em df_consistencia
if "property_entrepreneur_label" not in df_consistencia.columns and {"property_name", "entrepreneur_name"}.issubset(df_consistencia.columns):
    df_consistencia["property_entrepreneur_label"] = (
        df_consistencia["property_name"].fillna("") + " - " + df_consistencia["entrepreneur_name"].fillna("")
    )

dados_renomeados_mensais = {}
colunas_mensais_ausentes = []

for col_tecnica, col_pt in zip(COLUNAS_TECNICAS_MENSAIS_EM_ORDEM, NOMES_PORTUGUES_EM_ORDEM_MENSAIS):
    if col_tecnica in df_consistencia.columns:
        dados_renomeados_mensais[col_pt] = df_consistencia[col_tecnica].values
    else:
        dados_renomeados_mensais[col_pt] = pd.Series([None] * len(df_consistencia), index=df_consistencia.index).values
        colunas_mensais_ausentes.append(col_tecnica)

df_exportacao_mensal = pd.DataFrame(dados_renomeados_mensais, index=df_consistencia.index)
print(f"Indicadores mensais mapeados para português: {df_exportacao_mensal.shape[1]} colunas.")


Indicadores mensais mapeados para português: 168 colunas.


In [35]:
df_exportacao_mensal

,IdFazenda,Fazenda - Produtor,Código LR,Agroindústria,Região,Consultor,Mês de Referência,Receita com Venda de Leite (R$),Volume de Leite Vendido (litros),Preço Unitário do Leite (R$/litro),...,Consistência: Custo Volumoso/L,Consistência: Custo Concentrado/L,Consistência: Custo MDO Contratada/L,Total Critérios Atendidos,Total Critérios Avaliados,Total Critérios Violados,Status de Consistência,ID de Consistência,Critérios Violados,Detalhamento das Inconsistências
0,00220277-a58e-4b9e-9b89-e6acf8a1a400,FAZENDA RIO DO PEIXE - Jaci Guimaraes De Siqueira,LR05385,NESTLÉ,Goiânia - 9655,"MARLUS MARRA CARVALHO PINHEIRO, Pablo Freitas ...",2024-05-01,0.000000e+00,0.0,NaN,...,False,False,False,0,13,13,Inconsistente,1,CCS; CPP; Gordura; Proteína; VL/Total de vacas...,CCS (Ausente/NaN); CPP (Ausente/NaN); Gordura ...
1,00220277-a58e-4b9e-9b89-e6acf8a1a400,FAZENDA RIO DO PEIXE - Jaci Guimaraes De Siqueira,LR05385,NESTLÉ,Goiânia - 9655,"MARLUS MARRA CARVALHO PINHEIRO, Pablo Freitas ...",2024-08-01,5.346826e+05,167274.0,3.196448,...,False,True,True,12,13,1,Inconsistente,1,Custo de volumoso,Custo de volumoso (Inconsistente: 0.05 R$/L)
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,Fazenda São Bartolomeu - Edgar Jose de Azevedo...,LR01964,NESTLÉ,Ibiá - 1215,LORENA VIRGINIA ARAUJO,2025-04-01,1.129862e+05,35677.0,3.166920,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,Fazenda São Bartolomeu - Edgar Jose de Azevedo...,LR01964,NESTLÉ,Ibiá - 1215,LORENA VIRGINIA ARAUJO,2025-05-01,1.641377e+05,54884.0,2.990629,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
4,004cde8f-4688-4b22-acd9-76151b7dcc7a,Fazenda São Bartolomeu - Edgar Jose de Azevedo...,LR01964,NESTLÉ,Ibiá - 1215,LORENA VIRGINIA ARAUJO,2025-06-01,1.627506e+05,56311.0,2.890209,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13123,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,Fazenda Campo Alegre - Luiz Alexandre de Avelar,LR05853,NESTLÉ,Patos de Minas - 9188,GISELE MARIA CARVALHO MAGALHAES,2026-02-01,1.966946e+06,765080.0,2.570902,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
13124,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,Fazenda Campo Alegre - Luiz Alexandre de Avelar,LR05853,NESTLÉ,Patos de Minas - 9188,GISELE MARIA CARVALHO MAGALHAES,2026-03-01,2.179970e+06,765080.0,2.849336,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
13125,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,Fazenda Campo Alegre - Luiz Alexandre de Avelar,LR05853,NESTLÉ,Patos de Minas - 9188,GISELE MARIA CARVALHO MAGALHAES,2026-04-01,2.312813e+06,733654.0,3.152457,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum
13126,fff52e9f-83b6-44db-b7a4-01da7e77aeaa,Fazenda Campo Alegre - Luiz Alexandre de Avelar,LR05853,NESTLÉ,Patos de Minas - 9188,GISELE MARIA CARVALHO MAGALHAES,2026-05-01,2.335218e+06,751991.0,3.105380,...,True,True,True,13,13,0,Consistente,0,Nenhum,Nenhum


## 3.9 Exportação dos Arquivos Excel Mensais


In [36]:
# ==============================================================================
# 3.9 EXPORTAÇÃO DOS ARQUIVOS EXCEL MENSAIS
# ==============================================================================
# Garante que a raiz do projeto seja a pasta pai caso o notebook rode dentro da pasta /app
RAIZ_PROJETO = Path.cwd().parent if Path.cwd().name == 'app' else Path.cwd()
PASTA_SAIDA = RAIZ_PROJETO / 'data' / 'outputs' / 'monthly'
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

DATA_EXPORTACAO = datetime.now().strftime("%Y_%m_%d_%H%M%S")
CAMINHO_ARQUIVO = PASTA_SAIDA / f'{DATA_EXPORTACAO}_indicadores_mensais.xlsx'

# Preparar os dados em português para o Excel
df_exportacao = preparar_dataframe_para_excel(df_exportacao_mensal)

# Exportar com a formatação padrão
exportar_varias_abas_xlsx(
    abas={'Indicadores Mensais': df_exportacao},
    caminho_saida=CAMINHO_ARQUIVO,
    fonte='Aptos'
)

print('Exportação mensal concluída com sucesso.')
print(f'Linhas exportadas: {len(df_exportacao):,}')
print(f'Arquivo: {CAMINHO_ARQUIVO}')

Exportação mensal concluída com sucesso.
Linhas exportadas: 13,128
Arquivo: c:\Users\Guilherme\LABOR RURAL\Analytics - Departamento Analytics\TEMP\GUILHERME\SCRIPTS\projetcs\elabore-views\data\outputs\monthly\2026_07_31_170704_indicadores_mensais.xlsx


In [37]:
# # ==============================================================================
# # 3.9 EXPORTAÇÃO DOS ARQUIVOS EXCEL MENSAIS
# # ==============================================================================
# RAIZ_PROJETO = Path.cwd().parent if Path.cwd().name == "app" else Path.cwd()

# PASTA_SAIDA = RAIZ_PROJETO / "data" / "outputs" / "monthly"
# PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
# DATA_EXPORTACAO = datetime.now().strftime("%Y_%m_%d_%H%M%S")
# CAMINHO_ARQUIVO = ( PASTA_SAIDA / f"{DATA_EXPORTACAO}_indicadores_mensais.xlsx" )

# # Preparar os dados em português para o Excel
# df_exportacao = preparar_dataframe_para_excel( df_exportacao_mensal)

# # Criar o arquivo com a formatação padrão
# exportar_varias_abas_xlsx(abas={"Indicadores Mensais": df_exportacao}, caminho_saida=CAMINHO_ARQUIVO, fonte="Aptos")

# # Aplicar cabeçalho colorido e linhas alternadas
# aplicar_estilo_listrado_xlsx(
#     caminho_arquivo=CAMINHO_ARQUIVO,
#     cor_cabecalho="#247B72",
#     cor_texto_cabecalho="#FFFFFF",
#     cor_linha_alternada="#F2F2F2",
#     cor_linha_base="#FFFFFF",
#     primeira_linha_cinza=True,
# )

# print("Exportação mensal concluída com sucesso.")
# print(f"Linhas exportadas: {len(df_exportacao):,}")
# print(f"Arquivo: {CAMINHO_ARQUIVO}")

# 4. Indicadores Anuais - Janela Móvel, Regras e Exportação


## 4.1 Agregação por Janela Móvel de 12 Meses Completos


In [38]:
# ==============================================================================
# 4.1 AGREGAÇÃO POR JANELA MÓVEL DE 12 MESES COMPLETOS
# ==============================================================================
df_mensal_anuais = df_consistencia.copy()
df_mensal_anuais["reference_month"] = (pd.to_datetime(df_mensal_anuais["reference_month"], errors="coerce").dt.to_period("M").dt.to_timestamp())
df_mensal_anuais = (df_mensal_anuais.dropna(subset=["id_property", "reference_month"]).sort_values(["id_property", "reference_month"]).drop_duplicates(["id_property", "reference_month"], keep="last").reset_index(drop=True))

# Criar coluna de CCS, CPP, Gordura, Proteína ainda não calculados
df_mensal_anuais['abs_ccs'] = df_mensal_anuais['ccs'] * df_mensal_anuais['milk_produced']
df_mensal_anuais['abs_cpp'] = df_mensal_anuais['cpp'] * df_mensal_anuais['milk_produced']
df_mensal_anuais['abs_fat'] = df_mensal_anuais['fat'] * df_mensal_anuais['milk_produced']
df_mensal_anuais['abs_protein'] = df_mensal_anuais['protein'] * df_mensal_anuais['milk_produced']

# ==============================================================================
# ÁREA: RECONSTRUIR "COM RESERVA" E "FORRAGEIRA" (ainda não calculadas na Feature Engineering)
# ==============================================================================
# hectares_atividade já é a área "sem reserva" (benfeitorias/estradas + forrageiras, próprio + arrendado).
# Somando de volta a Reserva Legal e APP chegamos na área "com reserva" (equivalente à antiga areaAtividadeReserva).
COLUNAS_RESERVA_LEGAL = [c for c in ["hectares_owned_app_reserva_legal", "hectares_rented_app_reserva_legal"] if c in df_mensal_anuais.columns]
if COLUNAS_RESERVA_LEGAL:
    df_mensal_anuais["hectares_atividade_com_reserva"] = (
        df_mensal_anuais["hectares_atividade"].fillna(0)
        + df_mensal_anuais[COLUNAS_RESERVA_LEGAL].sum(axis=1, skipna=True)
    )

# Área de forrageira (própria + arrendada), equivalente à antiga areaForrageira.
COLUNAS_FORRAGEIRA = [c for c in ["hectares_owned_forrageiras", "hectares_rented_forrageiras"] if c in df_mensal_anuais.columns]
if COLUNAS_FORRAGEIRA:
    df_mensal_anuais["hectares_forrageira"] = df_mensal_anuais[COLUNAS_FORRAGEIRA].sum(axis=1, skipna=True)

COLUNAS_SOMA_ANUAL = [
    # Produção de Leite #
    "milk_produced", "milk_volume_sold", "milk_volume_derivatives", "discarded_quantity",
    # Renda #
    "total_milk_revenue", "total_activity_revenue", "concentrated_sold", "voluminous_sold",
    # COE #
    "general_expenses", "administration", "land_lease", "technical_assistance",
    "repairs", "hormones", "taxes_fees", "medicines_vaccines", "bedding_replacement",
    "reproduction", "milk_replacer", "milking_material", "milk_calves", "energy", "fuel",
    "voluminous_amount_total", "concentrate_amount_total", "mineral_amount_total",
    "hired_labor_expenses", "other_operating_expenses",
    "feeding_cost", "concentrate_mineral_cost",
    # COT #
    "family_labor_expenses", "monthly_depreciation_benfeitorias", "monthly_depreciation_maquinas_e_equipamentos",
    # Alimentação (quantidades consumidas) #
    "concentrate_consumed_quantity",
    "voluminous_consumed_quantity",
    "mineral_consumed_quantity",
    # MDO #
    "family_labor_quantity",
    "hired_labor_quantity",
    # QUALIDADE (auxiliares para média ponderada) #
    "abs_ccs", "abs_cpp", "abs_fat", "abs_protein",
    # Estoque de Capital #
    'monthly_depreciation_benfeitorias',
    'monthly_depreciation_maquinas_e_equipamentos'
]
COLUNAS_MEDIA_ANUAL = [
    # Animais #
    "lactating_cows", "total_cows", "total_cattle",
    # Quantidade de MDO #
    "hired_labor_quantity", "family_labor_quantity", "total_labor_quantity",
    # Área #
    "hectares_atividade", "hectares_atividade_com_reserva", "hectares_forrageira",
    "hectares_arrendada", "hectares_propria", "hectares_total",
    "raw_land_value_medio_ponderado",
    # Estoque de Capital (é um estoque médio, não um fluxo: usar média, não soma) === #
    "animal_capital_stock", "land_capital_stock",
    'monthly_average_capital_stock_benfeitorias',
    'monthly_average_capital_stock_maquinas_e_equipamentos'
]
COLUNAS_QUALIDADE = ["ccs", "cpp", "fat", "protein"]
COLUNAS_SOMA_ANUAL = [c for c in COLUNAS_SOMA_ANUAL if c in df_mensal_anuais.columns]
COLUNAS_MEDIA_ANUAL = [c for c in COLUNAS_MEDIA_ANUAL if c in df_mensal_anuais.columns]
COLUNAS_QUALIDADE = [c for c in COLUNAS_QUALIDADE if c in df_mensal_anuais.columns]

# ==============================================================================
# COBERTURA DE LANÇAMENTO DE DADOS POR JANELA ANUAL
# ==============================================================================
# A continuidade da janela (12 meses seguidos) só é garantida para a vw_revenue,
# que é a base de df_mensal_anuais. Isso não impede que uma propriedade tenha
# lançado receita todo mês mas deixado de lançar despesa/mão de obra/alimentação
# em alguns meses daquela mesma janela - o que subestima COE/COT/CT sem gerar
# nenhum erro (a soma anual simplesmente usa os meses que existem).
# Este relatório mede, para cada janela, quantos dos 12 meses têm cada fonte.
MAPA_COBERTURA = {
    "has_revenue_data": "months_with_revenue_data",
    "has_cattle_data": "months_with_cattle_data",
    "has_expense_data": "months_with_expense_data",
    "has_feeding_data": "months_with_feeding_data",
    "has_labor_data": "months_with_labor_data",
    "has_own_milk_data": "months_with_own_milk_data",
    "has_asset_data": "months_with_asset_data",
    "has_active_area_month": "months_with_active_area_data",
    "has_dairy_production_system_data": "months_with_dairy_production_system_data",
}
MAPA_COBERTURA = {k: v for k, v in MAPA_COBERTURA.items() if k in df_mensal_anuais.columns}

# Fontes usadas diretamente no cálculo de COE/COT/CT: são as que mais importam
# para explicar uma possível subestimação de custo.
FONTES_CRITICAS_PARA_CUSTO = {
    "has_expense_data": "Despesas (vw_expense)",
    "has_feeding_data": "Alimentação (vw_feeding)",
    "has_labor_data": "Mão de obra (vw_labor)",
    "has_asset_data": "Patrimônio/Depreciação (mvw_asset_payment_history)",
}

registros_anuais = []
registros_cobertura = []
for id_property, grupo in df_mensal_anuais.groupby("id_property", sort=False):
    grupo = grupo.sort_values("reference_month").reset_index(drop=True)
    for fim in range(11, len(grupo)):
        janela = grupo.iloc[fim - 11:fim + 1]
        meses = janela["reference_month"].dt.to_period("M")
        esperado = pd.period_range(meses.iloc[0], meses.iloc[-1], freq="M")
        if len(esperado) != 12 or list(meses) != list(esperado):
            continue
        registro = janela.iloc[-1].to_dict()
        registro["annual_period_start"] = janela["reference_month"].iloc[0]
        registro["annual_period_end"] = janela["reference_month"].iloc[-1]
        registro["annual_period"] = f"{registro['annual_period_start']:%b/%y}-{registro['annual_period_end']:%b/%y}"
        registro["months_in_annual_window"] = 12
        for coluna in COLUNAS_SOMA_ANUAL:
            registro[f"{coluna}_annual"] = janela[coluna].sum(min_count=1)
        for coluna in COLUNAS_MEDIA_ANUAL:
            registro[f"{coluna}_annual_average"] = janela[coluna].mean()
        volume = pd.to_numeric(janela["milk_produced"], errors="coerce")
        for coluna in COLUNAS_QUALIDADE:
            valores = pd.to_numeric(janela[coluna], errors="coerce")
            validos = valores.notna() & volume.notna() & volume.gt(0)
            registro[f"{coluna}_annual_weighted_average"] = (np.average(valores[validos], weights=volume[validos]) if validos.any() else np.nan)

        # --- Tolerância configurável de meses inconsistentes na janela anual ---
        # 0 = estrito (1 mês inconsistente já invalida a janela anual)
        # N = permite até N meses inconsistentes antes de marcar a janela anual como Inconsistente
        MAX_INCONSISTENT_MONTHS_ALLOWED = 1
        
        inconsistencias = pd.to_numeric(janela["consistency_id"], errors="coerce")
        registro["inconsistent_months_annual"] = int(inconsistencias.sum())
        registro["annual_consistency_id"] = int(registro["inconsistent_months_annual"] > MAX_INCONSISTENT_MONTHS_ALLOWED)
        registro["annual_consistency_status"] = "Consistente" if registro["annual_consistency_id"] == 0 else "Inconsistente"
        
        # --- Concatenar detalhamento das inconsistências mensais da janela ---
        detalhes_mensais_inconsistentes = []
        for _, linha_mes in janela.iterrows():
            if str(linha_mes.get("consistency_status", "")) == "Inconsistente":
                mes_str = f"{linha_mes['reference_month']:%b/%y}"
                detalhe_mes = str(linha_mes.get("violated_consistency_details", ""))
                if detalhe_mes and detalhe_mes != "Nenhum" and pd.notna(linha_mes.get("violated_consistency_details")):
                    detalhes_mensais_inconsistentes.append(f"{mes_str}: {detalhe_mes}")
                else:
                    criterios_mes = str(linha_mes.get("violated_consistency_criteria", ""))
                    detalhes_mensais_inconsistentes.append(f"{mes_str}: {criterios_mes}")

        registro["annual_monthly_inconsistency_details"] = (" | ".join(detalhes_mensais_inconsistentes) if detalhes_mensais_inconsistentes else "Nenhuma")

        registros_anuais.append(registro)

        # --- Cobertura de lançamento de dados desta mesma janela (relatório separado) ---
        registro_cobertura = {
            "id_property": id_property,
            "annual_period_start": registro["annual_period_start"],
            "annual_period_end": registro["annual_period_end"],
            "annual_period": registro["annual_period"],
        }
        fontes_incompletas = []
        for coluna_flag, nome_coluna_cobertura in MAPA_COBERTURA.items():
            meses_com_dado = int(janela[coluna_flag].sum())
            registro_cobertura[nome_coluna_cobertura] = meses_com_dado
            if coluna_flag in FONTES_CRITICAS_PARA_CUSTO and meses_com_dado < 12:
                fontes_incompletas.append(f"{FONTES_CRITICAS_PARA_CUSTO[coluna_flag]}: {meses_com_dado}/12")
        registro_cobertura["cost_data_status"] = "Completa" if not fontes_incompletas else "Incompleta"
        registro_cobertura["cost_data_gaps"] = "; ".join(fontes_incompletas) if fontes_incompletas else "Nenhum"
        registros_cobertura.append(registro_cobertura)

df_anuais = pd.DataFrame(registros_anuais)
if df_anuais.empty:
    raise ValueError("Nenhuma propriedade possui uma janela completa de 12 meses consecutivos.")
if df_anuais.duplicated(["id_property", "annual_period_end"]).any():
    raise ValueError("Foram geradas janelas anuais duplicadas por propriedade e mês final.")
print(f"Janelas anuais calculadas: {len(df_anuais):,}")

# ==============================================================================
# RELATÓRIO DE COBERTURA DE DADOS (SEPARADO DOS INDICADORES ANUAIS)
# ==============================================================================
# Este relatório não entra em df_anuais / df_indicadores_anuais / df_calculo_medias.
# Ele existe para diagnosticar, por propriedade e janela anual, se algum mês deixou
# de ter dado lançado em despesas/alimentação/mão de obra/patrimônio - o que
# subestimaria COE, COT e CT silenciosamente sem isso ficar visível em nenhum outro lugar.
df_cobertura_anual = pd.DataFrame(registros_cobertura)
df_cobertura_anual = df_cobertura_anual.merge(
    df_anuais[["id_property", "annual_period_end", "property_name", "labor_rural_code"]],
    on=["id_property", "annual_period_end"],
    how="left",
) if {"property_name", "labor_rural_code"}.issubset(df_anuais.columns) else df_cobertura_anual
df_cobertura_anual = df_cobertura_anual.sort_values(["id_property", "annual_period_end"]).reset_index(drop=True)

janelas_incompletas = int((df_cobertura_anual["cost_data_status"] == "Incompleta").sum())
print(f"Janelas anuais com lançamento de custo incompleto: {janelas_incompletas:,} de {len(df_cobertura_anual):,}")
display(df_cobertura_anual.head())

Janelas anuais calculadas: 5,333
Janelas anuais com lançamento de custo incompleto: 514 de 5,333


,id_property,annual_period_start,annual_period_end,annual_period,months_with_revenue_data,months_with_cattle_data,months_with_expense_data,months_with_feeding_data,months_with_labor_data,months_with_own_milk_data,months_with_asset_data,months_with_active_area_data,months_with_dairy_production_system_data,cost_data_status,cost_data_gaps,property_name,labor_rural_code
0,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-04-01,2026-03-01,Apr/25-Mar/26,12,12,12,12,12,10,12,12,12,Completa,Nenhum,Fazenda São Bartolomeu,LR01964
1,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-05-01,2026-04-01,May/25-Apr/26,12,12,12,12,12,11,12,12,12,Completa,Nenhum,Fazenda São Bartolomeu,LR01964
2,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-06-01,2026-05-01,Jun/25-May/26,12,12,12,12,12,11,12,12,12,Completa,Nenhum,Fazenda São Bartolomeu,LR01964
3,004cde8f-4688-4b22-acd9-76151b7dcc7a,2025-07-01,2026-06-01,Jul/25-Jun/26,12,12,12,12,12,11,12,12,12,Completa,Nenhum,Fazenda São Bartolomeu,LR01964
4,0122b511-4a6d-4ec8-b107-04346da8b2c8,2024-01-01,2024-12-01,Jan/24-Dec/24,12,12,12,12,12,0,12,12,2,Completa,Nenhum,Fazenda Babilônia,LR05866


## 4.2 Indicadores Derivados Anuais e Regras de Consistência Anual


In [39]:
# ==============================================================================
# 4.2 INDICADORES DERIVADOS ANUAIS E REGRAS DE CONSISTÊNCIA ANUAL
# ==============================================================================
pd.set_option('future.no_silent_downcasting', True)
def dividir_seguro(numerador: pd.Series, denominador: pd.Series) -> pd.Series:
    denom_seguro = denominador.where(denominador != 0, np.nan)
    resultado = numerador / denom_seguro
    return resultado.where(resultado.abs() != np.inf, np.nan)

def coluna_ou_none(df: pd.DataFrame, nome: str) -> pd.Series:
    """Retorna a coluna se ela existir; caso contrário, uma série de None (mesmo índice)."""
    if nome in df.columns:
        return df[nome]
    return pd.Series([None] * len(df), index=df.index)

dias_ano = (df_anuais["annual_period_end"] + pd.offsets.MonthEnd(0) - df_anuais["annual_period_start"] + pd.Timedelta(days=1)).dt.days


# ==============================================================================
# 1. RENDA
# ==============================================================================
df_anuais['rbl_rba'] = dividir_seguro(df_anuais['total_milk_revenue_annual'], df_anuais['total_activity_revenue_annual'])
# Versão em percentual (0-100), equivalente ao antigo rbl_rba já multiplicado por 100.
df_anuais['rbl_rba_percentage_annual'] = df_anuais['rbl_rba'] * 100

# ==============================================================================
# 1. PRODUÇÃO, QUALIDADE E MÃO DE OBRA (já existentes, mantidos)
# ==============================================================================
df_anuais["milk_daily_annual"] = dividir_seguro(df_anuais["milk_produced_annual"], dias_ano)
df_anuais["milk_revenue_liter_annual"] = dividir_seguro(df_anuais["total_milk_revenue_annual"], df_anuais["milk_produced_annual"])
df_anuais["milk_lactating_cow_day_annual"] = dividir_seguro(df_anuais["milk_daily_annual"], df_anuais["lactating_cows_annual_average"])
df_anuais["milk_daily_total_cows_annual"] = dividir_seguro(df_anuais["milk_daily_annual"], df_anuais["total_cows_annual_average"])
df_anuais["lactating_cows_total_cows_annual"] = dividir_seguro(df_anuais["lactating_cows_annual_average"], df_anuais["total_cows_annual_average"]) * 100
df_anuais["lactating_cows_total_cattle_annual"] = dividir_seguro(df_anuais["lactating_cows_annual_average"], df_anuais["total_cattle_annual_average"]) * 100
df_anuais["milk_total_labor_day_annual"] = dividir_seguro(df_anuais["milk_daily_annual"], df_anuais["total_labor_quantity_annual_average"])
df_anuais["lactating_cows_total_labor_annual"] = dividir_seguro(df_anuais["lactating_cows_annual_average"], df_anuais["total_labor_quantity_annual_average"])

# ==============================================================================
# 2. ÁREA (com reserva, forrageira e produção por área)
# ==============================================================================
df_anuais["milk_hectare_activity_annual"] = dividir_seguro(df_anuais["milk_produced_annual"], df_anuais["hectares_atividade_annual_average"])
df_anuais["milk_hectare_activity_com_reserva_annual"] = dividir_seguro(df_anuais["milk_produced_annual"], coluna_ou_none(df_anuais, "hectares_atividade_com_reserva_annual_average"))
df_anuais["milk_hectare_forrageira_annual"] = dividir_seguro(df_anuais["milk_produced_annual"], coluna_ou_none(df_anuais, "hectares_forrageira_annual_average"))
df_anuais["lactating_cows_hectare_activity_annual"] = dividir_seguro(df_anuais["lactating_cows_annual_average"], df_anuais["hectares_atividade_annual_average"])
df_anuais["lactating_cows_hectare_activity_com_reserva_annual"] = dividir_seguro(df_anuais["lactating_cows_annual_average"], coluna_ou_none(df_anuais, "hectares_atividade_com_reserva_annual_average"))
df_anuais["rented_area_percentage_annual"] = dividir_seguro(df_anuais["hectares_arrendada_annual_average"], df_anuais["hectares_total_annual_average"]) * 100
df_anuais["land_value_per_hectare_annual"] = coluna_ou_none(df_anuais, "raw_land_value_medio_ponderado_annual_average")

# ==============================================================================
# 3. QUALIDADE DO LEITE: GORDURA/PROTEÍNA POR VACA EM LACTAÇÃO/DIA
# ==============================================================================
# Réplica do indicador antigo Gordura_VL / Proteina_VL:
# (produção anual/365) * 1,032 * (percentual médio ponderado/100) / vacas em lactação médias
df_anuais["fat_lactating_cow_day_annual"] = dividir_seguro(
    (df_anuais["milk_produced_annual"] / 365) * 1.032 * (coluna_ou_none(df_anuais, "fat_annual_weighted_average") / 100),
    df_anuais["lactating_cows_annual_average"],
)
df_anuais["protein_lactating_cow_day_annual"] = dividir_seguro(
    (df_anuais["milk_produced_annual"] / 365) * 1.032 * (coluna_ou_none(df_anuais, "protein_annual_weighted_average") / 100),
    df_anuais["lactating_cows_annual_average"],
)

# ==============================================================================
# 4. ALIMENTAÇÃO: CUSTO, PREÇO DO CONCENTRADO E RELAÇÃO DE TROCA
# ==============================================================================
df_anuais["feeding_cost_liter_annual"] = dividir_seguro(df_anuais["feeding_cost_annual"], df_anuais["milk_produced_annual"])
df_anuais["concentrate_mineral_cost_liter_annual"] = dividir_seguro(df_anuais["concentrate_mineral_cost_annual"], df_anuais["milk_produced_annual"])
df_anuais["feeding_cost_milk_price_annual"] = dividir_seguro(df_anuais["feeding_cost_liter_annual"], df_anuais["milk_revenue_liter_annual"]) * 100

df_anuais["concentrate_mineral_consumed_quantity_annual"] = (
    df_anuais[["concentrate_consumed_quantity_annual", "mineral_consumed_quantity_annual"]].sum(axis=1, min_count=1)
    if {"concentrate_consumed_quantity_annual", "mineral_consumed_quantity_annual"}.issubset(df_anuais.columns)
    else None
)
df_anuais["concentrate_mineral_price_annual"] = dividir_seguro(df_anuais["concentrate_mineral_cost_annual"], coluna_ou_none(df_anuais, "concentrate_mineral_consumed_quantity_annual"))
df_anuais["milk_concentrate_exchange_ratio_annual"] = dividir_seguro(df_anuais["milk_revenue_liter_annual"], coluna_ou_none(df_anuais, "concentrate_mineral_price_annual"))

# ==============================================================================
# 4b. AGREGAÇÕES DE CUSTO USADAS NO CÁLCULO DE MÉDIAS ANTIGO
# ==============================================================================
# Aleitamento (sucedâneo comprado + leite de bezerro), equivalente ao antigo def_aleitaento_somaMovel.
COMPONENTES_ALEITAMENTO = [c for c in ["milk_replacer_annual", "milk_calves_annual"] if c in df_anuais.columns]
df_anuais["aleitamento_annual"] = df_anuais[COMPONENTES_ALEITAMENTO].sum(axis=1, min_count=1) if COMPONENTES_ALEITAMENTO else None

# Energia + combustível, equivalente ao antigo custoEnergiaCombustivel.
COMPONENTES_ENERGIA_COMBUSTIVEL = [c for c in ["energy_annual", "fuel_annual"] if c in df_anuais.columns]
df_anuais["energy_fuel_cost_annual"] = df_anuais[COMPONENTES_ENERGIA_COMBUSTIVEL].sum(axis=1, min_count=1) if COMPONENTES_ENERGIA_COMBUSTIVEL else None

# Depreciação total do estoque de capital (benfeitorias + máquinas), equivalente ao antigo
# depreciacaoEstoqueCapital_somaMovel. Falta o componente de forrageiras não-anuais/plantio.
COMPONENTES_DEPRECIACAO_TOTAL = [c for c in ["monthly_depreciation_benfeitorias_annual", "monthly_depreciation_maquinas_e_equipamentos_annual"] if c in df_anuais.columns]
df_anuais["total_depreciation_annual"] = df_anuais[COMPONENTES_DEPRECIACAO_TOTAL].sum(axis=1, min_count=1) if COMPONENTES_DEPRECIACAO_TOTAL else None

# ==============================================================================
# 5. COE, COT E CT (CUSTO OPERACIONAL EFETIVO, TOTAL E CUSTO TOTAL)
# ==============================================================================
COMPONENTES_COE_ATIVIDADE_ANUAL = [
    # "other_operating_expenses_annual" foi removida daqui de propósito: ela já é a soma de
    # general_expenses, administration, land_lease, technical_assistance, repairs, hormones,
    # taxes_fees, medicines_vaccines, bedding_replacement, reproduction e milk_replacer
    # (ver COLUNAS_OUTRAS_DESPESAS na Feature Engineering). Mantê-la aqui junto com essas
    # mesmas colunas individuais contava esse bloco de despesas duas vezes no COE.
    "general_expenses_annual", "administration_annual", "land_lease_annual", "technical_assistance_annual",
    "repairs_annual", "hormones_annual", "taxes_fees_annual", "medicines_vaccines_annual", "bedding_replacement_annual",
    "reproduction_annual", "milk_replacer_annual", "milk_calves_annual", "energy_annual", "fuel_annual",
    "voluminous_amount_total_annual", "concentrate_amount_total_annual", "mineral_amount_total_annual",
    "hired_labor_expenses_annual"
]
COMPONENTES_COE_LEITE = [
    # COE do Leite é o COE da Atividade vezes a renda bruta do leite/renda bruta da atividade + Material de Ordenha e sem custos com aleitamento.
    # (mesmo ajuste acima: "other_operating_expenses_annual" removida para não contar essas despesas duas vezes.)
    "general_expenses_annual", "administration_annual", "land_lease_annual", "technical_assistance_annual",
    "repairs_annual", "hormones_annual", "taxes_fees_annual", "medicines_vaccines_annual", "bedding_replacement_annual",
    "reproduction_annual", "milk_replacer_annual", "energy_annual", "fuel_annual", # Removido aleitamento daqui
    "voluminous_amount_total_annual", "concentrate_amount_total_annual", "mineral_amount_total_annual",
    "hired_labor_expenses_annual"
] # Isso aqui será multiplicado por Renda do Leite / Renda da Atividade.

# COE da Atividade
COMPONENTES_COE_ATIVIDADE_ANUAL = [c for c in COMPONENTES_COE_ATIVIDADE_ANUAL if c in df_anuais.columns]
df_anuais["coe_activity_annual"] = df_anuais[COMPONENTES_COE_ATIVIDADE_ANUAL].sum(axis=1, min_count=1) if COMPONENTES_COE_ATIVIDADE_ANUAL else None

# COE do Leite
COMPONENTES_COE_LEITE = [c for c in COMPONENTES_COE_LEITE if c in df_anuais.columns]
df_anuais["coe_milk_annual"] = (
    (df_anuais[COMPONENTES_COE_LEITE].sum(axis=1, min_count=1)) * df_anuais["rbl_rba"]
    + coluna_ou_none(df_anuais, "milking_material_annual")
) if COMPONENTES_COE_LEITE else None

COMPONENTES_COT_ADICIONAIS = [
    "family_labor_expenses_annual",
    "monthly_depreciation_benfeitorias_annual",
    "monthly_depreciation_maquinas_e_equipamentos_annual",
    # Falta o componente de depreciação de forrageiras não-anuais/plantio: sem fonte de dados ainda -> None.
]

COMPONENTES_COT_ADICIONAIS = [c for c in COMPONENTES_COT_ADICIONAIS if c in df_anuais.columns]
if "coe_activity_annual" in df_anuais.columns and COMPONENTES_COT_ADICIONAIS:
    df_anuais["cot_annual"] = df_anuais[["coe_activity_annual"] + COMPONENTES_COT_ADICIONAIS].sum(axis=1, min_count=1)
else:
    df_anuais["cot_annual"] = None

df_anuais['total_capital_stock_annual_average'] = (
    df_anuais[
        [
            'monthly_average_capital_stock_benfeitorias_annual',
            'monthly_average_capital_stock_maquinas_e_equipamentos_annual',
            'animal_capital_stock_annual_average',
            'land_capital_stock_annual_average'
        ]
    ]
    .sum(axis=1)
)
# Estoque de capital sem terra (para o custo de oportunidade do capital).
if {"total_capital_stock_annual_average", "land_capital_stock_annual_average"}.issubset(df_anuais.columns):
    df_anuais["capital_stock_no_land_annual_average"] = (
        df_anuais["total_capital_stock_annual_average"] - df_anuais["land_capital_stock_annual_average"]
    )
else:
    df_anuais["capital_stock_no_land_annual_average"] = None

# Custo de oportunidade do capital: 6% ao ano sobre o estoque de capital sem terra.
df_anuais["capital_opportunity_cost_annual"] = coluna_ou_none(df_anuais, "capital_stock_no_land_annual_average") * 0.06

if df_anuais["cot_annual"].notna().any() and df_anuais["capital_opportunity_cost_annual"].notna().any():
    df_anuais["ct_annual"] = df_anuais[["cot_annual", "capital_opportunity_cost_annual"]].sum(axis=1, min_count=1)
else:
    df_anuais["ct_annual"] = None

# ==============================================================================
# 6. RESULTADO ECONÔMICO: MARGENS, LUCRO E RCMA
# ==============================================================================
df_anuais["gross_margin_annual"] = df_anuais["total_activity_revenue_annual"] - coluna_ou_none(df_anuais, "coe_activity_annual")
df_anuais["net_margin_annual"] = df_anuais["total_activity_revenue_annual"] - coluna_ou_none(df_anuais, "cot_annual")
df_anuais["profit_annual"] = df_anuais["total_activity_revenue_annual"] - coluna_ou_none(df_anuais, "ct_annual")

COMPONENTES_RCMA = [c for c in ["concentrate_amount_total_annual", "mineral_amount_total_annual", "voluminous_amount_total_annual"] if c in df_anuais.columns]
if COMPONENTES_RCMA:
    df_anuais["rcma_annual"] = df_anuais["total_activity_revenue_annual"] - df_anuais[COMPONENTES_RCMA].sum(axis=1, min_count=1)
else:
    df_anuais["rcma_annual"] = None
df_anuais["rcma_lactating_cow_day_annual"] = dividir_seguro(coluna_ou_none(df_anuais, "rcma_annual"), df_anuais["lactating_cows_annual_average"] * 365)

# ==============================================================================
# 7. INDICADORES DE EFICIÊNCIA E RETORNO SOBRE O CAPITAL
# ==============================================================================
df_anuais["turnover_rate_annual"] = dividir_seguro(df_anuais["total_activity_revenue_annual"], df_anuais["total_capital_stock_annual_average"]) * 100
df_anuais["profitability_annual"] = dividir_seguro(coluna_ou_none(df_anuais, "net_margin_annual"), df_anuais["total_activity_revenue_annual"]) * 100

taxa_retorno_sem_terra = dividir_seguro(coluna_ou_none(df_anuais, "net_margin_annual"), coluna_ou_none(df_anuais, "capital_stock_no_land_annual_average")) * 100
df_anuais["capital_return_rate_no_land_annual"] = taxa_retorno_sem_terra.clip(lower=0)

taxa_retorno_com_terra = dividir_seguro(coluna_ou_none(df_anuais, "net_margin_annual"), df_anuais["total_capital_stock_annual_average"]) * 100
df_anuais["capital_return_rate_annual"] = taxa_retorno_com_terra.clip(lower=0)
# Versão sem o piso em zero, usada como auxiliar de estratificação (equivalente à antiga taxaRetornoCapitalComTerra_Geral).
df_anuais["capital_return_rate_annual_unfiltered"] = taxa_retorno_com_terra

# Pontos de cobertura (operacional e total), em litros/dia equivalentes.
df_anuais["operating_coverage_point_annual"] = dividir_seguro(coluna_ou_none(df_anuais, "cot_annual") / (12 * 30.42), df_anuais["milk_revenue_liter_annual"])
df_anuais["total_coverage_point_annual"] = dividir_seguro(coluna_ou_none(df_anuais, "ct_annual") / (12 * 30.42), df_anuais["milk_revenue_liter_annual"])


# ==============================================================================
df_anuais_final = df_anuais.copy()  # Defragmenta antes do bloco extensivo de colunas
# 8. RB, COE, COT, CT, MARGENS E LUCRO SOB DIFERENTES DENOMINADORES
# ==============================================================================
# Réplica sistemática do antigo bloco "colunas_economicas": cada medida econômica anual
# expressa por litro, % da receita, litros equivalentes, por vaca em lactação, por total
# de vacas e por hectare de área da atividade (sem e com reserva).
MEDIDAS_ECONOMICAS_ANUAIS = {
    "total_activity_revenue_annual": "activity_revenue",
    "coe_activity_annual": "coe",
    "cot_annual": "cot",
    "ct_annual": "ct",
    "gross_margin_annual": "gross_margin",
    "net_margin_annual": "net_margin",
    "profit_annual": "profit",
}
for coluna_base, prefixo in MEDIDAS_ECONOMICAS_ANUAIS.items():
    valores_base = coluna_ou_none(df_anuais_final, coluna_base)
    df_anuais_final[f"{prefixo}_liter_annual"] = dividir_seguro(valores_base, df_anuais_final["milk_produced_annual"])
    df_anuais_final[f"{prefixo}_milk_revenue_percentage_annual"] = dividir_seguro(valores_base, df_anuais_final["total_activity_revenue_annual"]) * 100
    df_anuais_final[f"{prefixo}_milk_equivalent_liters_annual"] = dividir_seguro(valores_base, df_anuais_final["milk_revenue_liter_annual"])
    df_anuais_final[f"{prefixo}_lactating_cow_annual"] = dividir_seguro(valores_base, df_anuais_final["lactating_cows_annual_average"])
    df_anuais_final[f"{prefixo}_total_cows_annual"] = dividir_seguro(valores_base, df_anuais_final["total_cows_annual_average"])
    df_anuais_final[f"{prefixo}_hectare_activity_annual"] = dividir_seguro(valores_base, df_anuais_final["hectares_atividade_annual_average"])
    df_anuais_final[f"{prefixo}_hectare_activity_com_reserva_annual"] = dividir_seguro(valores_base, coluna_ou_none(df_anuais_final, "hectares_atividade_com_reserva_annual_average"))

# ==============================================================================
# 9. ESTOQUE DE CAPITAL: TOTAL E QUEBRA POR COMPONENTE
# ==============================================================================
df_anuais_final["total_capital_stock_milk_daily_annual"] = dividir_seguro(df_anuais_final["total_capital_stock_annual_average"], df_anuais_final["milk_daily_annual"])
df_anuais_final["total_capital_stock_lactating_cow_annual"] = dividir_seguro(df_anuais_final["total_capital_stock_annual_average"], df_anuais_final["lactating_cows_annual_average"])

COMPONENTES_ESTOQUE_CAPITAL = {
    "monthly_average_capital_stock_benfeitorias_annual_average": "benfeitorias",
    "monthly_average_capital_stock_maquinas_e_equipamentos_annual_average": "maquinas_e_equipamentos",
    "animal_capital_stock_annual_average": "animais",
    "land_capital_stock_annual_average": "terra",
}
for coluna_base, sufixo in COMPONENTES_ESTOQUE_CAPITAL.items():
    df_anuais_final[f"capital_stock_{sufixo}_share_annual"] = dividir_seguro(coluna_ou_none(df_anuais_final, coluna_base), df_anuais_final["total_capital_stock_annual_average"]) * 100
# Estoque de capital em forrageiras não-anuais/plantio: sem fonte de dados ainda -> None.
df_anuais_final["capital_stock_forrageira_annual_average"] = None
df_anuais_final["capital_stock_forrageira_share_annual"] = None
df_anuais_final["forrageira_depreciation_annual"] = None

# ==============================================================================
df_anuais_final = df_anuais_final.copy()  # Defragmenta antes de adicionar placeholders
# 10. INVESTIMENTO: SEM FONTE DE DADOS AINDA (colunas mantidas como None)
# ==============================================================================
for coluna in [
    "investment_animals_annual",
    "investment_machinery_annual",
    "investment_improvements_annual",
    "investment_land_annual",
    "investment_total_annual",
    "investment_animals_capital_stock_annual",
    "investment_machinery_capital_stock_annual",
    "investment_improvements_capital_stock_annual",
    "investment_total_capital_stock_annual",
    "investment_revenue_annual",
    "investment_gross_margin_annual",
]:
    df_anuais_final[coluna] = None

# ==============================================================================
# 11. DIMENSÃO E CAMPOS AUXILIARES SEM FONTE DE DADOS NO MODELO NOVO
# ==============================================================================
df_anuais_final["agroindustry_code"] = None     # Código da agroindústria: só o nome está disponível.
df_anuais_final["production_system_annual"] = coluna_ou_none(df_anuais_final, "production_system")

# Placeholders herdados do notebook antigo (também eram sempre vazios ou não-informativos lá).
# "filter_1" NAO entra aqui: e calculado abaixo, a partir do merge com o consultor.
for coluna in ["filter_2", "filter_3", "filter_4", "monthly_status_annual", "lab_tests", "milk_quality_notes", "outsourcing", "transport", "vaccines"]:
    df_anuais_final[coluna] = None

# ==============================================================================
# CONSULTOR: MERGE TARDIO (DE PROPOSITO) E FILTRO 1
# Regra de negócio do Cálculo de Médias:
# - a propriedade continua sendo calculada uma única vez;
# - na análise de médias, a mesma propriedade pode ser duplicada para cada consultor vinculado;
# - essa duplicação é intencional e serve para análise por consultor;
# - a base duplicada não deve ser usada para consolidar totais gerais sem reagrupamento por propriedade.
# ==============================================================================
# Feito aqui, sobre df_anuais_final ja agregado -- e nao em df_integrada -- porque um
# merge cedo duplicaria linhas mensais para fazendas com mais de um consultor
# e inflaria todas as somas anuais pelo numero de consultores. Aqui a janela ja
# esta fechada e somada; duplicar a linha so repete o resultado, sem somar de novo.
if "df_dim_consultor" in dir() and not df_dim_consultor.empty:
    df_anuais_final = df_anuais_final.drop(columns=["id_consultant", "consultant_name"], errors="ignore")
    df_anuais_final = df_anuais_final.merge(
        df_dim_consultor[["id_property", "id_consultant", "consultant_name"]],
        on="id_property",
        how="left",
    )
    # consultant_name ja vem padronizado em df_dim_consultor
    
    # Filtro 1: quantos consultores distintos (= quantas vezes esta linha fazenda-periodo se repete).
    df_anuais_final["filter_1"] = (
        df_anuais_final.groupby(["id_property", "annual_period"])["id_consultant"]
        .transform("nunique")
    )
else:
    df_anuais_final["consultant_name"] = None
    df_anuais_final["filter_1"] = None

# Desfragmenta o DataFrame após todas as inserções de colunas (resolve PerformanceWarning).
df_anuais_final = df_anuais_final.copy()
df_anuais_final = df_anuais_final.replace([np.inf, -np.inf], np.nan)

# ==============================================================================
# 11b. REGRAS DE CONSISTÊNCIA NO NÍVEL ANUAL (AVALIAÇÃO DA JANELA DE 12 MESES)
# ==============================================================================
REGRAS_ANUAIS_DETALHADAS = [
    ("ccs_annual_weighted_average", 50, None, "CCS Anual", "", ".0f"),
    ("cpp_annual_weighted_average", 1, None, "CPP Anual", "", ".0f"),
    ("fat_annual_weighted_average", 2.5, 5.5, "Gordura Anual", "%", ".2f"),
    ("protein_annual_weighted_average", 2.4, 4.5, "Proteína Anual", "%", ".2f"),
    ("lactating_cows_total_cows_annual", 20, 99, "VL/Total de vacas Anual", "%", ".1f"),
    ("lactating_cows_total_cattle_annual", 15, 99, "VL/Rebanho total Anual", "%", ".1f"),
    ("milk_lactating_cow_day_annual", 3, 45, "Produção/VL Anual", " L/vaca/dia", ".1f"),
    ("milk_total_labor_day_annual", 20, 1500, "Produção/MDO Anual", " L/trabalhador/dia", ".1f"),
    ("lactating_cows_total_labor_annual", 0, 70, "VL/MDO Anual", " vacas/trabalhador", ".1f"),
    ("feeding_cost_milk_price_annual", 15, 150, "Alimentação/Preço do leite Anual", "%", ".1f"),
    ("feeding_cost_liter_annual", None, 3.0, "Custo de alimentação/L Anual", " R$/L", ".2f"),
    ("concentrate_mineral_cost_liter_annual", 0.30, 3.50, "Custo de concentrado/L Anual", " R$/L", ".2f"),
]

detalhes_anuais_lista = []
criterios_anuais_lista = []

for _, row in df_anuais_final.iterrows():
    motivos = []
    criterios = []
    for col_val, min_v, max_v, nome, unit, fmt in REGRAS_ANUAIS_DETALHADAS:
        val = row.get(col_val, np.nan)
        if pd.isna(val):
            motivos.append(f"{nome} (Ausente/NaN)")
            criterios.append(nome)
        elif min_v is not None and val <= min_v:
            motivos.append(f"{nome} (Abaixo do mín <{min_v}: {val:{fmt}}{unit})")
            criterios.append(nome)
        elif max_v is not None and val >= max_v:
            motivos.append(f"{nome} (Acima do máx >{max_v}: {val:{fmt}}{unit})")
            criterios.append(nome)
            
    detalhes_anuais_lista.append("; ".join(motivos) if motivos else "Nenhum")
    criterios_anuais_lista.append("; ".join(criterios) if criterios else "Nenhum")

df_anuais_final["annual_violated_consistency_criteria"] = criterios_anuais_lista
df_anuais_final["annual_violated_consistency_details"] = detalhes_anuais_lista

# ==============================================================================
# 12. MONTAR A TABELA FINAL
# ==============================================================================
COLUNAS_RESULTADO_ANUAL = [
    # Identificação e período #
    "id_property", "annual_period_start", "annual_period_end", "annual_period",
    "annual_consistency_status", "annual_consistency_id", "inconsistent_months_annual",
    "annual_monthly_inconsistency_details", "annual_violated_consistency_criteria", "annual_violated_consistency_details",
    "production_system_annual",
    # Produção, qualidade e mão de obra #
    "milk_produced_annual", "milk_daily_annual", "milk_revenue_liter_annual",
    "milk_lactating_cow_day_annual", "lactating_cows_total_cows_annual",
    "lactating_cows_total_cattle_annual", "milk_total_labor_day_annual",
    "lactating_cows_total_labor_annual", "fat_lactating_cow_day_annual", "protein_lactating_cow_day_annual",
    "milk_daily_total_cows_annual", "rbl_rba_percentage_annual",
    # Área #
    "milk_hectare_activity_annual", "milk_hectare_activity_com_reserva_annual", "milk_hectare_forrageira_annual",
    "lactating_cows_hectare_activity_annual", "lactating_cows_hectare_activity_com_reserva_annual",
    "rented_area_percentage_annual", "land_value_per_hectare_annual",
    # Alimentação #
    "feeding_cost_liter_annual", "concentrate_mineral_cost_liter_annual", "feeding_cost_milk_price_annual",
    "concentrate_mineral_price_annual", "milk_concentrate_exchange_ratio_annual",
    # Custos agregados #
    "coe_activity_annual", "coe_milk_annual", "cot_annual", "ct_annual", "capital_opportunity_cost_annual",
    "aleitamento_annual", "energy_fuel_cost_annual", "total_depreciation_annual",
    # Resultado econômico #
    "gross_margin_annual", "net_margin_annual", "profit_annual", "rcma_annual", "rcma_lactating_cow_day_annual",
    "turnover_rate_annual", "profitability_annual",
    "capital_return_rate_no_land_annual", "capital_return_rate_annual", "capital_return_rate_annual_unfiltered",
    "operating_coverage_point_annual", "total_coverage_point_annual",
    # Estoque de capital #
    "total_capital_stock_milk_daily_annual", "total_capital_stock_lactating_cow_annual",
    "capital_stock_benfeitorias_share_annual", "capital_stock_maquinas_e_equipamentos_share_annual",
    "capital_stock_animais_share_annual", "capital_stock_terra_share_annual",
    "capital_stock_forrageira_annual_average", "capital_stock_forrageira_share_annual", "forrageira_depreciation_annual",
    # Investimento (sem fonte de dados ainda) #
    "investment_animals_annual", "investment_machinery_annual", "investment_improvements_annual",
    "investment_land_annual", "investment_total_annual",
    "investment_animals_capital_stock_annual", "investment_machinery_capital_stock_annual",
    "investment_improvements_capital_stock_annual", "investment_total_capital_stock_annual",
    "investment_revenue_annual", "investment_gross_margin_annual",
    # Auxiliares sem dado disponível #
    "consultant_name", "agroindustry_code",
    "filter_1", "filter_2", "filter_3", "filter_4", "monthly_status_annual",
    "lab_tests", "milk_quality_notes", "outsourcing", "transport", "vaccines",
]
# Mantém apenas as colunas que de fato existem em df_anuais_final (evita KeyError se alguma dependência faltar).
COLUNAS_RESULTADO_ANUAL = [c for c in COLUNAS_RESULTADO_ANUAL if c in df_anuais_final.columns]

# Adiciona a grade sistemática de RB/COE/COT/CT/Margens/Lucro por litro, %, VL, área etc.
for prefixo in MEDIDAS_ECONOMICAS_ANUAIS.values():
    for sufixo in [
        "liter_annual", "milk_revenue_percentage_annual", "milk_equivalent_liters_annual",
        "lactating_cow_annual", "total_cows_annual", "hectare_activity_annual", "hectare_activity_com_reserva_annual",
    ]:
        nome_coluna = f"{prefixo}_{sufixo}"
        if nome_coluna in df_anuais_final.columns and nome_coluna not in COLUNAS_RESULTADO_ANUAL:
            COLUNAS_RESULTADO_ANUAL.append(nome_coluna)

COLUNAS_RESULTADO_ANUAL += [f"{c}_annual_weighted_average" for c in COLUNAS_QUALIDADE if f"{c}_annual_weighted_average" in df_anuais_final.columns]
COLUNAS_DIMENSAO_ANUAL = [c for c in ["property_name", "labor_rural_code", "entrepreneur_name", "agroindustry_name", "dairy_region", "property_status"] if c in df_anuais_final.columns]
df_indicadores_anuais = df_anuais_final[COLUNAS_DIMENSAO_ANUAL + COLUNAS_RESULTADO_ANUAL].copy()

# Rótulo combinado fazenda - produtor, equivalente ao antigo "fazenda-produtor".
if {"property_name", "entrepreneur_name"}.issubset(df_indicadores_anuais.columns):
    df_indicadores_anuais.insert(
        0,
        "property_entrepreneur_label",
        df_indicadores_anuais["property_name"].fillna("") + " - " + df_indicadores_anuais["entrepreneur_name"].fillna(""),
    )

print(f"Colunas no indicador anual: {df_indicadores_anuais.shape[1]:,}")
display(df_indicadores_anuais.head())


KeyError: "['monthly_average_capital_stock_benfeitorias_annual', 'monthly_average_capital_stock_maquinas_e_equipamentos_annual'] not in index"

## 4.3 Padronização e Mapeamento de Colunas Anuais


In [ ]:
# ==============================================================================
# 4.3 PADRONIZAÇÃO E MAPEAMENTO DE COLUNAS ANUAIS
# ==============================================================================
# Rótulo combinado fazenda - produtor diretamente em df_anuais_final (mesma lógica
# já usada para df_indicadores_anuais), para poder ser puxado pelo mapa abaixo.
if {"property_name", "entrepreneur_name"}.issubset(df_anuais_final.columns) and "property_entrepreneur_label" not in df_anuais_final.columns:
    df_anuais_final["property_entrepreneur_label"] = (
        df_anuais_final["property_name"].fillna("") + " - " + df_anuais_final["entrepreneur_name"].fillna("")
    )

# Mesma ordenação usada na exportação técnica, para que a planilha com nomes
# antigos saia com propriedade/período em ordem crescente.
df_anuais_final = df_anuais_final.sort_values(["id_property", "annual_period_end"]).reset_index(drop=True)

# Mapa (nome técnico antigo -> nome técnico atual em df_anuais_final).
# None indica que o indicador ainda não tem fonte de dados no modelo novo
# (Investimento, Estoque de capital em forrageiras, Consultor etc.) - a
# coluna final é criada mesmo assim, vazia, para manter a mesma estrutura
# do arquivo antigo até que essas fontes existam.

MAPA_NOME_ANTIGO_PARA_ATUAL = {
    "idFazenda": "id_property",
    "fazenda-produtor": "property_entrepreneur_label",
    "codAgroindustria": "labor_rural_code",
    "regiaoLeiteira": "dairy_region",
    "nomeagroindustria": "agroindustry_name",
    "nomeconsultor": "consultant_name",
    "intervalo_movel": "annual_period",
    "consistenciaAnual": "annual_consistency_status",
    "Status_Mensais": "monthly_status_annual",
    "sistema": "production_system_annual",
    "Filtro1": "filter_1",
    "Filtro2": "filter_2",
    "Filtro3": "filter_3",
    "Filtro4": "filter_4",
    "taxaRetornoCapitalComTerra_Geral": "capital_return_rate_annual_unfiltered",
    "areaAtividadeSemReserva_mediaMovel": "hectares_atividade_annual_average",
    "areaAtividadeReserva_mediaMovel": "hectares_atividade_com_reserva_annual_average",
    "areaPropria_mediaMovel": "hectares_propria_annual_average",
    "areaArrendada_mediaMovel": "hectares_arrendada_annual_average",
    "percentualAreaArrendada": "rented_area_percentage_annual",
    "precoTerraNua_somaMovel": "land_value_per_hectare_annual",
    "qtdeVacasEmLactacao_mediaMovel": "lactating_cows_annual_average",
    "totalVacas_mediaMovel": "total_cows_annual_average",
    "totalAnimais_mediaMovel": "total_cattle_annual_average",
    "quantidade_Concentrado_Minerais_Anual": "concentrate_mineral_consumed_quantity_annual",
    "MDOTotalDiaria_mediaMovel": "total_labor_quantity_annual_average",
    "MDOFamiliarDiaria_mediaMovel": "family_labor_quantity_annual_average",
    "MDOContratadaDiaria_mediaMovel": "hired_labor_quantity_annual_average",
    "CCS_mediaMovel": "ccs_annual_weighted_average",
    "CPP_mediaMovel": "cpp_annual_weighted_average",
    "Gordura_mediaMovel": "fat_annual_weighted_average",
    "Proteina_mediaMovel": "protein_annual_weighted_average",
    "absCCS_somaMovel": "abs_ccs_annual",
    "absCPP_somaMovel": "abs_cpp_annual",
    "absGordura_somaMovel": "abs_fat_annual",
    "absProteina_somaMovel": "abs_protein_annual",
    "Gordura_VL": "fat_lactating_cow_day_annual",
    "Proteina_VL": "protein_lactating_cow_day_annual",
    "VL_TV_100": "lactating_cows_total_cows_annual",
    "VL_TA_100": "lactating_cows_total_cattle_annual",
    "VL_areaSemReserva": "lactating_cows_hectare_activity_annual",
    "VL_areaComReserva": "lactating_cows_hectare_activity_com_reserva_annual",
    "VL_MDO": "lactating_cows_total_labor_annual",
    "leiteProduzido_somaMovel": "milk_produced_annual",
    "consumoLeiteDescartado_somaMovel": "discarded_quantity_annual",
    "producaoDiaria": "milk_daily_annual",
    "producaoDiaria_VL": "milk_lactating_cow_day_annual",
    "producaoDiaria_TV": "milk_daily_total_cows_annual",
    "producaoDiaria_MDO": "milk_total_labor_day_annual",
    "producaoAnual_AreaSemReserva": "milk_hectare_activity_annual",
    "producaoAnual_AreaComReserva": "milk_hectare_activity_com_reserva_annual",
    "producaoAnual_AreaForrageira": "milk_hectare_forrageira_annual",
    "areaForrageira_mediaMovel": "hectares_forrageira_annual_average",
    "def_rendaAtividade_somaMovel": "total_activity_revenue_annual",
    "def_rendaLeite_somaMovel": "total_milk_revenue_annual",
    "precoLeiteAnual": "milk_revenue_liter_annual",
    "precoConcentradoAnual": "concentrate_mineral_price_annual",
    "relacaoTroca": "milk_concentrate_exchange_ratio_annual",
    "estoqueCapital_semTerra_somaMovel": "capital_stock_no_land_annual_average",
    "estoqueCapital_comTerra_somaMovel": "total_capital_stock_annual_average",
    "def_estoqueCapitalBenfeitorias_somaMovel": "monthly_average_capital_stock_benfeitorias_annual",
    "def_estoqueCapitalMaquinas_somaMovel": "monthly_average_capital_stock_maquinas_e_equipamentos_annual",
    "estoqueCapitalAnimais_Mensal_somaMovel": "animal_capital_stock_annual_average",
    "def_estoqueTerra_Mensal_somaMovel": "land_capital_stock_annual_average",
    "estoqueCapitalPlantio_acumulado_somaMovel": "capital_stock_forrageira_annual_average",
    "def_estoqueCapitalBenfeitorias_somaMovel_ECTotalcomTerra": "capital_stock_benfeitorias_share_annual",
    "def_estoqueCapitalMaquinas_somaMovel_ECTotalcomTerra": "capital_stock_maquinas_e_equipamentos_share_annual",
    "estoqueCapitalAnimais_Mensal_somaMovel_ECTotalcomTerra": "capital_stock_animais_share_annual",
    "def_estoqueTerra_Mensal_somaMovel_ECTotalcomTerra": "capital_stock_terra_share_annual",
    "estoqueCapitalPlantio_acumulado_somaMovel_ECTotalcomTerra": "capital_stock_forrageira_share_annual",
    "coe_somaMovel_litrosEquivalente": "coe_milk_equivalent_liters_annual",
    "cotAnual_litrosEquivalente": "cot_milk_equivalent_liters_annual",
    "ctAnual_litrosEquivalente": "ct_milk_equivalent_liters_annual",
    "coe_somaMovel_precoLeite": "coe_milk_revenue_percentage_annual",
    "cotAnual_precoLeite": "cot_milk_revenue_percentage_annual",
    "ctAnual_precoLeite": "ct_milk_revenue_percentage_annual",
    "margemBrutaAnual": "gross_margin_annual",
    "margemBrutaAnual_litro": "gross_margin_liter_annual",
    "margemBrutaAnual_AreaSemReserva": "gross_margin_hectare_activity_annual",
    "margemBrutaAnual_AreaComReserva": "gross_margin_hectare_activity_com_reserva_annual",
    "margemBrutaAnual_VL": "gross_margin_lactating_cow_annual",
    "margemBrutaAnual_TotalVacas": "gross_margin_total_cows_annual",
    "margemLiquidaAnual": "net_margin_annual",
    "margemLiquidaAnual_litro": "net_margin_liter_annual",
    "margemLiquidaAnual_AreaSemReserva": "net_margin_hectare_activity_annual",
    "margemLiquidaAnual_AreaComReserva": "net_margin_hectare_activity_com_reserva_annual",
    "margemLiquidaAnual_VL": "net_margin_lactating_cow_annual",
    "margemLiquidaAnual_TotalVacas": "net_margin_total_cows_annual",
    "lucroAnual": "profit_annual",
    "lucroAnual_litro": "profit_liter_annual",
    "RCMA": "rcma_annual",
    "RCMA_VL": "rcma_lactating_cow_day_annual",
    "rbl_rba": "rbl_rba_percentage_annual",
    "estoqueCapital_comTerra_somaMovel_VL": "total_capital_stock_lactating_cow_annual",
    "estoqueCapital_comTerra_somaMovel_litro": "total_capital_stock_milk_daily_annual",
    "investimentoAnimais_ECAnimais": "investment_animals_capital_stock_annual",
    "investimentoMaquinas_ECMaquinas": "investment_machinery_capital_stock_annual",
    "investimentoBenfeitorias_ECBenfeitorias": "investment_improvements_capital_stock_annual",
    "investimento_ECcomTerra": "investment_total_capital_stock_annual",
    "investimento_RBA": "investment_revenue_annual",
    "investimento_MB": "investment_gross_margin_annual",
    "investimentoAnimais_somaMovel": "investment_animals_annual",
    "investimentoMaquinas_somaMovel": "investment_machinery_annual",
    "investimentoBenfeitorias_somaMovel": "investment_improvements_annual",
    "investimento_somaMovel": "investment_total_annual",
    "taxadegiro": "turnover_rate_annual",
    "lucratividade": "profitability_annual",
    "taxaRetornoCapitalSemTerra": "capital_return_rate_no_land_annual",
    "taxaRetornoCapitalComTerra": "capital_return_rate_annual",
    "pcot": "operating_coverage_point_annual",
    "pct": "total_coverage_point_annual",
    "coe_somaMovel": "coe_activity_annual",
    "def_gastoAcessoriosDespesasGerais_somaMovel": "general_expenses_annual",
    "def_aleitaento_somaMovel": "aleitamento_annual",
    "def_gastoArrendamento_somaMovel": "land_lease_annual",
    "custoAlimentacao_Concentrado_Minerais_somaMovel": "concentrate_mineral_cost_annual",
    "custoReceitasConcentrado_somaMovel": "concentrated_sold_annual",
    "def_gastoAdministrativo_somaMovel": "administration_annual",
    "def_gastoAssistenciaTecnica_somaMovel": "technical_assistance_annual",
    "custoEnergiaCombustivel": "energy_fuel_cost_annual",
    "Exames_Laboratoriais": "lab_tests",
    "def_gastoHormonios_somaMovel": "hormones_annual",
    "def_gastoImpostoTaxas_somaMovel": "taxes_fees_annual",
    "custoMDOContratada_somaMovel": "hired_labor_expenses_annual",
    "def_gastoMaterialOrdenha_somaMovel": "milking_material_annual",
    "def_gastoMedicamentosVacinas_somaMovel": "medicines_vaccines_annual",
    "Qualidade_Leite": "milk_quality_notes",
    "def_gastoReparosConsertos_somaMovel": "repairs_annual",
    "def_gastoReproducao_somaMovel": "reproduction_annual",
    "Terceirizacao": "outsourcing",
    "Transporte": "transport",
    "Vacinas": "vaccines",
    "custoAlimentacao_Volumoso_somaMovel": "voluminous_amount_total_annual",
    "custoReceitasVolumoso_somaMovel": "voluminous_sold_annual",
    "cotAnual": "cot_annual",
    "depreciacaoEstoqueCapital_somaMovel": "total_depreciation_annual",
    "def_familiarValortotal_somaMovel": "family_labor_expenses_annual",
    "ctAnual": "ct_annual",
    "custoOportunidadeCapital": "capital_opportunity_cost_annual",
}

# ==============================================================================
# ORDEM E NOMES ANTIGOS (idênticos ao colunas_finais/novos_nomes do notebook antigo)
# ==============================================================================
COLUNAS_ANTIGAS_EM_ORDEM = [
    "idFazenda", "fazenda-produtor", "codAgroindustria", "regiaoLeiteira", "nomeagroindustria",
    "nomeconsultor", "intervalo_movel", "consistenciaAnual", "Status_Mensais", "sistema",
    "Filtro1", "Filtro2", "Filtro3", "Filtro4", "taxaRetornoCapitalComTerra_Geral",
    "areaAtividadeSemReserva_mediaMovel", "areaAtividadeReserva_mediaMovel", "areaPropria_mediaMovel",
    "areaArrendada_mediaMovel", "percentualAreaArrendada", "precoTerraNua_somaMovel",
    "qtdeVacasEmLactacao_mediaMovel", "totalVacas_mediaMovel", "totalAnimais_mediaMovel",
    "quantidade_Concentrado_Minerais_Anual",
    "MDOTotalDiaria_mediaMovel", "MDOFamiliarDiaria_mediaMovel", "MDOContratadaDiaria_mediaMovel",
    "CCS_mediaMovel", "CPP_mediaMovel", "Gordura_mediaMovel", "Proteina_mediaMovel", "absCCS_somaMovel",
    "absCPP_somaMovel", "absGordura_somaMovel", "absProteina_somaMovel", "Gordura_VL", "Proteina_VL",
    "VL_TV_100", "VL_TA_100", "VL_areaSemReserva", "VL_areaComReserva", "VL_MDO",
    "leiteProduzido_somaMovel", "consumoLeiteDescartado_somaMovel", "producaoDiaria", "producaoDiaria_VL", "producaoDiaria_TV",
    "producaoDiaria_MDO", "producaoAnual_AreaSemReserva", "producaoAnual_AreaComReserva", "producaoAnual_AreaForrageira", "areaForrageira_mediaMovel",
    "def_rendaAtividade_somaMovel", "def_rendaLeite_somaMovel", "precoLeiteAnual", "def_rendaLeite_somaMovel",
    "precoConcentradoAnual", "relacaoTroca",
    "estoqueCapital_semTerra_somaMovel", "estoqueCapital_comTerra_somaMovel", "def_estoqueCapitalBenfeitorias_somaMovel",
    "def_estoqueCapitalMaquinas_somaMovel", "estoqueCapitalAnimais_Mensal_somaMovel", "def_estoqueTerra_Mensal_somaMovel", "estoqueCapitalPlantio_acumulado_somaMovel",
    "def_estoqueCapitalBenfeitorias_somaMovel_ECTotalcomTerra", "def_estoqueCapitalMaquinas_somaMovel_ECTotalcomTerra", "estoqueCapitalAnimais_Mensal_somaMovel_ECTotalcomTerra",
    "def_estoqueTerra_Mensal_somaMovel_ECTotalcomTerra", "estoqueCapitalPlantio_acumulado_somaMovel_ECTotalcomTerra",
    "coe_somaMovel_litrosEquivalente", "cotAnual_litrosEquivalente", "ctAnual_litrosEquivalente",
    "coe_somaMovel_precoLeite", "cotAnual_precoLeite", "ctAnual_precoLeite",
    "margemBrutaAnual", "margemBrutaAnual_litro", "margemBrutaAnual_AreaSemReserva", "margemBrutaAnual_AreaComReserva", "margemBrutaAnual_VL", "margemBrutaAnual_TotalVacas",
    "margemLiquidaAnual", "margemLiquidaAnual_litro", "margemLiquidaAnual_AreaSemReserva", "margemLiquidaAnual_AreaComReserva", "margemLiquidaAnual_VL", "margemLiquidaAnual_TotalVacas",
    "lucroAnual", "lucroAnual_litro",
    "RCMA", "RCMA_VL",
    "rbl_rba",
    "estoqueCapital_comTerra_somaMovel_VL", "estoqueCapital_comTerra_somaMovel_litro",
    "investimentoAnimais_ECAnimais", "investimentoMaquinas_ECMaquinas", "investimentoBenfeitorias_ECBenfeitorias", "investimento_ECcomTerra",
    "investimento_RBA", "investimento_MB", "investimentoAnimais_somaMovel", "investimentoMaquinas_somaMovel", "investimentoBenfeitorias_somaMovel", "investimento_somaMovel",
    "taxadegiro", "lucratividade", "taxaRetornoCapitalSemTerra", "taxaRetornoCapitalComTerra", "pcot", "pct",
    "coe_somaMovel", "def_gastoAcessoriosDespesasGerais_somaMovel", "def_aleitaento_somaMovel", "def_gastoArrendamento_somaMovel",
    "custoAlimentacao_Concentrado_Minerais_somaMovel", "custoReceitasConcentrado_somaMovel", "def_gastoAdministrativo_somaMovel",
    "def_gastoAssistenciaTecnica_somaMovel", "custoEnergiaCombustivel", "Exames_Laboratoriais", "def_gastoHormonios_somaMovel", "def_gastoImpostoTaxas_somaMovel",
    "custoMDOContratada_somaMovel", "def_gastoMaterialOrdenha_somaMovel", "def_gastoMedicamentosVacinas_somaMovel", "Qualidade_Leite", "def_gastoReparosConsertos_somaMovel",
    "def_gastoReproducao_somaMovel", "Terceirizacao", "Transporte", "Vacinas", "custoAlimentacao_Volumoso_somaMovel", "custoReceitasVolumoso_somaMovel",
    "cotAnual", "depreciacaoEstoqueCapital_somaMovel", "def_familiarValortotal_somaMovel", "ctAnual", "custoOportunidadeCapital",
]

NOMES_PORTUGUES_EM_ORDEM = [
    "IDFazenda", "Fazenda - Produtor", "Código LR", "Região", "Agroindústria",
    "Consultor", "Período", "Status - Ind. Anuais", "Status - Ind. Mensais", "Sistema de produção atual",
    "Filtro 1", "Filtro 2", "Filtro 3", "Filtro 4", "AUXILIAR PARA ESTRATIFICAÇÃO - TRCCT CALCULADA",
    "Área destinada à atividade (hectare)", "Área destinada à atividade considerando reserva (hectare)", "Área própria considerando reserva (hectare)",
    "Área arrendada considerando reserva (hectare)", "Percentual de área arrendada (%)", "Preço médio da terra própria (R$/hectare)",
    "Vacas em lactação (animais/mês)", "Total de vacas (animais/mês)", "Total de animais (animais/mês)",
    "Consumo de concentrado anual (Kg/Ano)",
    "Mão de obra total (trabalhador)", "Mão de obra familiar (trabalhador)", "Mão de obra contratada (trabalhador)",
    "CCS (Contagem de células somáticas) (x1000 células/ml)", "CPP (Contagem padrão em placas) (x1000 UFC/ml)", "Gordura (%)", "Proteína (%)",
    "AUXILIAR PARA MÉDIAS - CCS (Contagem de células somáticas) (x1000 células/ml)", "AUXILIAR PARA MÉDIAS - CPP (Contagem padrão em placas) (x1000 UFC/ml)",
    "AUXILIAR PARA MÉDIAS - Gordura (%)", "AUXILIAR PARA MÉDIAS - Proteína (%)", "Gordura (kg/vaca em lactação/dia)", "Proteína (kg/vaca em lactação/dia)",
    "Vacas em lactação/total de vacas (%)", "Vacas em lactação/total de animais (%)",
    "Vacas em lactação/área destinada à atividade (animais/hectare)", "Vacas em lactação/área destinada à atividade considerando reserva (animais/hectare)",
    "Vacas em lactação/mão de obra total (animais/trabalhador/dia)",
    "Produção anual de leite (litros/ano)", "Leite descartado (litros/ano)", "Produção diária de leite (litros/dia)",
    "Produção/vacas em lactação (litros/animal/dia)", "Produção/total de vacas (litros/animal/dia)",
    "Produção/mão de obra total (litros/trabalhador/dia)", "Produção/área destinada à atividade (litros/hectare/ano)",
    "Produção/área destinada à atividade considerando reserva (litros/hectare/ano)", "Produção / área de produção de forrageira (litros/hectare/ano)",
    "AUXILIAR PARA MÉDIAS - Produção / área de produção de forrageira (litros/hectare/ano)",
    "Renda bruta da atividade leiteira (R$/ano)", "Renda bruta do leite (R$/ano)", "Preço médio do leite (R$/litro)",
    "AUXILIAR PARA MÉDIAS - Preço médio do leite (R$/litro)", "Preço médio do concentrado (R$/Kg)", "Relação de troca leite/concentrado (Kg/L)",
    "Estoque de capital total sem terra (R$)", "Estoque de capital total com terra (R$)", "Estoque de capital em benfeitorias (R$)",
    "Estoque de capital em máquinas (R$)", "Estoque de capital em animais (R$)", "Estoque de capital em terra (R$)", "Estoque de capital em forrageiras não-anuais (R$)",
    "Estoque de capital em benfeitorias/estoque de capital total com terra (%)", "Estoque de capital em máquinas/estoque de capital total com terra (%)",
    "Estoque de capital em animais/estoque de capital total com terra (%)", "Estoque de capital em terra/estoque de capital total com terra (%)",
    "Estoque de capital em forrageiras não-anuais/estoque de capital total com terra (%)",
    "COE da atividade leiteira em equivalentes litros de leite (litros/ano)", "COT da atividade leiteira em equivalentes litros de leite (litros/ano)",
    "CT da atividade leiteira em equivalentes litros de leite (litros/ano)",
    "COE do leite/preço do leite (%)", "COT do leite/preço do leite (%)", "CT do leite/preço do leite (%)",
    "Margem bruta da atividade (R$/ano)", "Margem bruta unitária (R$/litro)", "Margem bruta/área destinada à atividade (R$/hectare/ano)",
    "Margem bruta/área destinada à atividade considerando reserva (R$/hectare/ano)", "Margem bruta/vacas em lactação (R$/animal/ano)", "Margem bruta/total de vacas (R$/animal/ano)",
    "Margem líquida da atividade (R$/ano)", "Margem líquida unitária (R$/litro)", "Margem líquida/área destinada à atividade (R$/hectare/ano)",
    "Margem líquida/área destinada à atividade considerando reserva (R$/hectare/ano)", "Margem líquida/vacas em lactação (R$/animal/ano)", "Margem líquida/total de vacas (R$/animal/ano)",
    "Lucro total (R$/ano)", "Lucro unitário (R$/litro)",
    "RMCA (Receita Menos Custo com Alimentação) (R$/ano)", "RMCA (Receita Menos Custo com Alimentação) (R$/vaca em lactação/dia)",
    "Renda do leite/renda atividade (%)",
    "Estoque de capital total com terra/vaca em lactação (R$/animal)", "Estoque de capital total com terra/produção diária de leite (R$/litro/dia)",
    "Investimento em animais/estoque de capital em animais (%)", "Investimento em máquinas e equipamento/estoque de capital em máquinas e equipamentos (%)",
    "Investimento em benfeitorias/estoque de capital em benfeitorias (%)", "Investimento anual/estoque de capital total com terra (%)",
    "Investimento anual/renda bruta da atividade (%)", "Investimento anual/margem bruta (%)",
    "AUXILIAR PARA MÉDIAS - Investimento em animais", "AUXILIAR PARA MÉDIAS - Investimento em máquinas e equipamento",
    "AUXILIAR PARA MÉDIAS - Investimento em benfeitorias", "AUXILIAR PARA MÉDIAS - Investimento anual",
    "Taxa de giro do estoque de capital total (%)", "Lucratividade operacional (%)",
    "Taxa de remuneração do capital sem terra (% ao ano)", "Taxa de remuneração do capital com terra (% ao ano)",
    "Ponto de cobertura operacional total da atividade (litros/dia)", "Ponto de cobertura total da atividade (litros/dia)",
    "Custo operacional efetivo - R$/ano (Atividade)", "Acessórios e despesas em geral - R$/ano (Atividade)", "Aleitamento - R$/ano (Atividade)",
    "Arrendamento/Aluguel - R$/ano (Atividade)", "Concentrados e Minerais de ingestão livre - R$/ano (Atividade)", "Concentrados (Vendido) - R$/ano (Atividade)",
    "Despesas administrativas - R$/ano (Atividade)", "Despesas com assistência técnica - R$/ano (Atividade)", "Energia e combustível - R$/ano (Atividade)",
    "Exames laboratoriais - R$/ano (Atividade)", "Hormônios - R$/ano (Atividade)", "Impostos e taxas - R$/ano (Atividade)",
    "Mão de obra contratada - R$/ano (Atividade)", "Material de ordenha - R$/ano (Atividade)", "Medicamentos - R$/ano (Atividade)",
    "Qualidade do leite - R$/ano (Atividade)", "Reparos e consertos de máquinas e benfeitorias - R$/ano (Atividade)", "Reprodução - R$/ano (Atividade)",
    "Terceirização de recria - R$/ano (Atividade)", "Transporte e descontos no leite - R$/ano (Atividade)", "Vacinas - R$/ano (Atividade)",
    "Volumosos - R$/ano (Atividade)", "Volumosos (Vendido) - R$/ano (Atividade)",
    "Custo operacional total - R$/ano (Atividade)", "Depreciação - R$/ano (Atividade)", "Mão de obra familiar - R$/ano (Atividade)",
    "Custo total - R$/ano (Atividade)", "Remuneração do capital - R$/ano (Atividade)",
]

assert len(COLUNAS_ANTIGAS_EM_ORDEM) == len(NOMES_PORTUGUES_EM_ORDEM) == 140, (
    "As listas de nomes antigos e novos precisam ter 140 posições cada, igual ao notebook original."
)

# ==============================================================================
# MONTAR A TABELA COM OS NOMES/ORDEM DO CALCULO_MEDIAS.XLSX ANTIGO
# ==============================================================================
dados_renomeados = {}
colunas_sem_correspondencia = []

for nome_antigo, nome_pt in zip(COLUNAS_ANTIGAS_EM_ORDEM, NOMES_PORTUGUES_EM_ORDEM):

    nome_atual = MAPA_NOME_ANTIGO_PARA_ATUAL.get(nome_antigo)

    if nome_atual is not None and nome_atual in df_anuais_final.columns:
        valores = df_anuais_final[nome_atual]
    else:
        valores = pd.Series([None] * len(df_anuais_final), index=df_anuais_final.index)
        colunas_sem_correspondencia.append((nome_antigo, nome_pt))

    # "AUXILIAR PARA MÉDIAS - Preço médio do leite" repete a mesma coluna de origem
    # que "Renda bruta do leite (R$/ano)" - isso é intencional (auxiliar de recálculo
    # de médias ponderadas no BI), igual ao notebook antigo.
    if nome_pt in dados_renomeados:
        nome_pt = f"{nome_pt} (2)"

    dados_renomeados[nome_pt] = valores.values

df_calculo_medias = pd.DataFrame(dados_renomeados, index=df_anuais_final.index)

print(f"Colunas mapeadas com dado real: {140 - len(colunas_sem_correspondencia)} / 140")
print(f"Colunas sem correspondência no modelo novo (ficam vazias): {len(colunas_sem_correspondencia)}")
for nome_antigo, nome_pt in colunas_sem_correspondencia:
    print(f"  - {nome_antigo} -> {nome_pt}")

display(df_calculo_medias.head())

## 4.4 Exportação dos Arquivos Excel Anuais (Multi-Abas)


In [ ]:
# # ==============================================================================
# # 4.4 EXPORTAÇÃO DOS ARQUIVOS EXCEL ANUAIS (MULTI-ABAS)
# # ==============================================================================
# # Garante que a raiz do projeto seja a pasta pai caso o notebook rode dentro da pasta /app
# RAIZ_PROJETO = Path.cwd().parent if Path.cwd().name == 'app' else Path.cwd()

# DATA_EXPORTACAO = datetime.now().strftime("%Y_%m_%d_%H%M%S")
# PASTA_SAIDA = RAIZ_PROJETO / 'data' / 'outputs' / 'annual'
# PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
# CAMINHO_ANUAIS = PASTA_SAIDA / f"{DATA_EXPORTACAO}_indicadores_anuais.xlsx"

# df_exportacao_anual = preparar_dataframe_para_excel(df_indicadores_anuais)
# df_exportacao_anual = df_exportacao_anual.sort_values(["id_property", "annual_period_end"]).reset_index(drop=True)

# # Aba extra com os mesmos nomes/ordem de colunas do antigo calculo_medias.xlsx.
# df_exportacao_calculo_medias = preparar_dataframe_para_excel(df_calculo_medias)

# # Aba extra com o relatório de cobertura de lançamento de dados (separado dos indicadores).
# df_exportacao_cobertura = preparar_dataframe_para_excel(df_cobertura_anual)

# # Aba extra com o relatório de auditoria de consistência mensal e anual.
# COLUNAS_CONSISTENCIA_EXPORT = [
#     c for c in [
#         "id_property", "property_name", "labor_rural_code", "annual_period",
#         "annual_consistency_status", "inconsistent_months_annual",
#         "annual_monthly_inconsistency_details", "annual_violated_consistency_details"
#     ] if c in df_indicadores_anuais.columns
# ]
# df_exportacao_consistencia = preparar_dataframe_para_excel(df_indicadores_anuais[COLUNAS_CONSISTENCIA_EXPORT])

# exportar_varias_abas_xlsx(
#     abas={
#         "Indicadores Anuais": df_exportacao_anual,
#         "Cálculo de Médias": df_exportacao_calculo_medias,
#         "Cobertura de Dados": df_exportacao_cobertura,
#         "Consistência de Dados": df_exportacao_consistencia,
#     },
#     caminho_saida=CAMINHO_ANUAIS,
#     fonte="Aptos",
# )

# print(f"Linhas anuais exportadas: {len(df_exportacao_anual):,}")
# print(f"Arquivo: {CAMINHO_ANUAIS}")